In [14]:
# Parameters
run_date = "2026-01-01"  # papermill replacement
import os
output_dir = os.environ.get("ORION_SIGNALS_DIR", "../signals")
config_path = os.environ.get("DATUM_API_CONFIG_PATH", "../ops/datum_api_config.json")
dry_run = False

# ensure output exists
os.makedirs(output_dir, exist_ok=True)


In [15]:
# Import basic modules
import pandas as pd
from datum_api_client import DatumApi
import datetime
from datetime import timedelta

# Import warnings
import warnings
warnings.filterwarnings("ignore")
# pip install xlrd
# pip install openpyxl

# Download excel with ticker


# ===== Спільний швидкий HTTP-шар Datum API (використовують клітинки 11, 13, 42) =====
# keep-alive сесія на потік + спільна пауза для всіх потоків після 429 + повтори лише там, де вони допомагають.
import threading
import time as _time          # приватні псевдоніми: інші клітинки можуть перепризначити `time`/`random`
import random as _random
import json as _json
import requests

try:
    import orjson as _orjson            # необов'язково: у 3-5 разів швидший парсинг великих відповідей
    _json_loads = _orjson.loads
except Exception:
    _json_loads = _json.loads

_DATUM_TLS = threading.local()
_DATUM_TOKEN_LOCK = threading.Lock()
_DATUM_COOLDOWN = {"until": 0.0}
_DATUM_COOLDOWN_LOCK = threading.Lock()

def _datum_cooldown(seconds: float = 0.0):
    """seconds > 0: ставить спільну паузу для ВСІХ потоків (після 429); завжди чекає, поки пауза діє."""
    if seconds > 0:
        with _DATUM_COOLDOWN_LOCK:
            _DATUM_COOLDOWN["until"] = max(_DATUM_COOLDOWN["until"], _time.monotonic() + seconds)
    wait = _DATUM_COOLDOWN["until"] - _time.monotonic()
    if wait > 0:
        _time.sleep(min(wait, 120))

def datum_get_df(endpoint, params=None, *, timeout=60, max_retries=6, backoff=1.0):
    """GET до Datum API -> DataFrame (може бути порожнім).
    Повторює лише 429 / 5xx / мережеві збої; інші 4xx (404, 400...) падають одразу.
    Кидає виняток, коли спроби вичерпано."""
    if not (hasattr(DatumApi, "get_token_str") and getattr(DatumApi, "api_domain", "")):
        return DatumApi.data_request(endpoint, params)          # запасний шлях
    for attempt in range(max_retries + 1):
        try:
            _datum_cooldown()                                    # чекаємо, якщо інший потік отримав 429
            sess = getattr(_DATUM_TLS, "s", None)
            if sess is None:
                sess = _DATUM_TLS.s = requests.Session()
            with _DATUM_TOKEN_LOCK:                              # токен оновлює один потік
                token = DatumApi.get_token_str()
            r = sess.get(DatumApi.api_domain + endpoint, params=params,
                         headers={"Authorization": token}, timeout=timeout)
            r.raise_for_status()
            return pd.DataFrame(_json_loads(r.content))
        except Exception as ex:
            resp = getattr(ex, "response", None)
            code = getattr(resp, "status_code", None)
            if code is not None and code != 429 and code < 500:
                raise
            if attempt == max_retries:
                raise
            delay = backoff * (2 ** attempt) * (0.9 + 0.2 * _random.random())
            if code == 429:
                ra = resp.headers.get("Retry-After") if resp is not None else None
                if ra and str(ra).isdigit():
                    delay = max(delay, float(ra))
                _datum_cooldown(delay)                           # пауза для всіх потоків
            else:
                _time.sleep(delay)


In [ ]:
# ===== RAM-бюджет: ноутбук 16 ГБ, інші програми вже займають частину пам'яті =====
# Процес ноутбука тримає в RAM ≤ RAM_LIMIT_GB. Усе, що більше, іде БАТЧАМИ на диск
# (завантаження -> день-файли, обробка -> iter_batches, сортування -> бакети),
# а проміжні файли видаляються в кінці (клітинка з recompute_tp_deviation_devsig).
import os, gc
import pyarrow as pa

RAM_LIMIT_GB      = 6.0    # м'який ліміт RSS процесу; розміри батчів нижче масштабуються від нього
HARD_CAP_EXTRA_GB = 4.0    # жорстка стеля Windows (рахує COMMIT, а він помітно більший за RSS) = RAM_LIMIT_GB + це (MemoryError замість свопу/фрізу ноута). None = вимкнено
CLEANUP_INTERMEDIATE = True  # видаляти проміжні parquet/tmp після готового final.parquet

_k = RAM_LIMIT_GB / 6.0
FETCH_FLUSH_ROWS    = int(1_500_000 * _k)   # буфер завантаження (рядків) до скидання на диск
FETCH_WORKERS       = 16                    # HTTP-потоків для акцій (було 32)
ETF_FETCH_WORKERS   = 12                    # для ETF (було 16)
BATCH_ROWS          = int(1_000_000 * _k)   # рядків на батч у iter_batches (було 2 млн)
SORT_ROWS_IN_MEMORY = int(4_000_000 * _k)   # рядків у бакеті фінального сортування (було 20 млн; ~1 ГБ на 4 млн x 23 колонки; final має ~1 млрд рядків -> ~255 бакетів)
SORT_FLUSH_BYTES    = int(1_500_000_000 * _k)  # скидати бакети на диск, коли пул Arrow займає стільки (байт) - ліміт у БАЙТАХ, а не в рядках
IO_WORKERS          = 3                     # паралельних читачів gz-CSV (було 4)
CSV_BLOCK_SIZE      = 16 << 20              # блок читання CSV, байт (було 64 МБ)

pa.set_cpu_count(4)
pa.set_io_thread_count(4)

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:
    psutil = None

def rss_gb() -> float:
    return _PROC.memory_info().rss / 1024**3 if psutil else float("nan")

def mem_diag(label: str = "") -> str:
    """Один рядок діагностики: working set, committed (те, що рахує жорстка стеля), пул Arrow."""
    try:
        mi = _PROC.memory_info()
        rss, vms = mi.rss / 1024**3, mi.vms / 1024**3
    except Exception:
        rss = vms = float("nan")
    pool = pa.total_allocated_bytes() / 1024**3
    return f"[mem {label}] RSS={rss:.2f} ГБ | commit={vms:.2f} ГБ | arrow-пул={pool:.2f} ГБ"

def _ram_over_soft() -> bool:
    """True, коли RSS > 80% ліміту -> завантаження достроково скидає буфер на диск."""
    v = rss_gb()
    return v == v and v > 0.8 * RAM_LIMIT_GB

def ram_guard(label: str = "") -> float:
    """Повертає пам'ять ОС: gc + release_unused() пулу Arrow; друкує RSS відносно ліміту."""
    gc.collect()
    try:
        pa.default_memory_pool().release_unused()
    except Exception:
        pass
    v = rss_gb()
    print(f"🧠 RAM {label}: {v:.2f} / {RAM_LIMIT_GB:.1f} ГБ" + ("  ⚠️ понад ліміт" if v > RAM_LIMIT_GB else ""))
    return v

def _apply_hard_cap(limit_bytes: int) -> bool:
    """Windows Job Object: ліміт committed-пам'яті процесу. Перевищення -> MemoryError у клітинці
    (файли на диску лишаються, повторний запуск продовжує з маніфесту), а не своп і зависання системи."""
    if os.name != "nt":
        return False
    import ctypes
    from ctypes import wintypes

    class IO_COUNTERS(ctypes.Structure):
        _fields_ = [(n, ctypes.c_ulonglong) for n in (
            "ReadOperationCount", "WriteOperationCount", "OtherOperationCount",
            "ReadTransferCount", "WriteTransferCount", "OtherTransferCount")]

    class BASIC(ctypes.Structure):
        _fields_ = [("PerProcessUserTimeLimit", ctypes.c_int64), ("PerJobUserTimeLimit", ctypes.c_int64),
                    ("LimitFlags", wintypes.DWORD), ("MinimumWorkingSetSize", ctypes.c_size_t),
                    ("MaximumWorkingSetSize", ctypes.c_size_t), ("ActiveProcessLimit", wintypes.DWORD),
                    ("Affinity", ctypes.c_size_t), ("PriorityClass", wintypes.DWORD),
                    ("SchedulingClass", wintypes.DWORD)]

    class EXT(ctypes.Structure):
        _fields_ = [("BasicLimitInformation", BASIC), ("IoInfo", IO_COUNTERS),
                    ("ProcessMemoryLimit", ctypes.c_size_t), ("JobMemoryLimit", ctypes.c_size_t),
                    ("PeakProcessMemoryUsed", ctypes.c_size_t), ("PeakJobMemoryUsed", ctypes.c_size_t)]

    k32 = ctypes.WinDLL("kernel32", use_last_error=True)
    k32.CreateJobObjectW.restype = wintypes.HANDLE
    k32.CreateJobObjectW.argtypes = [ctypes.c_void_p, wintypes.LPCWSTR]
    k32.GetCurrentProcess.restype = wintypes.HANDLE
    k32.SetInformationJobObject.argtypes = [wintypes.HANDLE, ctypes.c_int, ctypes.c_void_p, wintypes.DWORD]
    k32.AssignProcessToJobObject.argtypes = [wintypes.HANDLE, wintypes.HANDLE]

    job = k32.CreateJobObjectW(None, None)
    if not job:
        return False
    info = EXT()
    info.BasicLimitInformation.LimitFlags = 0x100            # JOB_OBJECT_LIMIT_PROCESS_MEMORY
    info.ProcessMemoryLimit = int(limit_bytes)
    if not k32.SetInformationJobObject(job, 9, ctypes.byref(info), ctypes.sizeof(info)):   # 9 = ExtendedLimitInformation
        return False
    if not k32.AssignProcessToJobObject(job, k32.GetCurrentProcess()):
        return False
    globals()["_RAM_JOB_HANDLE"] = job                       # тримаємо handle живим
    return True

if HARD_CAP_EXTRA_GB is not None and "_RAM_JOB_HANDLE" not in globals():
    _cap = RAM_LIMIT_GB + HARD_CAP_EXTRA_GB
    _ok = False
    try:
        _ok = _apply_hard_cap(int(_cap * 1024**3))
    except Exception as _e:
        print("⚠️ жорстку стелю не встановлено:", _e)
    print(f"🔒 жорстка стеля {_cap:.1f} ГБ: {'увімкнено' if _ok else 'недоступна (працює лише м’який ліміт)'}")

print(f"RAM-бюджет: {RAM_LIMIT_GB:.1f} ГБ | fetch flush {FETCH_FLUSH_ROWS:,} рядків, {FETCH_WORKERS} потоків | "
      f"batch {BATCH_ROWS:,} | сорт-бакет {SORT_ROWS_IN_MEMORY:,}")
ram_guard("старт")


In [16]:
import os
from pathlib import Path

def find_orion_home() -> Path:
    # 1) найнадійніше — env var
    env = os.environ.get("ORION_HOME")
    if env:
        p = Path(env).expanduser().resolve()
        if not p.exists():
            raise FileNotFoundError(f"ORION_HOME points to missing path: {p}")
        return p

    # 2) fallback: шукаємо папку OriON вгору від поточної директорії
    here = Path.cwd().resolve()
    for parent in [here] + list(here.parents):
        if parent.name.lower() == "orion":
            return parent
        cand = parent / "OriON"
        if cand.exists() and cand.is_dir():
            return cand.resolve()

    raise RuntimeError("Cannot locate OriON. Set ORION_HOME env var.")

ORION_HOME = find_orion_home()

CRACEN_DIR = ORION_HOME / "CRACEN"
WORK_DIR   = CRACEN_DIR / "work"     # батчі/маніфести/тимчасові файли
FINAL_PATH = CRACEN_DIR / "final.parquet"

CRACEN_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

BATCH_DIR = WORK_DIR / "CRACEN_batch"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
ETF_DIR = WORK_DIR / "ETF"
ETF_DIR.mkdir(parents=True, exist_ok=True)


print("ORION_HOME:", ORION_HOME)
print("CRACEN_DIR :", CRACEN_DIR)
print("WORK_DIR  :", WORK_DIR)
print("FINAL_PATH:", FINAL_PATH)


ORION_HOME: C:\datum-api-examples-main\OriON
CRACEN_DIR : C:\datum-api-examples-main\OriON\CRACEN
WORK_DIR  : C:\datum-api-examples-main\OriON\CRACEN\work
FINAL_PATH: C:\datum-api-examples-main\OriON\CRACEN\final.parquet


In [17]:
from datetime import datetime, timedelta, date   # НЕ імпортуємо `time`: воно затирає модуль time у всьому ноутбуці
from calendar import monthrange

# === ТЗ користувача: Europe/Kyiv ===
tz = None  # якщо нема zoneinfo, все одно працює локально

now_local = datetime.now(tz) if tz else datetime.now()
yesterday = (now_local - timedelta(days=1)).date()

# Ковзне вікно: завжди WINDOW_DAYS календарних днів, що закінчується вчора.
# Щодня вікно зсувається на 1 день: нові дні докачуються, старі видаляються (prune_batches).
WINDOW_DAYS = 700
start_date = yesterday - timedelta(days=WINDOW_DAYS - 1)

# формати з часом
start = f"{start_date:%Y-%m-%d} 00:00:00"
end   = f"{yesterday:%Y-%m-%d} 23:59:59"

# формати без часу
start_date_str = f"{start_date:%Y-%m-%d}"
end_date_str   = f"{yesterday:%Y-%m-%d}"

# Ковзні sigma / corr / beta: для КОЖНОГО дня рахуються на ROLL_WINDOW попередніх торгових днях
# (дні з даними у ряді гепів, без днів звітів), а не на календарних.
ROLL_WINDOW = 60
# запас календарних днів перед вікном, щоб уже перший день вікна мав ROLL_WINDOW днів історії
LOOKBACK_DAYS = 130
gaps_start_date_str = f"{start_date - timedelta(days=LOOKBACK_DAYS):%Y-%m-%d}"

# вивід
print("🕓 Повні дати (з часом):")
print("start =", start)
print("end   =", end)
print("\n📅 Короткі дати (без часу):")
print("start_date_str =", start_date_str)
print("end_date_str   =", end_date_str)
print("gaps_start_date_str =", gaps_start_date_str, f"(+{LOOKBACK_DAYS} днів історії), ROLL_WINDOW =", ROLL_WINDOW)


🕓 Повні дати (з часом):
start = 2024-02-18 00:00:00
end   = 2026-11-12 23:59:59

📅 Короткі дати (без часу):
start_date_str = 2024-02-18
end_date_str   = 2026-11-12
gaps_start_date_str = 2023-10-11 (+130 днів історії), ROLL_WINDOW = 60


In [18]:
import datetime

yesterday = (datetime.date.today() - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

params = {
    "date_as_of": yesterday,
    "format": "json_records"
}

df = DatumApi.data_request("/calculations/median_premarket_value_traded_ex_finr_3m", params)

print(df.head())
print("Рядків:", len(df))


  ticker      value
0      A  111513.82
1     AA  822884.78
2    AAA       0.00
3   AAAA       0.00
4   AAAB       0.00
Рядків: 13666


In [19]:
def filter_nonempty(df: pd.DataFrame) -> pd.DataFrame:
    """
    Повертає копію датафрейму лише з тими рядками,
    де колонка 'value' не порожня і не дорівнює нулю.
    """
    if "value" not in df.columns:
        raise ValueError("У датафреймі немає колонки 'value'")
    v = pd.to_numeric(df["value"], errors="coerce")  # '' і текст -> NaN
    return df[v.notna() & (v != 0)].copy()


# --- крок 1: відсіюємо рядки з порожнім 'value'
df_filtered = filter_nonempty(df)

# --- крок 2: тягнемо дані по тікерах
ticker_params = {
    'fields': 'lvl3,market_cap',
    'active': True,
    'listed': True
}
tickers_df = DatumApi.data_request('/tickers', ticker_params)
# жодної фільтрації по ticker/lvl3/market_cap; лише прибираємо дублікати тикерів,
# щоб merge не розмножував рядки
tickers_df = tickers_df.drop_duplicates(subset="ticker")

# --- крок 3: додаємо потрібні колонки у df_filtered
TICKERS = df_filtered.merge(
    tickers_df,
    how='left',
    on='ticker'
)

print(f"Після фільтрації залишилось: {len(TICKERS):,} рядків")
TICKERS.head()


Після фільтрації залишилось: 7,140 рядків


,ticker,value,lvl3,market_cap
0,A,111513.82,Medical Equipment & Devices,48776.0
1,AA,822884.78,Metals & Mining,10677.0
2,AAAU,473398.57,None,NaN
3,AACBR,1734.90,NaN,NaN
4,AACG,32.54,Software,33.0


In [20]:
def get_unique_lvl2(df: pd.DataFrame) -> list:
    """
    Повертає список унікальних значень з колонки 'lvl3' у датафреймі.
    Пропуски (NaN) ігноруються.
    """
    if "lvl3" not in df.columns:
        raise ValueError("У датафреймі немає колонки 'lvl3'")
    
    unique_vals = sorted(df["lvl3"].dropna().unique().tolist())
    print(f"🔹 Знайдено {len(unique_vals)} унікальних значень у колонці 'lvl3'")
    return unique_vals


In [21]:
unique_lvl2 = get_unique_lvl2(TICKERS)
unique_lvl2


🔹 Знайдено 60 унікальних значень у колонці 'lvl3'


['Advertising & Marketing',
 'Aerospace & Defense',
 'Apparel & Textile Products',
 'Asset Management',
 'Automotive',
 'Banking',
 'Beverages',
 'Biotech & Pharma',
 'Cable & Satellite',
 'Chemicals',
 'Commercial Support Services',
 'Construction Materials',
 'Consumer Services',
 'Containers & Packaging',
 'Diversified Industrials',
 'E-Commerce Discretionary',
 'Electric Utilities',
 'Electrical Equipment',
 'Engineering & Construction',
 'Entertainment Content',
 'Food',
 'Forestry, Paper & Wood Products',
 'Gas & Water Utilities',
 'Health Care Facilities & Svcs',
 'Home & Office Products',
 'Home Construction',
 'Household Products',
 'IT Services',
 'Industrial Intermediate Prod',
 'Industrial Support Services',
 'Institutional Financial Svcs',
 'Insurance',
 'Internet Media & Services',
 'Investment Management',
 'Leisure Facilities & Services',
 'Leisure Products',
 'Machinery',
 'Medical Equipment & Devices',
 'Metals & Mining',
 'Oil & Gas Services & Equip',
 'Oil & Gas Sup

In [22]:
sector_to_etf = {
    # --- Finance ---
    "Asset Management": ["XLF", "IBIT"],
    "Banking": ["XLF", "KRE", "IBIT"],
    "Institutional Financial Svcs": ["XLF", "IBIT"],
    "Insurance": ["XLF"],
    "Specialty Finance": ["XLF", "IBIT"],

    # --- Tech / Internet / Growth ---
    "Software": ["IGV"],
    "IT Services": ["QQQ"],
    "Semiconductors": ["SOXX", "DRAM"],
    "Technology Hardware": ["QQQ", "SOXX"],
    "Internet Media & Services": ["QQQ", "KWEB", "FXI"],
    "E-Commerce Discretionary": ["QQQ", "KWEB", "XRT"],
    "Cable & Satellite": ["SPY", "IWM"],
    "Telecommunications": ["QQQ"],
    "Publishing & Broadcasting": ["QQQ"],
    "Entertainment Content": ["QQQ", "ARKK"],
    "Advertising & Marketing": ["QQQ"],

    # --- Industrials ---
    "Aerospace & Defense": ["NASA", "ITA"],
    "Diversified Industrials": ["XLB", "IWM"],
    "Industrial Intermediate Prod": ["XLB", "IWM"],
    "Industrial Support Services": ["IWM", "SPY"],
    "Engineering & Construction": ["XLB", "IWM"],
    "Machinery": ["XLB", "IWM"],
    "Transportation & Logistics": ["IWM", "SPY"],
    "Transportation Equipment": ["IWM", "SPY"],
    "Commercial Support Services": ["IWM", "SPY"],
    "Containers & Packaging": ["XLB", "IWM"],
    "Electrical Equipment": ["QQQ", "XLB"],

    # --- Energy / Materials / Commodities ---
    "Oil & Gas Services & Equip": ["XOP", "XLE", "UNG"],
    "Oil & Gas Supply Chain": ["XOP", "XLE", "UNG"],
    "Elec & Gas Marketing & Trading": ["XLU", "XLE"],
    "Electric Utilities": ["XLU"],
    "Gas & Water Utilities": ["XLU"],
    "Renewable Energy": ["XLE", "ARKK"],
    "Chemicals": ["XLB", "URA"],
    "Metals & Mining": ["GDX", "FCX", "COPX", "URA", "SLV"],
    "Steel": ["XLB", "GDX"],
    "Construction Materials": ["XLB", "IWM"],
    "Forestry, Paper & Wood Products": ["XLB", "IWM"],

    # --- Health Care ---
    "Biotech & Pharma": ["XBI"],
    "Medical Equipment & Devices": ["XLV"],
    "Health Care Facilities & Svcs": ["XLV", "IWM"],

    # --- Consumer Staples / Discretionary ---
    "Food": ["XLP"],
    "Beverages": ["XLP"],
    "Household Products": ["XLP"],
    "Tobacco & Cannabis": ["XLP"],
    "Retail - Consumer Staples": ["XLP", "XRT"],
    "Wholesale - Consumer Staples": ["XLP"],

    "Retail - Discretionary": ["XRT", "IWM", "SPY"],
    "Wholesale - Discretionary": ["XRT", "IWM"],
    "Consumer Services": ["IWM", "SPY"],
    "Leisure Facilities & Services": ["IWM", "SPY"],
    "Leisure Products": ["XRT", "IWM"],
    "Apparel & Textile Products": ["XRT", "IWM"],
    "Automotive": ["QQQ", "IWM"],
    "Home & Office Products": ["XRT", "IWM"],
    "Home Construction": ["XLB", "IWM"],

    # --- Real Estate ---
    "REIT": ["IWM", "SPY"],
    "Real Estate Owners & Developers": ["IWM", "SPY"],
    "Real Estate Services": ["IWM", "SPY"],
}

In [23]:
# ------------------------------------------------

In [24]:
import numpy as np
import pandas as pd


def calc_corr_beta_from_joined_gaps(df_pair: pd.DataFrame, x_col: str = "gap_x", y_col: str = "gap_y"):
    """
    Local corr/beta calculation without /calculations/corr_beta API.

    corr = Pearson(x, y)
    beta = cov(x, y) / var(y)
    """
    d = df_pair[[x_col, y_col]].copy()
    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d[y_col] = pd.to_numeric(d[y_col], errors="coerce")
    d = d.dropna()

    n = len(d)
    if n < 2:
        return np.nan, np.nan, n

    var_y = float(d[y_col].var(ddof=1))
    corr = float(d[x_col].corr(d[y_col]))

    if np.isnan(var_y) or var_y == 0:
        return corr, np.nan, n

    cov_xy = float(d[[x_col, y_col]].cov(ddof=1).iloc[0, 1])
    beta = cov_xy / var_y
    return corr, beta, n


def corr_beta_best_match_from_gaps(
    stock_gap_df: pd.DataFrame,
    etf_gap_df: pd.DataFrame,
    x_tickers,
    y_tickers,
    *,
    min_overlap: int = 20,
    output_path: str = None,
):
    """
    Pick best Y per X using local gaps.

    stock_gap_df: ['ticker','date','gap']
    etf_gap_df:   ['ticker','date','gap']
    """
    sx = stock_gap_df.copy()
    sy = etf_gap_df.copy()

    for df in (sx, sy):
        if not {"ticker", "date", "gap"}.issubset(df.columns):
            raise KeyError("Expected columns ['ticker','date','gap'] in gap data")
        df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
        df["gap"] = pd.to_numeric(df["gap"], errors="coerce")

    s_map = {t: g[["date", "gap"]].copy() for t, g in sx.groupby("ticker")}
    e_map = {t: g[["date", "gap"]].copy() for t, g in sy.groupby("ticker")}

    rows_all = []
    rows_best = []

    for x in [str(v).upper() for v in x_tickers]:
        g_x = s_map.get(x)
        if g_x is None or g_x.empty:
            continue

        best = None
        for y in [str(v).upper() for v in y_tickers]:
            g_y = e_map.get(y)
            if g_y is None or g_y.empty:
                continue

            m = g_x.merge(g_y, on="date", how="inner", suffixes=("_x", "_y")).dropna()
            corr, beta, n = calc_corr_beta_from_joined_gaps(m, x_col="gap_x", y_col="gap_y")
            if n < int(min_overlap) or pd.isna(corr):
                continue

            row = {
                "x_ticker": x,
                "y_ticker": y,
                "corr": corr,
                "beta": beta,
                "n_overlap": n,
            }
            rows_all.append(row)
            if best is None or row["corr"] > best["corr"]:
                best = row

        if best is not None:
            rows_best.append({
                "x_ticker": best["x_ticker"],
                "best_y_ticker": best["y_ticker"],
                "best_corr": best["corr"],
                "beta_with_best": best["beta"],
                "n_overlap": best["n_overlap"],
            })

    df_all = pd.DataFrame(rows_all)
    df_pairs = pd.DataFrame(rows_best)

    if df_pairs.empty:
        df_pairs = pd.DataFrame(columns=["x_ticker", "best_y_ticker", "best_corr", "beta_with_best", "n_overlap"])

    if output_path:
        df_pairs.to_csv(output_path, index=False)

    return df_all, df_pairs



In [25]:
def corr_beta_best_etf_by_lvl3(
    tickers_df,
    sector_to_etf,
    sector_col="lvl3",
    ticker_col="ticker",
    output_path="CRACEN/work/x_to_best_etf_by_lvl3.csv",
    bench_path="ticker_bench.csv",
    fallback_etfs=None,
    start_date=None,
    end_date=None,
    last_n_reports=20,
    cap_q=0.8,
    min_hist_for_cap=20,
    min_overlap=20,
    max_workers_net=12,        # 64 потоки без пауз давали суцільні 429
    max_fail_ratio=0.15,      # 4.4% остаточних відмов виміряно як НОРМУ для цього
                              # API навіть з retry; бар упритул до норми валив би
                              # прогін через раз. Ловить справжню відмову, не шум.
):
    import random
    import time
    import numpy as np
    import pandas as pd
    from concurrent.futures import ThreadPoolExecutor

    if fallback_etfs is None:
        fallback_etfs = ["SPY", "QQQ", "IWM"]
    if start_date is None:
        start_date = globals().get("start_date_str")
    if end_date is None:
        end_date = globals().get("end_date_str")
    if not start_date or not end_date:
        raise ValueError("Provide start_date/end_date or define start_date_str/end_date_str")
    if sector_col not in tickers_df.columns:
        raise ValueError(f"tickers_df has no '{sector_col}' column")
    if ticker_col not in tickers_df.columns:
        raise ValueError(f"tickers_df has no '{ticker_col}' column")

    s_date, e_date = str(start_date)[:10], str(end_date)[:10]

    def _pick_col(df, candidates):
        cols_l = {c.lower().strip(): c for c in df.columns}
        return next((cols_l[c.lower()] for c in candidates if c.lower() in cols_l), None)

    _fails = {"gaps": 0, "reports": 0}

    def _request_with_retry(endpoint, params, *, max_retries=6, backoff=1.0):
        """Спільний шар з клітинки 1: keep-alive, пауза для всіх потоків після 429,
        повтори лише 429/5xx/мережевих збоїв."""
        fn = globals().get("datum_get_df")
        if fn is None:                                   # запасний шлях, якщо клітинку 1 не виконано
            return DatumApi.data_request(endpoint, params)
        return fn(endpoint, params, max_retries=max_retries, backoff=backoff)

    def _fetch_gap(t):
        try:
            dfg = _request_with_retry("/daily/gaps", {
                "ticker": t, "start_date": s_date, "end_date": e_date, "format": "json_records",
            })
            if dfg is None or dfg.empty:
                return None
            dcol = _pick_col(dfg, ["date", "move_date", "dt", "datetime", "day"])
            if dcol is None or "gap" not in dfg.columns:
                return None
            out = pd.DataFrame({
                "ticker": t,
                "date": pd.to_datetime(dfg[dcol], errors="coerce").dt.strftime("%Y-%m-%d"),
                "gap": pd.to_numeric(dfg["gap"], errors="coerce"),
            }).dropna()
            return out if not out.empty else None
        except Exception as ex:
            _fails["gaps"] += 1
            print(f"WARN gaps fail {t}: {ex}")
            return None

    def _fetch_report(t):
        try:
            rep = _request_with_retry("/reports", {
                "ticker": t, "start_move_date": s_date, "end_move_date": e_date,
            })
            if rep is None or rep.empty:
                return None
            dcol = _pick_col(rep, ["report_date", "move_date", "date", "dt", "datetime", "published_at", "created_at"])
            if dcol is None:
                return None
            out = pd.DataFrame({
                "ticker": t,
                "date": pd.to_datetime(rep[dcol], errors="coerce").dt.strftime("%Y-%m-%d"),
            }).dropna().sort_values("date").tail(int(last_n_reports))
            return out if not out.empty else None
        except Exception as ex:
            _fails["reports"] += 1
            print(f"WARN reports fail {t}: {ex}")
            return None

    # ── prepare ──────────────────────────────────────────────────────────────
    df_clean = tickers_df[[ticker_col, sector_col]].drop_duplicates().copy()
    df_clean[ticker_col] = df_clean[ticker_col].astype(str).str.strip().str.upper()
    df_clean[sector_col] = df_clean[sector_col].fillna("UNKNOWN").astype(str)

    stocks = sorted(df_clean[ticker_col].dropna().unique())
    etf_candidates = set(fallback_etfs)
    for sec in df_clean[sector_col].unique():
        etf_candidates.update(sector_to_etf.get(sec, fallback_etfs) or fallback_etfs)
    etf_candidates = sorted(str(x).upper() for x in etf_candidates)

    print(f"START gaps: stocks={len(stocks)} etfs={len(etf_candidates)} window={s_date}..{e_date}")

    # ── parallel fetch: all gaps + reports in one pool ────────────────────────
    all_gap_tickers = sorted(set(stocks) | set(etf_candidates))
    with ThreadPoolExecutor(max_workers=min(max_workers_net,
                                            len(all_gap_tickers) + len(stocks))) as ex:
        gap_futs  = {t: ex.submit(_fetch_gap,    t) for t in all_gap_tickers}
        rep_futs  = {t: ex.submit(_fetch_report, t) for t in stocks}

    gap_res = {t: f.result() for t, f in gap_futs.items()}
    rep_res = {t: f.result() for t, f in rep_futs.items()}

    _total = len(gap_futs) + len(rep_futs)
    _failed = _fails["gaps"] + _fails["reports"]
    if _total and _failed / _total > max_fail_ratio:
        raise RuntimeError(
            f"API відмовив на {_failed} з {_total} запитів "
            f"({_failed / _total * 100:.1f}% > {max_fail_ratio * 100:.0f}%). "
            "Далі вийшла б порожня або обрізана мапа bench/corr/beta."
        )

    def _concat(parts, cols):
        valid = [p for p in parts if p is not None]
        return pd.concat(valid, ignore_index=True) if valid else pd.DataFrame(columns=cols)

    stock_gaps = _concat([gap_res.get(t) for t in stocks],         ["ticker", "date", "gap"])
    etf_gaps   = _concat([gap_res.get(t) for t in etf_candidates], ["ticker", "date", "gap"])
    reports    = _concat([rep_res.get(t) for t in stocks],          ["ticker", "date"]).drop_duplicates()

    # ── exclude report days ──────────────────────────────────────────────────
    if not reports.empty and not stock_gaps.empty:
        m = stock_gaps.merge(reports.assign(_rep=1), on=["ticker", "date"], how="left")
        stock_gaps = m[m["_rep"].isna()][["ticker", "date", "gap"]].copy()

    # ── drop anomalies (vectorised) ──────────────────────────────────────────
    if not stock_gaps.empty:
        stock_gaps["gap"] = pd.to_numeric(stock_gaps["gap"], errors="coerce")
        abs_gap = stock_gaps["gap"].abs()
        cnts = stock_gaps.groupby("ticker")["gap"].transform("count")
        eligible = cnts >= min_hist_for_cap
        if eligible.any():
            caps = (stock_gaps[eligible]
                    .groupby("ticker")["gap"]
                    .agg(lambda x: np.quantile(x.abs(), cap_q)))
            cap_mapped = stock_gaps["ticker"].map(caps)
            stock_gaps = stock_gaps[(~eligible) | (abs_gap <= cap_mapped)].reset_index(drop=True)

    # ── build ETF numpy matrix: rows=dates, cols=etfs ───────────────────────
    if not etf_gaps.empty:
        etf_pivot = etf_gaps.pivot_table(index="date", columns="ticker", values="gap", aggfunc="first")
        date_to_row = {d: i for i, d in enumerate(etf_pivot.index)}
        etf_to_col  = {c: i for i, c in enumerate(etf_pivot.columns)}
        etf_arr     = etf_pivot.to_numpy(dtype=float)
    else:
        date_to_row = {}; etf_to_col = {}; etf_arr = np.empty((0, 0))

    stock_map = (
        {t: g.set_index("date")["gap"] for t, g in stock_gaps.groupby("ticker")}
        if not stock_gaps.empty else {}
    )

    # ── corr/beta loop ───────────────────────────────────────────────────────
    rows_all = []
    rows_best = []
    sectors_with_fallback = []

    for sec in sorted(df_clean[sector_col].unique()):
        etfs_raw = sector_to_etf.get(sec)
        if not etfs_raw:
            etfs_raw = list(fallback_etfs)
            sectors_with_fallback.append(sec)
        etfs = [str(e).upper() for e in etfs_raw]
        etfs_ok  = [e for e in etfs if e in etf_to_col]
        col_idxs = [etf_to_col[e] for e in etfs_ok]
        if not col_idxs:
            continue

        for st in sorted(df_clean.loc[df_clean[sector_col] == sec, ticker_col].unique()):
            sx = stock_map.get(st)
            if sx is None or sx.empty:
                continue

            st_dates = [d for d in sx.index if d in date_to_row]
            if not st_dates:
                continue

            row_idxs = [date_to_row[d] for d in st_dates]
            x_vals   = sx[st_dates].to_numpy(dtype=float)
            y_mat    = etf_arr[np.ix_(row_idxs, col_idxs)]   # (n_dates, n_etfs)
            valid_x  = ~np.isnan(x_vals)

            best_corr = -np.inf
            best_etf  = None
            best_beta = np.nan
            best_n    = 0

            for j, etf in enumerate(etfs_ok):
                mask = valid_x & ~np.isnan(y_mat[:, j])
                n = int(mask.sum())
                if n < min_overlap:
                    continue

                xv = x_vals[mask];  yv = y_mat[mask, j]
                xc = xv - xv.mean(); yc = yv - yv.mean()
                var_y = float((yc * yc).sum() / (n - 1))
                if var_y == 0 or np.isnan(var_y):
                    continue
                cov_xy = float((xc * yc).sum() / (n - 1))
                std_x  = float(np.sqrt((xc * xc).sum() / (n - 1)))
                if std_x == 0:
                    continue
                corr = cov_xy / (std_x * np.sqrt(var_y))
                beta = cov_xy / var_y

                rows_all.append({"x_ticker": st, "y_ticker": etf, "corr": corr, "beta": beta, "lvl3": sec, "n_overlap": n})

                if corr > best_corr:
                    best_corr, best_etf, best_beta, best_n = corr, etf, beta, n

            if best_etf is not None:
                rows_best.append({"x_ticker": st, "best_y_ticker": best_etf, "best_corr": best_corr,
                                  "beta_with_best": best_beta, "lvl3": sec, "n_overlap": best_n})

    # ── output ───────────────────────────────────────────────────────────────
    df_all   = pd.DataFrame(rows_all)
    df_pairs = pd.DataFrame(rows_best) if rows_best else pd.DataFrame(
        columns=["x_ticker", "best_y_ticker", "best_corr", "beta_with_best", "lvl3", "n_overlap"])

    df_pairs.to_csv(output_path, index=False)
    df_bench = df_pairs.rename(columns={"x_ticker": "ticker", "best_y_ticker": "benchmark"})[["ticker", "benchmark"]]
    df_bench.to_csv(bench_path, index=False)

    print(f"DONE pairs: {len(df_pairs)} -> {output_path}")
    print(f"DONE bench map: {len(df_bench)} -> {bench_path}")
    if sectors_with_fallback:
        print("Fallback ETF used for sectors:", ", ".join(map(str, sectors_with_fallback[:20])))

    return df_all, df_pairs


In [26]:
# Local corr/beta: stock gaps vs ETF gaps, excluding report days and stock outliers
df_all, df_pairs = corr_beta_best_etf_by_lvl3(
    tickers_df=TICKERS,
    sector_to_etf=sector_to_etf,
    sector_col="lvl3",
    ticker_col="ticker",
    output_path=str(WORK_DIR / "x_to_best_etf_by_lvl3.csv"),
    bench_path=str(WORK_DIR / "ticker_bench_.csv"),
    start_date=start_date_str,
    end_date=end_date_str,
    last_n_reports=20,
    cap_q=0.8,
    min_hist_for_cap=20,
    min_overlap=20,
)

# Prepare parquet for downstream merge (cell 31)
corr_beta_pairs = df_pairs.rename(columns={
    "best_y_ticker": "y_ticker",
    "best_corr": "corr",
    "beta_with_best": "beta",
})[["x_ticker", "y_ticker", "corr", "beta"]].copy()

corr_beta_pairs.to_parquet(str(WORK_DIR / "corr_beta_pairs.parquet"), index=False)

print(df_all.head())
print("Rows in raw stock-etf pairs:", len(df_all))
print(df_pairs.head())
print("Saved:", str(WORK_DIR / "corr_beta_pairs.parquet"))



START gaps: stocks=7140 etfs=28 window=2024-02-18..2026-11-12
DONE pairs: 7108 -> C:\datum-api-examples-main\OriON\CRACEN\work\x_to_best_etf_by_lvl3.csv
DONE bench map: 7108 -> C:\datum-api-examples-main\OriON\CRACEN\work\ticker_bench_.csv
Fallback ETF used for sectors: Investment Management, Other Financial Services, UNKNOWN
  x_ticker y_ticker      corr      beta                     lvl3  n_overlap
0      ADV      QQQ  0.232630  0.244242  Advertising & Marketing        540
1     AEHL      QQQ  0.127700  0.327820  Advertising & Marketing        549
2      APP      QQQ  0.473392  0.773991  Advertising & Marketing        541
3     BAOS      QQQ  0.143170  0.339135  Advertising & Marketing        545
4     CDLX      QQQ  0.358732  0.532218  Advertising & Marketing        540
Rows in raw stock-etf pairs: 16075
  x_ticker best_y_ticker  best_corr  beta_with_best                     lvl3  \
0      ADV           QQQ   0.232630        0.244242  Advertising & Marketing   
1     AEHL           

In [27]:
# ================== FAST INTRADAY SUITE w/ BLUE OCEAN (updated + rolling prune) ==================
import os, gc, time, glob, re, csv, uuid, random
import pandas as pd
from datetime import timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED
from typing import List, Optional, Tuple
import threading, requests

# ---------- utils ----------
def _p(*args, **kwargs):
    kwargs.setdefault("flush", True)
    print(*args, **kwargs)

_PSUTIL = False
try:
    import psutil
    _PSUTIL = True
except Exception:
    psutil = None

def _proc_rss_mb() -> float:
    if _PSUTIL:
        try:
            return psutil.Process(os.getpid()).memory_info().rss / (1024**2)
        except Exception:
            pass
    try:
        import resource
        ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        if ru > 10**9:
            return ru / (1024**2)
        return ru / 1024.0
    except Exception:
        pass
    try:
        import tracemalloc
        if not tracemalloc.is_tracing():
            tracemalloc.start()
        cur, _ = tracemalloc.get_traced_memory()
        return cur / (1024**2)
    except Exception:
        return float('nan')

def _mem_str(prefix: str = "") -> str:
    mb = _proc_rss_mb()
    if mb != mb:
        return f"{prefix}🧠 RAM: n/a"
    return f"{prefix}🧠 RAM: {mb:,.1f} MB"

def _ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def _fmt_eta(sec: float) -> str:
    sec = max(0, int(sec))
    d, r = divmod(sec, 86400)
    h, r = divmod(r, 3600)
    m, s = divmod(r, 60)
    if d > 0: return f"{d}d {h:02d}:{m:02d}:{s:02d}"
    if h > 0: return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def _next_batch_index(out_dir: str, out_prefix: str) -> int:
    pat_csv = os.path.join(out_dir, f"{out_prefix}_*.csv")
    pat_gz  = os.path.join(out_dir, f"{out_prefix}_*.csv.gz")
    files = glob.glob(pat_csv) + glob.glob(pat_gz)
    mx = 0
    rx = re.compile(rf'^{re.escape(out_prefix)}_(?:\d{{4}}-\d{{2}}-\d{{2}}_)?(\d+)\.csv(?:\.gz)?$')
    for f in files:
        m = rx.match(os.path.basename(f))
        if m:
            mx = max(mx, int(m.group(1)))
    return mx + 1 if mx > 0 else 1

# ---------- manifest helpers (old ticker-level) ----------
def _manifest_path(out_dir: str) -> str:
    return os.path.join(out_dir, "done_tickers.txt")

def _load_done_tickers_manifest(out_dir: str) -> Optional[set]:
    path = _manifest_path(out_dir)
    if not os.path.exists(path):
        return None
    done = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t = line.strip()
            if t:
                done.add(t)
    return done

def _append_done_tickers_manifest(out_dir: str, tickers: List[str]):
    if not tickers:
        return
    path = _manifest_path(out_dir)
    existing = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if s:
                    existing.add(s)
    new_ones = [t for t in set(tickers) if t not in existing]
    if not new_ones:
        return
    with open(path, "a", encoding="utf-8") as f:
        for t in new_ones:
            f.write(t + "\n")

# ---------- day-manifest helpers (NEW) ----------
def _manifest_days_path(out_dir: str) -> str:
    return os.path.join(out_dir, "done_days.csv")

def _load_done_days_manifest(out_dir: str) -> dict:
    """
    Return dict: ticker -> set({'YYYY-MM-DD', ...})
    """
    path = _manifest_days_path(out_dir)
    if not os.path.exists(path):
        return {}
    done = {}
    with open(path, "r", encoding="utf-8", newline="") as f:
        rdr = csv.reader(f)
        header = next(rdr, None)
        col_idx = {"ticker": 0, "date": 1}
        if header and "ticker" in header and "date" in header:
            col_idx["ticker"] = header.index("ticker")
            col_idx["date"]   = header.index("date")
        else:
            f.seek(0); rdr = csv.reader(f)
        for row in rdr:
            if not row:
                continue
            tk = row[col_idx["ticker"]].strip()
            dt = row[col_idx["date"]].strip()
            if tk and dt:
                done.setdefault(tk, set()).add(dt)
    return done

def _append_done_days_manifest(out_dir: str, df: pd.DataFrame):
    if df is None or df.empty or "dt" not in df.columns or "ticker" not in df.columns:
        return
    path = _manifest_days_path(out_dir)
    _ensure_dir(out_dir)
    tmp = pd.DataFrame({
        "ticker": df["ticker"].astype(str).str.strip().str.upper(),
        "date":   df["dt"].astype(str).str.slice(0, 10),
    }).drop_duplicates()
    tmp.to_csv(path, mode="a", header=not os.path.exists(path), index=False)

def _prune_done_days_manifest_slow(out_dir: str, start_date_str: str, end_date_str: str) -> dict:
    """
    Rolling-window prune for done_days.csv:
    - keep only rows where start_date_str <= date <= end_date_str
    - returns refreshed dict: ticker -> set(dates)
    Safe to call even if file doesn't exist.
    """
    path = _manifest_days_path(out_dir)
    if not os.path.exists(path):
        return {}

    # normalize inputs
    start_date_str = str(start_date_str)[:10]
    end_date_str   = str(end_date_str)[:10]
    if start_date_str > end_date_str:
        # swap just in case
        start_date_str, end_date_str = end_date_str, start_date_str

    rows = []
    with open(path, "r", encoding="utf-8", newline="") as f:
        rdr = csv.reader(f)
        header = next(rdr, None)
        col_idx = {"ticker": 0, "date": 1}
        has_header = bool(header and "ticker" in header and "date" in header)
        if has_header:
            col_idx["ticker"] = header.index("ticker")
            col_idx["date"]   = header.index("date")
        else:
            # no header; treat first row as data
            if header:
                rdr = [header] + list(rdr)  # include first row back as data
            else:
                rdr = []
        for row in rdr:
            if not row:
                continue
            try:
                tk = str(row[col_idx["ticker"]]).strip().upper()
                dt = str(row[col_idx["date"]]).strip()[:10]
            except Exception:
                continue
            if not tk or not dt:
                continue
            if start_date_str <= dt <= end_date_str:
                rows.append((tk, dt))

    # rewrite file (dedup)
    _ensure_dir(out_dir)
    tmp_path = path + ".tmp"
    seen = set()
    with open(tmp_path, "w", encoding="utf-8", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["ticker","date"])
        for tk, dt in rows:
            key = (tk, dt)
            if key in seen:
                continue
            seen.add(key)
            wr.writerow([tk, dt])
    os.replace(tmp_path, path)

    # build dict cache
    done = {}
    for tk, dt in seen:
        done.setdefault(tk, set()).add(dt)
    return done

# ---------- rolling window: видалення старих днів із батчів ----------
def _batch_rx(prefix: str):
    # intraday_0007.csv[.gz]  (старий батч-суміш)  або  intraday_2026-11-12_0007.csv[.gz]  (партиція за днем)
    return re.compile(rf'^{re.escape(prefix)}_(?:(\d{{4}}-\d{{2}}-\d{{2}})_)?(\d+)\.csv(?:\.gz)?$')

def _legacy_dt_range(path: str):
    """(min_date, max_date) 'YYYY-MM-DD' для батча-суміші; (None, None) якщо порожній."""
    try:
        import pyarrow as pa, pyarrow.csv as pacsv, pyarrow.compute as pc
        t = pacsv.read_csv(path, convert_options=pacsv.ConvertOptions(
            include_columns=["dt"], column_types={"dt": pa.string()}))
        if t.num_rows == 0:
            return None, None
        mm = pc.min_max(pc.utf8_slice_codeunits(t.column("dt"), 0, 10)).as_py()
        return mm["min"], mm["max"]
    except Exception:
        s = pd.read_csv(path, usecols=["dt"], dtype=str)["dt"].dropna().str.slice(0, 10)
        return (None, None) if s.empty else (s.min(), s.max())

def _rewrite_legacy_from(path: str, start_date_str: str):
    """Переписує батч-суміш на місці, лишаючи рядки з датою >= start_date_str (тексти без змін)."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    keep = df["dt"].str.slice(0, 10) >= start_date_str
    tmp = path + ".tmp"
    df[keep].to_csv(tmp, index=False, compression="gzip" if path.endswith(".gz") else None)
    os.replace(tmp, path)

def prune_batches(out_dir: str, out_prefix: str, start_date_str: str, verbose: bool = True) -> dict:
    """
    Тримає в out_dir лише дні >= start_date_str.
    - файл-партиція за днем (intraday_YYYY-MM-DD_NNNN): дата в імені -> старий просто видаляється;
    - старий файл-суміш (intraday_NNNN): діапазон дат береться з кешу batches_meta.csv
      (рахується один раз), повністю старий видаляється, частково старий переписується на місці.
    - якщо даних немає взагалі, а маніфест днів лишився - маніфест видаляється, інакше він
      «брехав» би, що дні вже завантажені.
    """
    start = str(start_date_str)[:10]
    rx = _batch_rx(out_prefix)
    files = [f for f in sorted(glob.glob(os.path.join(out_dir, f"{out_prefix}_*.csv*")))
             if rx.match(os.path.basename(f))]
    stats = {"removed": 0, "rewritten": 0, "kept": 0}

    if not files:
        mp = _manifest_days_path(out_dir)
        if os.path.exists(mp):
            os.remove(mp)
            if verbose:
                _p("   🧹 даних немає, а done_days.csv лишився -> маніфест скинуто")
        return stats

    meta_path = os.path.join(out_dir, "batches_meta.csv")
    meta = {}
    if os.path.exists(meta_path):
        try:
            mdf = pd.read_csv(meta_path, dtype=str, keep_default_na=False)
            for r in mdf.itertuples(index=False):
                meta[r.file] = (int(r.size), r.min_date or None, r.max_date or None)
        except Exception:
            meta = {}

    legacy = []
    for f in files:
        m = rx.match(os.path.basename(f))
        if m.group(1):                      # партиція за днем
            if m.group(1) < start:
                os.remove(f)
                stats["removed"] += 1
            else:
                stats["kept"] += 1
        else:
            legacy.append(f)

    unknown = [f for f in legacy
               if meta.get(os.path.basename(f), (None,))[0] != os.path.getsize(f)]
    if unknown:
        if verbose:
            _p(f"   🧹 індексую {len(unknown)} старих батчів (одноразово)...")
        with ThreadPoolExecutor(max_workers=8) as ex:
            for f, rng in zip(unknown, ex.map(_legacy_dt_range, unknown)):
                meta[os.path.basename(f)] = (os.path.getsize(f), rng[0], rng[1])

    new_meta = {}
    for f in legacy:
        name = os.path.basename(f)
        size, mn, mx = meta[name]
        if mx is None or mx < start:
            os.remove(f)
            stats["removed"] += 1
            continue
        if mn is not None and mn < start:
            _rewrite_legacy_from(f, start)
            mn, mx = _legacy_dt_range(f)
            if mx is None:
                os.remove(f)
                stats["removed"] += 1
                continue
            stats["rewritten"] += 1
        else:
            stats["kept"] += 1
        new_meta[name] = (os.path.getsize(f), mn, mx)

    pd.DataFrame(
        [(k, v[0], v[1] or "", v[2] or "") for k, v in new_meta.items()],
        columns=["file", "size", "min_date", "max_date"],
    ).to_csv(meta_path, index=False)

    if verbose:
        _p(f"   🧹 вікно від {start}: видалено файлів {stats['removed']}, "
           f"переписано {stats['rewritten']}, лишилось {len(files) - stats['removed']}")
    return stats

def prune_done_days_manifest(out_dir: str, start_date_str: str, end_date_str: str) -> dict:
    """Ковзне вікно для done_days.csv (pandas: мільйони рядків за секунди). Повертає ticker -> set(дат)."""
    path = _manifest_days_path(out_dir)
    if not os.path.exists(path):
        return {}
    a, b = str(start_date_str)[:10], str(end_date_str)[:10]
    if a > b:
        a, b = b, a
    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
        if list(df.columns[:2]) != ["ticker", "date"]:
            raise ValueError("manifest without header")
        df["ticker"] = df["ticker"].str.strip().str.upper()
        df["date"] = df["date"].str.strip().str.slice(0, 10)
        df = df[(df["ticker"] != "") & (df["date"] >= a) & (df["date"] <= b)]
        df = df.drop_duplicates(["ticker", "date"])
        tmp = path + ".tmp"
        df[["ticker", "date"]].to_csv(tmp, index=False)
        os.replace(tmp, path)
        return {tk: set(g) for tk, g in df.groupby("ticker")["date"]}
    except Exception:
        return _prune_done_days_manifest_slow(out_dir, start_date_str, end_date_str)

def compact_day_parts(out_dir: str, out_prefix: str, gzip: bool = True, workers: int = 4,
                      verbose: bool = True) -> int:
    """Зливає всі частини одного дня (intraday_YYYY-MM-DD_NNNN) в один файл intraday_YYYY-MM-DD_0000:
    прибирає дублікати (ticker, dt) і сортує за (ticker, dt). Ідемпотентно; краш посередині безпечний
    (файл пишеться атомарно, а повторний запуск просто зіллє залишки ще раз)."""
    rx = _batch_rx(out_prefix)
    by_day = {}
    for f in glob.glob(os.path.join(out_dir, f"{out_prefix}_*.csv*")):
        m = rx.match(os.path.basename(f))
        if m and m.group(1):
            by_day.setdefault(m.group(1), []).append((int(m.group(2)), f))
    todo = {d: [f for _, f in sorted(v)] for d, v in by_day.items() if len(v) > 1}
    if not todo:
        return 0

    def _merge(day):
        fs = todo[day]
        # Arrow замість pandas(dtype=str): рядки не стають python-об'єктами -> у 5-10 разів менше RAM
        import numpy as np, pyarrow as pa, pyarrow.csv as pacsv
        tabs = []
        names = pacsv.open_csv(fs[0]).schema.names      # схема однакова в усіх частинах: відкриваємо один раз
        _co = pacsv.ConvertOptions(column_types={n: pa.string() for n in names})
        for f in fs:
            tabs.append(pacsv.read_csv(f, convert_options=_co))
        tbl = pa.concat_tables(tabs)
        del tabs
        tbl = tbl.append_column("_i", pa.array(np.arange(tbl.num_rows, dtype="int64")))
        last = tbl.group_by(["ticker", "dt"]).aggregate([("_i", "max")]).column("_i_max")   # keep="last"
        tbl = tbl.take(last).sort_by([("ticker", "ascending"), ("dt", "ascending")]).drop(["_i"])
        target = os.path.join(out_dir, f"{out_prefix}_{day}_0000.csv" + (".gz" if gzip else ""))
        tmp = target + ".tmp"
        try:
            wopt = pacsv.WriteOptions(quoting_style="needed")
        except TypeError:
            wopt = pacsv.WriteOptions()
        if gzip:
            with pa.CompressedOutputStream(tmp, "gzip") as sink:
                pacsv.write_csv(tbl, sink, wopt)
        else:
            pacsv.write_csv(tbl, tmp, wopt)
        n = tbl.num_rows
        del tbl, last
        os.replace(tmp, target)
        for f in fs:
            if os.path.abspath(f) != os.path.abspath(target):
                os.remove(f)
        return n

    if verbose:
        _p(f"   🧩 зливаю частини {len(todo)} днів в один файл на день...")
    days = sorted(todo)
    with ThreadPoolExecutor(max_workers=max(1, int(workers))) as ex:
        list(ex.map(_merge, days))
    return len(days)

def verify_window(out_dir: str, out_prefix: str, start_date_str: str, end_date_str: str,
                  verbose: bool = True) -> dict:
    """Швидка перевірка цілісності вікна (лише імена файлів і маніфест): діапазон днів,
    відсутні будні (свята) та чи є найновіший день."""
    rx = _batch_rx(out_prefix)
    have = set()
    n_files = 0
    for f in glob.glob(os.path.join(out_dir, f"{out_prefix}_*.csv*")):
        m = rx.match(os.path.basename(f))
        if m:
            n_files += 1
            if m.group(1):
                have.add(m.group(1))
    mp = _manifest_days_path(out_dir)
    if os.path.exists(mp):
        try:
            have |= set(pd.read_csv(mp, usecols=["date"], dtype=str)["date"].dropna().unique())
        except Exception:
            pass
    a, b = str(start_date_str)[:10], str(end_date_str)[:10]
    weekdays = [d.strftime("%Y-%m-%d") for d in pd.date_range(a, b) if d.weekday() < 5]
    missing = [d for d in weekdays if d not in have]
    last = max(have) if have else None
    stale_from = (pd.Timestamp(b) - pd.Timedelta(days=4)).strftime("%Y-%m-%d")
    info = {"files": n_files, "first": min(have) if have else None, "last": last,
            "missing_weekdays": missing, "stale": last is None or last < stale_from}
    if verbose:
        _p(f"   📅 вікно {a}..{b}: файлів {n_files}, дані {info['first']}..{last}, "
           f"будніх днів без даних (свята?) {len(missing)}")
        if missing:
            _p("      " + ", ".join(missing[:12]) + (" ..." if len(missing) > 12 else ""))
        if info["stale"]:
            _p("   ⚠️ найновіших днів немає - перевірте помилки та запустіть ще раз")
    return info

def _dates_between(start: str, end: str) -> List[pd.Timestamp]:
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)
    if pd.isna(s) or pd.isna(e) or s > e:
        return []
    days = pd.date_range(start=s.normalize(), end=e.normalize(), freq="D")
    return list(days)

def _group_consecutive_dates(dates: List[pd.Timestamp], max_span_days: int) -> List[Tuple[str, str]]:
    """
    Build [start_dt, end_dt] ranges from a list of dates, merging consecutive days,
    capped by max_span_days per range. Datetimes are ISO to seconds.
    """
    if not dates:
        return []
    dates = sorted(pd.to_datetime(dates))
    out = []
    run_start = dates[0]
    run_end = dates[0]
    for d in dates[1:]:
        consecutive = (d - run_end).days == 1
        span_len = (d - run_start).days + 1
        if consecutive and span_len <= max_span_days:
            run_end = d
        else:
            a = run_start.strftime("%Y-%m-%d 00:00:00")
            b = (run_end + pd.Timedelta(hours=23, minutes=59, seconds=59)).strftime("%Y-%m-%d %H:%M:%S")
            out.append((a, b))
            run_start = d
            run_end = d
    a = run_start.strftime("%Y-%m-%d 00:00:00")
    b = (run_end + pd.Timedelta(hours=23, minutes=59, seconds=59)).strftime("%Y-%m-%d %H:%M:%S")
    out.append((a, b))
    return out

# ================== SMALL (low-level) FUNCTIONS ==================
def _build_ranges(start: str, end: str, chunk_days: int) -> List[Tuple[str, str]]:
    """Розбиває [start,end] на неперекривні відрізки по chunk_days днів."""
    chunk_days = max(1, int(chunk_days))
    s = pd.to_datetime(start)
    e = pd.to_datetime(end)
    if pd.isna(s) or pd.isna(e) or s > e:
        return []
    out: List[Tuple[str, str]] = []
    cur = s
    step = timedelta(days=chunk_days) - timedelta(seconds=1)
    while cur <= e:
        r_end = min(cur + step, e)
        out.append((cur.strftime("%Y-%m-%d %H:%M:%S"), r_end.strftime("%Y-%m-%d %H:%M:%S")))
        cur = r_end + timedelta(seconds=1)
    return out

def _call(endpoint, params, max_retries, backoff):
    """None = запит не вдався; DataFrame (можливо порожній) = успішна відповідь.
    Мережа/повтори/пауза після 429 - у спільному шарі datum_get_df (клітинка 1)."""
    fn = globals().get("datum_get_df")
    try:
        if fn is None:
            return DatumApi.data_request(endpoint, params)
        return fn(endpoint, params, max_retries=max_retries, backoff=backoff)
    except Exception:
        return None

def _one_call_v3_dt_fast(ticker, start_dt, end_dt, interval, *, chart_type="ohlcv",
                         include_boats=False, max_retries=6, retry_backoff=1.0,
                         normalize_ohlcv=False):
    params = {"ticker": ticker.upper(), "interval": int(interval), "chart_type": chart_type,
              "start_datetime": start_dt, "end_datetime": end_dt,
              "include_boats": bool(include_boats), "format": "json_records"}
    df = _call("/intraday/v3", params, max_retries, retry_backoff)
    if df is not None and not df.empty and "ticker" not in df.columns:
        df.insert(0, "ticker", ticker.upper())
    return df

def _one_call_blue_ocean_v3_dt_fast(ticker, start_dt, end_dt, interval, *, chart_type="ohlcv",
                                    max_retries=6, retry_backoff=1.0, normalize_ohlcv=False):
    params = {"ticker": ticker.upper(), "interval": int(interval), "chart_type": chart_type,
              "start_datetime": start_dt, "end_datetime": end_dt, "format": "json_records"}
    df = _call("/intraday/v3/blue_ocean", params, max_retries, retry_backoff)
    if df is not None and not df.empty and "ticker" not in df.columns:
        df.insert(0, "ticker", ticker.upper())
    return df

def fetch_intraday_v3_datetime_fast(
    ticker: str,
    start: str,
    end: str,
    interval: int = 1,
    *,
    chunk_days: int = 14,
    parallel: bool = True,
    max_workers: int = 12,
    chart_type: str = "ohlcv",
    include_boats: bool = False,
    sort_by_dt: bool = False,   # дефолт: викл (швидкість)
    dedup_dt: bool = False,     # дефолт: викл (швидкість)
    normalize_ohlcv: bool = False,  # дефолт: викл (швидкість)
    include_blue_ocean: bool = False,   # ⬅️ НОВЕ
) -> pd.DataFrame:
    """
    Тягне інтрадійні дані основного вікна і, опційно, Blue Ocean (20:00–04:00),
    зшиває в один DF. За замовчуванням без зайвих перетворень.
    """
    ranges = _build_ranges(start, end, chunk_days)
    if not ranges:
        return pd.DataFrame()

    worker_count = max(1, min(int(max_workers), len(ranges) * (2 if include_blue_ocean else 1))) if parallel else 1
    parts: List[pd.DataFrame] = []

    if parallel and (len(ranges) > 1 or include_blue_ocean):
        with ThreadPoolExecutor(max_workers=worker_count) as ex:
            futs = []
            for (a, b) in ranges:
                futs.append(ex.submit(
                    _one_call_v3_dt_fast, ticker, a, b, interval,
                    chart_type=chart_type, include_boats=include_boats,
                    normalize_ohlcv=normalize_ohlcv
                ))
                if include_blue_ocean:
                    futs.append(ex.submit(
                        _one_call_blue_ocean_v3_dt_fast, ticker, a, b, interval,
                        chart_type=chart_type, normalize_ohlcv=normalize_ohlcv
                    ))
            for f in as_completed(futs):
                df = f.result()
                if df is not None and not df.empty:
                    parts.append(df)
    else:
        for (a, b) in ranges:
            df_main = _one_call_v3_dt_fast(
                ticker, a, b, interval,
                chart_type=chart_type, include_boats=include_boats,
                normalize_ohlcv=normalize_ohlcv
            )
            if df_main is not None and not df_main.empty:
                parts.append(df_main)
            if include_blue_ocean:
                df_bo = _one_call_blue_ocean_v3_dt_fast(
                    ticker, a, b, interval,
                    chart_type=chart_type, normalize_ohlcv=normalize_ohlcv
                )
                if df_bo is not None and not df_bo.empty:
                    parts.append(df_bo)

    if not parts:
        return pd.DataFrame()

    out = pd.concat(parts, ignore_index=True)

    # Якщо підмішували Blue Ocean — відсортуємо по 'dt'
    if "dt" in out.columns and (include_blue_ocean or sort_by_dt or dedup_dt):
        if dedup_dt:
            out = out.drop_duplicates(subset=["ticker", "dt"], keep="last")
        out = out.sort_values("dt", kind="mergesort").reset_index(drop=True)

    return out

# ================== BIG (all tickers, batch writer) [UPDATED + rolling prune] ==================
def fetch_intraday_v3_for_all_tickers(
    tickers_df: pd.DataFrame,
    start: str,
    end: str,
    *,
    interval: int = 1,
    chunk_days: int = 30,
    parallel_chunks: bool = True,
    max_workers_chunks: int = 12,
    parallel_tickers: bool = True,
    max_workers_tickers: int = 6,
    save_every: int = 50,
    out_dir: str = "intraday_batches",
    out_prefix: str = "batch",
    gzip: bool = True,
    keep_columns: Optional[List[str]] = None,
    verbose: bool = True,
    progress_every: int = 10,
    mem_log_every: int = 50,
    use_manifest: bool = False,            # old ticker-level manifest (off by default)
    use_day_manifest: bool = True,         # NEW: day-level manifest (on by default)
    flush_when_rows: Optional[int] = None,
    flush_every_seconds: Optional[int] = None,
    include_blue_ocean: bool = False,      # pass-through
    max_workers: Optional[int] = None,     # загальна к-сть потоків (за замовч. tickers*chunks)
    retry_failed_passes: int = 1,          # додатковий прохід лише для тікерів із помилками
    retry_cooldown_sec: int = 30,          # пауза перед повторним проходом
    compact: bool = True,                  # злити частини дня в один файл + дедуп + сортування
    compact_workers: int = 4,         # Arrow-злиття одного дня ~0.5 ГБ -> 4 потоки в межах бюджету RAM
    empty_day_grace_days: int = 3,         # порожні дні новіші за N днів не вважаємо «готовими» (дані можуть ще не з'явитись)
) -> pd.DataFrame:
    assert "ticker" in tickers_df.columns, "tickers_df має містити колонку 'ticker'"

    # ⬆️ НОРМАЛІЗАЦІЯ: тікери у верхній регістр одразу на вході
    tickers: List[str] = [str(t).strip().upper() for t in tickers_df["ticker"].dropna().unique()]
    if not tickers:
        return pd.DataFrame(columns=["ticker","error"])

    _ensure_dir(out_dir)

    # --- rolling window bounds as YYYY-MM-DD for manifest prune ---
    start_date_str = str(start)[:10]
    end_date_str   = str(end)[:10]

    # --- ковзне вікно: старі дні геть, перед тим як читати маніфест ---
    prune_batches(out_dir, out_prefix, start_date_str, verbose=verbose)

    has_batches = bool(glob.glob(os.path.join(out_dir, f"{out_prefix}_*.csv*")))
    done_tickers = set()
    if use_manifest:
        m = _load_done_tickers_manifest(out_dir)
        if m and has_batches:
            done_tickers = m
        elif m and not has_batches and verbose:
            _p("⚠️ Знайшов done_tickers.txt, але немає batch-файлів → ігнорую маніфест.")
        # apply old manifest only if explicitly enabled
        tickers = [t for t in tickers if t not in done_tickers]

    # NEW: day manifest load (+ rolling prune)
    if use_day_manifest:
        # prune done_days.csv to the current rolling window and get fresh dict
        done_days_by_ticker = prune_done_days_manifest(out_dir, start_date_str, end_date_str)
    else:
        done_days_by_ticker = {}

    # ⬆️ НОРМАЛІЗАЦІЯ: ключі маніфесту у верхній регістр
    if use_day_manifest and done_days_by_ticker:
        done_days_by_ticker = {str(k).upper(): set(v) for k, v in done_days_by_ticker.items()}

    # --- попереднє відсікання повністю готових тікерів ---
    all_days_list = _dates_between(start, end)
    all_days_str = [d.strftime("%Y-%m-%d") for d in all_days_list]

    def _ticker_fully_done(tk: str) -> bool:
        if not use_day_manifest:
            return False
        s = done_days_by_ticker.get(str(tk).upper(), set())
        return bool(s) and all(dd in s for dd in all_days_str)

    fully_done = [t for t in tickers if _ticker_fully_done(t)]
    tickers = [t for t in tickers if t not in fully_done]

    total = len(tickers)
    batch_idx = _next_batch_index(out_dir, out_prefix)
    run_id = str(uuid.uuid4())[:8]

    errors, buffer = [], []
    written_total = 0
    start_time = time.time()
    last_flush_at = start_time
    rows_in_buffer = 0

    if verbose:
        psutil_note = "psutil" if _PSUTIL else "fallback"

        # логіка визначення діапазонів дат для логів
        if use_day_manifest and done_days_by_ticker:
            # усі дні в маніфесті
            all_done_dates = sorted({d for s in done_days_by_ticker.values() for d in s})
            skipped_days_str = (
                f"{all_done_dates[0]} → {all_done_dates[-1]} ({len(all_done_dates)} днів)"
                if all_done_dates else "–"
            )
            # усі дні, які ще не оброблені (порівняно з повним діапазоном)
            all_days_full = [d.strftime("%Y-%m-%d") for d in _dates_between(start, end)]
            remaining_days = sorted(set(all_days_full) - set(all_done_dates))
            remaining_days_str = (
                f"{remaining_days[0]} → {remaining_days[-1]} ({len(remaining_days)} днів)"
                if remaining_days else "–"
            )
        else:
            skipped_days_str = "–"
            remaining_days_str = f"{start[:10]} → {end[:10]}"

        _p(f"▶ Починаємо: {total} тікерів (пропущено {len(done_tickers)} по старому маніфесту) | {start} → {end} | interval={interval} | next batch #{batch_idx:04d} | run={run_id}")
        _p(f"   workers: tickers={max_workers_tickers if parallel_tickers else 1}, chunks={max_workers_chunks if parallel_chunks else 0} | blue_ocean={include_blue_ocean} | day-manifest={use_day_manifest}")
        _p(f"   day-manifest: повністю готових тікерів пропущено = {len(fully_done)}")
        _p(f"   📉 пропущені дні (в межах window): {skipped_days_str}")
        _p(f"   📈 обробляються дні: {remaining_days_str}")
        _p(_mem_str(f"   {psutil_note} · старт · "))

    def _make_batch_path(idx: int, day: Optional[str] = None) -> str:
        mid = f"{day}_" if day else ""
        fname = f"{out_prefix}_{mid}{idx:04d}.csv" + (".gz" if gzip else "")
        return os.path.join(out_dir, fname)

    def _append_df_for_buffer(dfi: pd.DataFrame):
        nonlocal rows_in_buffer
        if dfi is None or dfi.empty:
            return
        buffer.append(dfi)
        rows_in_buffer += len(dfi)

    def _flush():
        nonlocal buffer, batch_idx, written_total, rows_in_buffer, last_flush_at
        if not buffer:
            return None
        if verbose:
            _p(_mem_str("   ⏳ перед flush · "))

        out = pd.concat(buffer, ignore_index=True)
        buffer.clear()                 # частини вже в `out` - не тримаємо другу копію в RAM

        def _write(df_part, path):
            tmp_path = path + ".tmp"
            if gzip:
                df_part.to_csv(tmp_path, index=False, compression={"method": "gzip", "compresslevel": 1})
            else:
                df_part.to_csv(tmp_path, index=False, compression=None)
            os.replace(tmp_path, path)

        # партиції за днем: кожен день у своєму файлі -> видалення старого дня = видалення файлів
        n_files = 0
        if "dt" in out.columns:
            day_s = out["dt"].astype(str).str.slice(0, 10)
            in_window = (day_s >= start_date_str) & (day_s <= end_date_str)   # рядки поза вікном не пишемо
            out, day_s = out[in_window], day_s[in_window]
            for day, g in out.groupby(day_s, sort=True):
                _write(g.sort_values(["ticker", "dt"], kind="mergesort"), _make_batch_path(batch_idx, day))
                n_files += 1
        else:
            _write(out, _make_batch_path(batch_idx))
            n_files = 1

        written_total += len(out)
        if verbose:
            _p(f"💾 batch {batch_idx} → {len(out):,} рядків у {n_files} файлах по днях (всього {written_total:,} у цьому запуску)")

        # NEW: update day-manifest (та одразу оновити кеш у пам'яті)
        if use_day_manifest:
            if "dt" not in out.columns:
                for alt in ("datetime", "time", "t"):
                    if alt in out.columns:
                        out = out.rename(columns={alt: "dt"})
                        break
            _append_done_days_manifest(out_dir, out)
        else:
            if "ticker" in out.columns:
                _append_done_tickers_manifest(out_dir, out["ticker"].astype(str).unique().tolist())

        # (опційно) відмічати повністю закриті тікери у старому маніфесті, якщо він увімкнений
        if use_manifest and use_day_manifest:
            def _ticker_fully_done_local(tk: str) -> bool:
                s = done_days_by_ticker.get(str(tk).upper(), set())
                return bool(s) and all(dd in s for dd in all_days_str)
            fully_done_now = []
            for tk in out.get("ticker", pd.Series(dtype=str)).astype(str).str.upper().unique():
                if _ticker_fully_done_local(tk):
                    fully_done_now.append(tk)
            _append_done_tickers_manifest(out_dir, fully_done_now)

        batch_idx += 1
        buffer.clear()
        rows_in_buffer = 0
        del out
        gc.collect()
        last_flush_at = time.time()

        if verbose:
            _p(_mem_str("   ✅ після flush · "))
        return n_files

    def _progress_line(done: int) -> str:
        pct = (done / total * 100) if total else 100.0
        elapsed = time.time() - start_time
        per_item = elapsed / done if done > 0 else 0.0
        remaining = (total - done) * per_item if done > 0 else 0.0
        return f"{pct:6.2f}% | ETA {_fmt_eta(remaining)}"

    def _should_print(idx: int, is_error: bool, is_last: bool) -> bool:
        if is_error or is_last:
            return True
        if progress_every and progress_every > 0:
            return (idx % progress_every == 0)
        return True

    def _maybe_log_mem(idx: int):
        if not verbose:
            return
        if mem_log_every and mem_log_every > 0 and idx % mem_log_every == 0:
            _p(_mem_str("   • RAM · "))

    def _maybe_flush_by_thresholds(processed: int):
        if not buffer:
            return
        need_rows = ((flush_when_rows is not None and rows_in_buffer >= flush_when_rows)
                     or globals().get("_ram_over_soft", lambda: False)())   # RAM-бюджет: скидаємо на диск раніше
        need_time = (flush_every_seconds is not None and (time.time() - last_flush_at) >= flush_every_seconds)
        need_periodic = (save_every and processed % save_every == 0)
        if need_rows or need_time or need_periodic:
            _flush()

    # NEW: якщо за діапазон взагалі не прийшли дані — позначаємо дні як виконані
    def _mark_days_done(tk: str, days_list: List[pd.Timestamp]):
        if not use_day_manifest or not days_list:
            return
        df_mark = pd.DataFrame({
            "ticker": str(tk).upper(),
            "dt": [pd.Timestamp(d).strftime("%Y-%m-%d 00:00:00") for d in days_list],
        })
        _append_done_days_manifest(out_dir, df_mark)
        done_days_by_ticker.setdefault(str(tk).upper(), set()).update(
            [pd.Timestamp(d).strftime("%Y-%m-%d") for d in days_list]
        )

    workers = max_workers or max(1, (max_workers_tickers if parallel_tickers else 1) *
                                    (max_workers_chunks if parallel_chunks else 1))
    today = pd.Timestamp.today().normalize()

    # --- плануємо задачі: (тікер, діапазон, blue_ocean?) ---
    state, jobs = {}, []
    _ranges_cache = {}      # багато тікерів мають однакову множину відсутніх днів -> діапазони рахуємо раз
    for tk in tickers:
        done_set = done_days_by_ticker.get(tk, set()) if use_day_manifest else set()
        key = tuple(ds for ds in all_days_str if ds not in done_set)
        if not key:
            continue
        rngs = _ranges_cache.get(key)
        if rngs is None:
            rngs = _ranges_cache[key] = _group_consecutive_dates(
                list(pd.to_datetime(list(key))), max(1, int(chunk_days)))
        st = state[tk] = {"pending": 0, "ranges": {}}
        for (a, b) in rngs:
            st["ranges"][(a, b)] = {"ok": True, "parts": []}
            for bo in ((False, True) if include_blue_ocean else (False,)):
                jobs.append((tk, a, b, bo))
                st["pending"] += 1
    total = len(state)

    def _task(tk, a, b, bo):
        if bo:
            return _one_call_blue_ocean_v3_dt_fast(tk, a, b, interval, chart_type="ohlcv")
        return _one_call_v3_dt_fast(tk, a, b, interval, chart_type="ohlcv", include_boats=False)

    def _finish(tk):
        st = state.pop(tk)
        parts, ok_days, failed = [], [], 0
        for (a, b), r in st["ranges"].items():
            if not r["ok"]:                       # діапазон з помилкою відкидаємо повністю -> повтор наступного разу
                failed += 1
                continue
            parts += r["parts"]
            ok_days += [d for d in pd.date_range(a[:10], b[:10])
                        if d <= today - pd.Timedelta(days=int(empty_day_grace_days))]
        dfo = None
        if parts:
            dfo = pd.concat(parts, ignore_index=True)
            if "dt" in dfo.columns:
                dfo = (dfo.drop_duplicates(subset=["ticker", "dt"], keep="last")
                          .sort_values("dt", kind="mergesort").reset_index(drop=True))
            if keep_columns:
                base = ["ticker", "dt"]
                dfo = dfo[[c for c in base + [c for c in keep_columns if c not in base] if c in dfo.columns]]
        have = set(dfo["dt"].astype(str).str.slice(0, 10)) if dfo is not None and "dt" in dfo.columns else set()
        # успішні, але порожні дні (вихідні/свята) - теж «готові»; дні з даними запише flush
        _mark_days_done(tk, [d for d in ok_days if d.strftime("%Y-%m-%d") not in have])
        done_days_by_ticker.setdefault(tk, set()).update(have)
        return dfo, failed

    processed = 0
    # Обмежуємо кількість задач «у польоті»: раніше всі задачі подавались пулу одразу, і поки головний потік
    # скидав буфер на диск, потоки далі качали й копили готові відповіді в RAM (гігабайти).
    max_inflight = max(8, int(workers) * 2)
    job_iter = iter(jobs)
    futs = {}
    with ThreadPoolExecutor(max_workers=workers) as pool:   # не `ex`: нижче є `except ... as ex`
        def _fill():
            while len(futs) < max_inflight:
                j_ = next(job_iter, None)
                if j_ is None:
                    return
                futs[pool.submit(_task, *j_)] = j_
        _fill()
        while futs:
            done_, _ = wait(list(futs), return_when=FIRST_COMPLETED)
            for f in done_:
                tk, a, b, _bo = futs.pop(f)
                df = f.result()
                st = state[tk]
                r = st["ranges"][(a, b)]
                if df is None:
                    r["ok"] = False
                elif not df.empty:
                    r["parts"].append(df)
                st["pending"] -= 1
                del f, df
                if st["pending"]:
                    continue

                processed += 1
                try:
                    dfo, failed = _finish(tk)
                except Exception as ex:
                    dfo, failed = None, 0
                    errors.append({"ticker": tk, "error": f"finish: {ex}"})
                if failed:
                    errors.append({"ticker": tk, "error": f"{failed} діапазон(ів) не завантажено, буде повторено"})
                if dfo is not None and not dfo.empty:
                    _append_df_for_buffer(dfo)
                if verbose and _should_print(processed, bool(failed), processed == total):
                    n = 0 if dfo is None else len(dfo)
                    _p(f"[{processed}/{total}] {'✗' if failed else ('✓' if n else '–')} {tk}: {n:,} рядків | {_progress_line(processed)}")
                _maybe_log_mem(processed)
                _maybe_flush_by_thresholds(processed)
            _fill()

    _flush()

    # --- другий прохід лише для тікерів із помилками: після паузи і з меншою паралельністю ---
    if errors and retry_failed_passes > 0:
        bad = sorted({e["ticker"] for e in errors})
        if verbose:
            _p(f"🔁 повтор для {len(bad)} тікерів із помилками через {retry_cooldown_sec} с (менша паралельність)")
        time.sleep(retry_cooldown_sec)
        errors = fetch_intraday_v3_for_all_tickers(
            pd.DataFrame({"ticker": bad}), start, end,
            interval=interval, chunk_days=chunk_days,
            parallel_chunks=parallel_chunks, max_workers_chunks=max_workers_chunks,
            parallel_tickers=parallel_tickers, max_workers_tickers=max_workers_tickers,
            save_every=save_every, out_dir=out_dir, out_prefix=out_prefix, gzip=gzip,
            keep_columns=keep_columns, verbose=verbose, progress_every=progress_every,
            mem_log_every=mem_log_every, use_manifest=use_manifest, use_day_manifest=use_day_manifest,
            flush_when_rows=flush_when_rows, flush_every_seconds=flush_every_seconds,
            include_blue_ocean=include_blue_ocean, max_workers=max(4, workers // 4),
            retry_failed_passes=retry_failed_passes - 1, retry_cooldown_sec=retry_cooldown_sec,
            compact=False, empty_day_grace_days=empty_day_grace_days,
        ).to_dict("records")

    if compact:
        compact_day_parts(out_dir, out_prefix, gzip=gzip, workers=compact_workers, verbose=verbose)
        verify_window(out_dir, out_prefix, start_date_str, end_date_str, verbose=verbose)

    total_elapsed = time.time() - start_time
    if verbose:
        _p(_mem_str("   🔚 фінал · "))
        _p(f"🏁 Готово. Нових рядків: {written_total:,} → {out_dir} | Час: {_fmt_eta(total_elapsed)}")
    return pd.DataFrame(errors)


In [28]:
TICKERS

,ticker,value,lvl3,market_cap
0,A,111513.82,Medical Equipment & Devices,48776.0
1,AA,822884.78,Metals & Mining,10677.0
2,AAAU,473398.57,None,NaN
3,AACBR,1734.90,NaN,NaN
4,AACG,32.54,Software,33.0
...,...,...,...,...
7135,ZVRA,4056.64,Biotech & Pharma,664.0
7136,ZWS,848.50,Machinery,7457.0
7137,ZYBT,2269.88,Biotech & Pharma,44.0
7138,ZYME,39186.90,Biotech & Pharma,1841.0


In [29]:
import pandas as pd

# список тикерів
tickers = [
    'ARKK', 'COPX', 'DRAM', 'FCX', 'FXI', 'GDX', 'IBIT', 'IGV', 'ITA', 'IWM',
    'KRE', 'KWEB', 'NASA', 'QQQ', 'SLV', 'SOXX', 'SPY', 'UNG', 'URA', 'XBI',
    'XLB', 'XLE', 'XLF', 'XLP', 'XLU', 'XLV', 'XOP', 'XRT',
]


# створення DataFrame
df = pd.DataFrame(tickers, columns=['ticker'])

print(df)


   ticker
0    ARKK
1    COPX
2    DRAM
3     FCX
4     FXI
5     GDX
6    IBIT
7     IGV
8     ITA
9     IWM
10    KRE
11   KWEB
12   NASA
13    QQQ
14    SLV
15   SOXX
16    SPY
17    UNG
18    URA
19    XBI
20    XLB
21    XLE
22    XLF
23    XLP
24    XLU
25    XLV
26    XOP
27    XRT


In [30]:
keep = ["ticker","dt","o","h","l","c","v"]

errs = fetch_intraday_v3_for_all_tickers(
    tickers_df=TICKERS,
    start=start,
    end=end,
    interval=1,
    chunk_days=30,
    max_workers=FETCH_WORKERS,                # один плоский пул; якщо будуть 429/таймаути, знизьте до 16
    save_every=0,                  # flush лише за порогом рядків нижче (і в кінці)
    flush_when_rows=FETCH_FLUSH_ROWS,    # обмежує RAM: ~2-3 ГБ у буфері на великому першому прогоні
    out_dir=str(BATCH_DIR),
    out_prefix="intraday",
    gzip=True,                     # клітинка 19 читає лише intraday_*.csv.gz
    keep_columns=keep,
    verbose=True,
    progress_every=50,
    include_blue_ocean=True,
)


   🧹 вікно від 2024-02-18: видалено файлів 0, переписано 0, лишилось 1176
▶ Починаємо: 7140 тікерів (пропущено 0 по старому маніфесту) | 2024-02-18 00:00:00 → 2026-11-12 23:59:59 | interval=1 | next batch #1177 | run=30949c1e
   workers: tickers=6, chunks=12 | blue_ocean=True | day-manifest=True
   day-manifest: повністю готових тікерів пропущено = 0
   📉 пропущені дні (в межах window): 2024-12-13 → 2026-11-12 (700 днів)
   📈 обробляються дні: 2024-02-18 → 2024-12-12 (299 днів)
   psutil · старт · 🧠 RAM: 268.7 MB
[50/7140] ✓ ACAD: 227,810 рядків |   0.70% | ETA 04:53:04
   • RAM · 🧠 RAM: 2,603.2 MB
[100/7140] ✓ ADIL: 90,563 рядків |   1.40% | ETA 05:52:35
   • RAM · 🧠 RAM: 3,518.6 MB
   ⏳ перед flush · 🧠 RAM: 3,518.6 MB
💾 batch 1177 → 15,026,204 рядків у 737 файлах по днях (всього 15,026,204 у цьому запуску)
   ✅ після flush · 🧠 RAM: 6,397.5 MB
[150/7140] ✓ AGGY: 54,927 рядків |   2.10% | ETA 05:02:19
   • RAM · 🧠 RAM: 3,303.9 MB
[200/7140] ✓ AIN: 96,243 рядків |   2.80% | ETA 04:02:43

In [31]:
# ================== FAST LOCAL POST-PROCESSING (no API calls) ==================
# Читання батчів робить pyarrow (нативний багатопотоковий CSV-парсер) замість
# pandas.read_csv, а premarket-фільтр і денні open/close рахуються за ОДИН прохід
# по gz-файлах замість двох.
import os, gc, glob, time, threading
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pacsv
import pyarrow.compute as pc
import pyarrow.dataset as pads
from concurrent.futures import ThreadPoolExecutor

_INTRADAY_COLS = ["ticker", "dt", "o", "h", "l", "c", "v"]

# Парсимо у float64, щоб денні open/close збігалися з попередньою (pandas) реалізацією
# біт у біт; premarket-рядки перед записом кастимо назад у float32 — стара схема на диску.
_INTRADAY_TYPES = {
    "ticker": pa.string(), "dt": pa.string(),
    "o": pa.float64(), "h": pa.float64(), "l": pa.float64(),
    "c": pa.float64(), "v": pa.float64(),
}
_PREMARKET_SCHEMA = pa.schema([
    ("ticker", pa.string()), ("dt", pa.string()),
    ("o", pa.float32()), ("h", pa.float32()), ("l", pa.float32()),
    ("c", pa.float32()), ("v", pa.float32()),
])


# ---------- спільні хелпери для мерджів по тікеру (використовують клітинки 28/40/44) ----------
def _ticker_codes(s: pd.Series):
    """Факторизує колонку тікерів і нормалізує (strip+upper) ЛИШЕ унікальні значення.

    Повертає (codes, categories); codes[i] == -1 означає null. Сенс у тому, щоб
    strip/upper і словниковий пошук виконувались над ~10^3 унікальними тікерами,
    а не над ~10^6 рядками батча.
    """
    cat = s.astype("category")
    cats = pd.Index(cat.cat.categories.astype(str)).str.strip().str.upper()
    return cat.cat.codes.to_numpy(), cats


def _expand(values: np.ndarray, codes: np.ndarray, fill):
    """Розгортає масив «на категорію» назад «на рядок»; де codes == -1 ставить fill."""
    out = np.empty(codes.shape, dtype=values.dtype if values.dtype != object else object)
    valid = codes >= 0
    out[valid] = values[codes[valid]]
    if (~valid).any():
        out[~valid] = fill
    return out


def _map_by_ticker(codes: np.ndarray, cats: pd.Index, mapping: dict, numeric: bool):
    vals = cats.map(mapping)
    if numeric:
        vals = pd.to_numeric(pd.Series(vals), errors="coerce").to_numpy(dtype="float64")
        return _expand(vals, codes, np.nan)
    vals = np.asarray(pd.Series(vals).astype(object).to_numpy(), dtype=object)
    return _expand(vals, codes, None)


# ---------- читання intraday-батчів ----------
def _open_intraday_csv(path: str, block_size: int):
    return pacsv.open_csv(
        path,
        read_options=pacsv.ReadOptions(use_threads=True, block_size=block_size),
        convert_options=pacsv.ConvertOptions(
            include_columns=_INTRADAY_COLS, column_types=_INTRADAY_TYPES
        ),
    )


def _time_date_mask(dt_arr, start_time, end_time, date_min, date_max):
    t = pc.utf8_slice_codeunits(dt_arr, 11, 19)
    mask = pc.and_(pc.greater_equal(t, start_time), pc.less_equal(t, end_time))
    if date_min is not None or date_max is not None:
        d = pc.utf8_slice_codeunits(dt_arr, 0, 10)
        if date_min is not None:
            mask = pc.and_(mask, pc.greater_equal(d, date_min))
        if date_max is not None:
            mask = pc.and_(mask, pc.less_equal(d, date_max))
    return mask


def _oc_from_table(tbl: pa.Table) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Перший open у вікні 09:30–09:35 і останній close у 15:55–15:59 на (ticker, date)."""
    OPEN_FROM, OPEN_TO = "09:30", "09:35"
    CLOSE_FROM, CLOSE_TO = "15:55", "15:59"

    hhmm = pc.utf8_slice_codeunits(tbl.column("dt"), 11, 16)
    m_o = pc.and_(pc.and_(pc.greater_equal(hhmm, OPEN_FROM), pc.less_equal(hhmm, OPEN_TO)),
                  pc.is_valid(tbl.column("o")))
    m_c = pc.and_(pc.and_(pc.greater_equal(hhmm, CLOSE_FROM), pc.less_equal(hhmm, CLOSE_TO)),
                  pc.is_valid(tbl.column("c")))

    def _reduce(mask, val_col, keep):
        sub = tbl.select(["ticker", "dt", val_col]).filter(mask)
        if sub.num_rows == 0:
            return pd.DataFrame(columns=["ticker", "date", "dt", val_col])
        d = sub.to_pandas()
        d["date"] = d["dt"].str.slice(0, 10)
        d = d.sort_values("dt", kind="stable").drop_duplicates(["ticker", "date"], keep=keep)
        return d[["ticker", "date", "dt", val_col]]

    return _reduce(m_o, "o", "first"), _reduce(m_c, "c", "last")


def _finalize_oc(open_parts: List[pd.DataFrame], close_parts: List[pd.DataFrame]) -> pd.DataFrame:
    opens = (pd.concat(open_parts, ignore_index=True) if open_parts
             else pd.DataFrame(columns=["ticker", "date", "dt", "o"]))
    closes = (pd.concat(close_parts, ignore_index=True) if close_parts
              else pd.DataFrame(columns=["ticker", "date", "dt", "c"]))
    if len(opens):
        opens = opens.sort_values("dt", kind="stable").drop_duplicates(["ticker", "date"], keep="first")
    if len(closes):
        closes = closes.sort_values("dt", kind="stable").drop_duplicates(["ticker", "date"], keep="last")
    opens = opens[["ticker", "date", "o"]].rename(columns={"o": "open"})
    closes = closes[["ticker", "date", "c"]].rename(columns={"c": "close"})

    out = (opens.merge(closes, on=["ticker", "date"], how="outer")
                .sort_values(["ticker", "date"], kind="stable").reset_index(drop=True))
    out["open"] = pd.to_numeric(out["open"], errors="coerce").round(2)
    out["close"] = pd.to_numeric(out["close"], errors="coerce").round(2)
    return out


def collect_premarket_and_oc(
    folder: str,
    out_path: str,
    oc_out_path: Optional[str] = None,
    pattern: str = "intraday_*.csv.gz",
    start_time: str = "00:00:00",
    end_time: str = "23:59:00",
    *,
    date_min: Optional[str] = None,     # 'YYYY-MM-DD' inclusive
    date_max: Optional[str] = None,     # 'YYYY-MM-DD' inclusive
    build_oc: bool = True,
    max_workers: int = 4,
    block_size: int = 64 << 20,
    compression: str = "snappy",
    sort_chunk: bool = True,
    verbose: bool = True,
):
    """ОДИН прохід по {folder}/{pattern}: пише відфільтровані рядки у out_path і
    (опційно) денні open/close у oc_out_path.

    Замінює пару collect_premarket_by_day_stream + build_oc_from_intraday_folder,
    які раніше читали й розтискали ті самі gz-файли двічі.

    Порядок рядків МІЖ файлами недетермінований (файли читаються паралельно).
    Це безпечно: усі наступні кроки — мерджі по ключах, а фінальний файл
    фізично сортується у клітинці 47. Для точного старого порядку — max_workers=1.
    """
    files = sorted(glob.glob(os.path.join(folder, pattern)))
    if not files:
        raise FileNotFoundError(f"Не знайдено файлів {pattern} у {folder}")

    if date_min is not None:
        date_min = str(date_min)[:10]
    if date_max is not None:
        date_max = str(date_max)[:10]
    if date_min is not None and date_max is not None and date_min > date_max:
        date_min, date_max = date_max, date_min

    if os.path.exists(out_path):
        os.remove(out_path)

    t0 = time.time()
    lock = threading.Lock()
    state = {"writer": None, "in": 0, "out": 0, "files": 0}
    open_parts: List[pd.DataFrame] = []
    close_parts: List[pd.DataFrame] = []

    def _handle(path: str):
        reader = _open_intraday_csv(path, block_size)
        loc_open, loc_close = [], []
        n_in = n_out = 0
        while True:
            try:
                rb = reader.read_next_batch()
            except StopIteration:
                break
            tbl = pa.Table.from_batches([rb])
            n_in += tbl.num_rows

            if build_oc:
                o_df, c_df = _oc_from_table(tbl)
                if len(o_df):
                    loc_open.append(o_df)
                if len(c_df):
                    loc_close.append(c_df)

            cut = tbl.filter(_time_date_mask(tbl.column("dt"), start_time, end_time,
                                             date_min, date_max))
            if cut.num_rows:
                cut = cut.cast(_PREMARKET_SCHEMA)
                if sort_chunk:
                    cut = cut.sort_by([("ticker", "ascending"), ("dt", "ascending")])
                n_out += cut.num_rows
                with lock:
                    if state["writer"] is None:
                        state["writer"] = pq.ParquetWriter(out_path, cut.schema,
                                                           compression=compression)
                    state["writer"].write_table(cut)

        with lock:
            state["in"] += n_in
            state["out"] += n_out
            state["files"] += 1
            open_parts.extend(loc_open)
            close_parts.extend(loc_close)
            if verbose:
                print(f"  [{state['files']}/{len(files)}] {os.path.basename(path)}  "
                      f"in={state['in']:,} out={state['out']:,} "
                      f"elapsed={time.time()-t0:,.1f}s", flush=True)

    rng_str = f"  date=[{date_min or '…'}..{date_max or '…'}]" if (date_min or date_max) else ""
    if verbose:
        print(f"▶️ START: {folder}/{pattern} → {out_path}  files={len(files)} "
              f"workers={max_workers}{rng_str}", flush=True)
    try:
        if max_workers and max_workers > 1:
            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                list(ex.map(_handle, files))
        else:
            for p in files:
                _handle(p)
    finally:
        if state["writer"] is not None:
            state["writer"].close()

    df_oc = None
    if build_oc:
        df_oc = _finalize_oc(open_parts, close_parts)
        if oc_out_path:
            os.makedirs(os.path.dirname(oc_out_path) or ".", exist_ok=True)
            df_oc.to_parquet(oc_out_path, index=False)

    if verbose:
        print(f"✅ DONE  in={state['in']:,} → out={state['out']:,}  to {out_path}  "
              f"in {time.time()-t0:,.1f}s", flush=True)
    return out_path, df_oc


def collect_premarket_by_day_stream(
    folder: str,
    out_path: str = "CRACEN/work/premarket_rows.parquet",
    pattern: str = "intraday_*.csv.gz",
    start_time: str = "00:00:00",
    end_time: str = "23:59:00",
    *,
    date_min: Optional[str] = None,
    date_max: Optional[str] = None,
    aggregate: bool = False,
    chunksize: int = 200_000,          # лишено для сумісності виклику; pyarrow керує блоками сам
    compression: str = "snappy",
    log_every: int = 5,                # лишено для сумісності виклику
    sort_chunk: bool = True,
    max_workers: int = 4,
    verbose: bool = True,
):
    """Тонка обгортка над collect_premarket_and_oc (build_oc=False) — щоб старі
    виклики працювали без змін."""
    if aggregate:
        raise NotImplementedError("aggregate=True у цьому пайплайні не використовується")
    path, _ = collect_premarket_and_oc(
        folder, out_path, oc_out_path=None, pattern=pattern,
        start_time=start_time, end_time=end_time,
        date_min=date_min, date_max=date_max, build_oc=False,
        max_workers=max_workers, compression=compression,
        sort_chunk=sort_chunk, verbose=verbose,
    )
    return path


In [32]:
# Один прохід по gz-батчах: одразу і premarket-рядки, і денні open/close.
_, df_oc = collect_premarket_and_oc(
    folder=str(BATCH_DIR),
    out_path=str(WORK_DIR / "premarket_rows.parquet"),
    oc_out_path=str(WORK_DIR / "oc.parquet"),
    start_time="00:00:00",
    end_time="23:59:00",
    date_min=start_date_str,
    date_max=end_date_str,
    build_oc=True,
    max_workers=IO_WORKERS,
    block_size=CSV_BLOCK_SIZE,
)
print(df_oc.head())


▶️ START: C:\datum-api-examples-main\OriON\CRACEN\work\CRACEN_batch/intraday_*.csv.gz → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows.parquet  files=1972 workers=4  date=[2024-02-18..2026-11-12]
  [1/1972] intraday_0004.csv.gz  in=256,170 out=256,170 elapsed=0.5s
  [2/1972] intraday_0002.csv.gz  in=560,867 out=560,867 elapsed=0.5s
  [3/1972] intraday_0003.csv.gz  in=877,672 out=877,672 elapsed=0.5s
  [4/1972] intraday_0001.csv.gz  in=1,254,488 out=1,254,488 elapsed=0.5s
  [5/1972] intraday_0008.csv.gz  in=1,565,173 out=1,565,173 elapsed=1.0s
  [6/1972] intraday_0005.csv.gz  in=1,898,213 out=1,898,213 elapsed=1.0s
  [7/1972] intraday_0007.csv.gz  in=2,262,096 out=2,262,096 elapsed=1.0s
  [8/1972] intraday_0006.csv.gz  in=2,616,277 out=2,616,277 elapsed=1.0s
  [9/1972] intraday_0012.csv.gz  in=2,950,085 out=2,950,085 elapsed=1.5s
  [10/1972] intraday_0011.csv.gz  in=3,291,286 out=3,291,286 elapsed=1.5s
  [11/1972] intraday_0010.csv.gz  in=3,633,453 out=3,633,453 elapsed=1.5

In [33]:
def build_oc_from_intraday_folder(
    folder: str,
    out_path: Optional[str] = None,
    chunksize: int = 250_000,      # лишено для сумісності виклику
    *,
    max_workers: int = 4,
) -> pd.DataFrame:
    """Денні open (перший бар 09:30–09:35) і close (останній бар 15:55–15:59).

    Раніше кожен відібраний рядок проходив через itertuples + оновлення python-словника;
    тепер редукція повністю векторна (sort + drop_duplicates у pyarrow/pandas).
    Клітинка 19 уже рахує це за один прохід — ця функція лишається як самостійний шлях.
    """
    files = sorted(
        glob.glob(os.path.join(folder, "intraday_*.csv")) +
        glob.glob(os.path.join(folder, "intraday_*.csv.gz"))
    )
    if not files:
        raise FileNotFoundError(f"Не знайдено intraday_*.csv[.gz] у: {folder}")

    open_parts, close_parts = [], []
    lock = threading.Lock()

    def _handle(path):
        reader = _open_intraday_csv(path, 64 << 20)
        lo, lc = [], []
        while True:
            try:
                rb = reader.read_next_batch()
            except StopIteration:
                break
            o_df, c_df = _oc_from_table(pa.Table.from_batches([rb]))
            if len(o_df):
                lo.append(o_df)
            if len(c_df):
                lc.append(c_df)
        with lock:
            open_parts.extend(lo)
            close_parts.extend(lc)

    if max_workers and max_workers > 1:
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            list(ex.map(_handle, files))
    else:
        for p in files:
            _handle(p)

    out = _finalize_oc(open_parts, close_parts)

    if out_path is None:
        out_path = os.path.join(folder, "oc.parquet")
    if not out_path.endswith(".parquet"):
        out_path += ".parquet"
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    out.to_parquet(out_path, index=False)
    return out


In [34]:
# oc.parquet уже побудовано в клітинці 19 (той самий прохід по файлах).
# Перебудовуємо лише якщо його немає.
_oc_path = str(WORK_DIR / "oc.parquet")
if os.path.exists(_oc_path):
    df_oc = pd.read_parquet(_oc_path)
else:
    df_oc = build_oc_from_intraday_folder(folder=str(BATCH_DIR), out_path=_oc_path)
print(df_oc.head())


  ticker        date    open   close
0      A  2024-02-20  133.67  134.18
1      A  2024-02-21  133.36  135.33
2      A  2024-02-22  132.48  132.02
3      A  2024-02-23  132.80  132.04
4      A  2024-02-26  131.78  131.50


In [35]:
import os
import gc
import time
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pq = None


def _prep_oc_prev_close(df_oc: pd.DataFrame) -> pd.DataFrame:
    """['ticker','date','open','prev_close'], date — звичайний рядок 'YYYY-MM-DD'."""
    oc = df_oc[["ticker", "date", "open", "close"]].copy()
    oc["ticker"] = oc["ticker"].astype(str)
    oc["date"] = oc["date"].astype(str).str.slice(0, 10)
    oc = oc.sort_values(["ticker", "date"], kind="stable")
    oc["prev_close"] = oc.groupby("ticker", sort=False)["close"].shift(1)
    return oc[["ticker", "date", "open", "prev_close"]].reset_index(drop=True)


def _date_from_dt(s: pd.Series) -> pd.Series:
    """'YYYY-MM-DD' з 'dt'. Для рядкового dt це зріз (C-рівень), без парсингу дат."""
    if pd.api.types.is_string_dtype(s) or s.dtype == object:
        return s.astype(str).str.slice(0, 10)
    return s.dt.strftime("%Y-%m-%d")


def enrich_with_day_open_and_prev_close(df_intraday: pd.DataFrame,
                                        df_oc: pd.DataFrame,
                                        _oc_prepared: Optional[pd.DataFrame] = None,
                                        sort_out: bool = False) -> pd.DataFrame:
    """
    Додає до кожного рядка інтрадею 'open' дня та 'prev_close' (учорашній close).
    Очікує в df_intraday: ['ticker','dt',...]
              в df_oc:       ['ticker','date','open','close']

    Прискорення проти попередньої версії:
      - 'dt' лишається рядком: ні pd.to_datetime, ні .dt.date (це були python-об'єкти
        на кожен рядок — найдорожча операція в усьому пайплайні);
      - prev_close рахується ОДИН раз (передається через _oc_prepared), а не на кожен батч;
      - без per-batch sort_values (фінальне сортування робить клітинка 47).
    """
    right = _oc_prepared if _oc_prepared is not None else _prep_oc_prev_close(df_oc)
    out = df_intraday
    out = out[out["dt"].notna()]
    out = out.assign(date=_date_from_dt(out["dt"]))
    out = out.merge(right, on=["ticker", "date"], how="left")
    if sort_out:
        out = out.sort_values(["ticker", "dt"], kind="stable").reset_index(drop=True)
    return out


def merge_intraday_and_oc_to_file(
    intraday_parquet_path: str,
    oc_parquet_path: str,
    out_path: str,
    batch_rows: int = BATCH_ROWS,
    log_every_batches: int = 5,
    sort_out: bool = False,
):
    """
    Читає великий Parquet батчами і мерджить денні open/prev_close з oc.parquet.
    """
    if pq is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow.parquet).")

    t0 = time.time()

    df_oc = pd.read_parquet(oc_parquet_path)
    missing = {"ticker", "date", "open", "close"} - set(df_oc.columns)
    if missing:
        raise KeyError(f"OC-файл не містить колонок: {sorted(missing)}")
    oc_prepared = _prep_oc_prev_close(df_oc)      # рахуємо ОДИН раз, не на кожен батч

    if os.path.exists(out_path):
        os.remove(out_path)
    if not out_path.endswith((".csv", ".gz", ".parquet")):
        raise ValueError("Підтримані формати: .csv, .csv.gz, .parquet")

    parquet_writer = None
    total_rows = 0

    pf = pq.ParquetFile(intraday_parquet_path)
    cols = ["ticker", "dt", "o", "h", "l", "c", "v"]

    print(f"▶️ START: merge → {out_path} | batch_rows≈{batch_rows:,}", flush=True)

    for batch_idx, batch in enumerate(pf.iter_batches(batch_size=batch_rows, columns=cols), 1):
        df_batch = batch.to_pandas()
        df_enriched = enrich_with_day_open_and_prev_close(
            df_batch, None, _oc_prepared=oc_prepared, sort_out=sort_out)

        if out_path.endswith(".parquet"):
            table = pa.Table.from_pandas(df_enriched, preserve_index=False)
            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
            parquet_writer.write_table(table)
        else:
            df_enriched.to_csv(
                out_path, index=False, mode="a", header=(total_rows == 0),
                compression="infer" if out_path.endswith(".gz") else None,
            )

        total_rows += len(df_enriched)
        if batch_idx % log_every_batches == 0:
            print(f"[batch {batch_idx:>4}] rows={total_rows:,}  "
                  f"elapsed={time.time()-t0:,.1f}s", flush=True)

    if parquet_writer is not None:
        parquet_writer.close()

    print(f"✅ DONE  rows={total_rows:,}  -> {out_path}  in {time.time()-t0:,.1f}s", flush=True)


def merge_intraday_and_oc_in_folder(
    folder: str = "CRACEN",
    intraday_filename: str = "premarket_rows.parquet",
    oc_filename: str = "oc.parquet",
    out_filename: str = "premarket_rows_enriched.parquet",
    batch_rows: int = BATCH_ROWS,
    log_every_batches: int = 5,
) -> str:
    intraday_path = os.path.join(folder, intraday_filename)
    oc_path = os.path.join(folder, oc_filename)
    out_path = os.path.join(folder, out_filename)

    if not os.path.exists(intraday_path):
        raise FileNotFoundError(f"Не знайдено інтрадей: {intraday_path}")
    if not os.path.exists(oc_path):
        raise FileNotFoundError(f"Не знайдено OC-файл: {oc_path}")
    os.makedirs(folder, exist_ok=True)

    merge_intraday_and_oc_to_file(
        intraday_parquet_path=intraday_path,
        oc_parquet_path=oc_path,
        out_path=out_path,
        batch_rows=batch_rows,
        log_every_batches=log_every_batches,
    )
    return out_path


In [36]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# out_path = merge_intraday_and_oc_in_folder(folder=str(WORK_DIR))
# print("Результат у файлі:", out_path)


In [37]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pq = None


def add_pct_vs_prev_close_parquet(
    input_path: str,
    output_path: str,
    batch_rows: int = BATCH_ROWS,
    round_to: Optional[int] = None,
    log_every: int = 5,
    value_col: str = "o",        # ← що порівнюємо з prev_close (дефолт: open)
    out_col: str = "Stack%",     # назва колонки з відсотком
) -> str:
    """
    Потоково читає великий Parquet, додає колонку:
        out_col = (value_col / prev_close - 1) * 100
    Обчислення в numpy (без проміжних pandas-Series і .replace), без gc на кожен батч.
    """
    if pq is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow.parquet).")

    t0 = time.time()
    if os.path.exists(output_path):
        os.remove(output_path)

    pf = pq.ParquetFile(input_path)
    parquet_writer = None
    total = 0

    print(f"▶️ START  file={input_path}  →  {output_path}  "
          f"batch_rows≈{batch_rows:,}  using={value_col}", flush=True)

    for batches, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        for c in ("prev_close", value_col):
            if c not in df.columns:
                raise KeyError(f"У вхідному файлі немає колонки '{c}'.")

        num = pd.to_numeric(df[value_col], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        den = pd.to_numeric(df["prev_close"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            pct = np.where(den == 0, np.nan, num / den - 1.0) * 100.0   # 0 → NaN, без ділення на 0
        if round_to is not None:
            pct = np.round(pct, int(round_to))

        df[out_col] = pct

        table = pa.Table.from_pandas(df, preserve_index=False)
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(output_path, table.schema, compression="snappy")
        parquet_writer.write_table(table)

        total += len(df)
        if batches % log_every == 0:
            print(f"[batch {batches:>4}] rows={total:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if parquet_writer is not None:
        parquet_writer.close()

    print(f"✅ DONE  rows={total:,}  →  {output_path}  in {time.time()-t0:,.1f}s", flush=True)
    return output_path


def add_pct_vs_prev_close_in_folder(
    folder: str = "CRACEN",
    input_filename: str = "premarket_rows_enriched.parquet",
    output_filename: str = "premarket_rows_enriched_with_pct.parquet",
    batch_rows: int = BATCH_ROWS,
    round_to: Optional[int] = 2,
    log_every: int = 5,
    value_col: str = "o",          # ← open vs prev_close
    out_col: str = "Stack%",       # назва нової колонки
) -> str:
    """Обгортка під твою папку CRACEN."""
    os.makedirs(folder, exist_ok=True)
    src = os.path.join(folder, input_filename)
    dst = os.path.join(folder, output_filename)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Не знайдено вхідний файл: {src}")
    return add_pct_vs_prev_close_parquet(
        input_path=src,
        output_path=dst,
        batch_rows=batch_rows,
        round_to=round_to,
        log_every=log_every,
        value_col=value_col,
        out_col=out_col,
    )


In [38]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# out_path = add_pct_vs_prev_close_in_folder(folder=str(WORK_DIR))
# print("Готово:", out_path)


In [39]:
import os
import pandas as pd
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def _read_mapping(mapping_path: str) -> pd.DataFrame:
    """
    Read ticker->bench mapping from csv/xlsx and return ['ticker','bench'].
    """
    ext = os.path.splitext(mapping_path)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(mapping_path)
    else:
        try:
            df = pd.read_csv(mapping_path, sep=None, engine="python")
        except Exception:
            df = pd.read_csv(mapping_path)

    cols = {c.lower().strip(): c for c in df.columns}
    tcol = cols.get("ticker") or cols.get("symbol") or list(df.columns)[0]
    bcol = cols.get("bench") or cols.get("benchmark") or cols.get("etf") or list(df.columns)[1]

    out = df[[tcol, bcol]].rename(columns={tcol: "ticker", bcol: "bench"}).dropna()
    out["ticker"] = out["ticker"].astype(str).str.upper().str.strip()
    out["bench"] = out["bench"].astype(str).str.upper().str.strip()
    out = out[(out["ticker"] != "") & (out["bench"] != "")]
    return out.drop_duplicates(ignore_index=True)


def corr_beta_from_mapping_to_parquet(*args, **kwargs):
    """
    Deprecated API path removed.
    corr/beta is now computed locally in cell 12 and written in cell 13/28.
    """
    raise RuntimeError(
        "corr_beta_from_mapping_to_parquet is deprecated. "
        "Use local calculation in cell 12 and parquet output in cell 13/28."
    )



In [40]:
import os
import pandas as pd

out_path = str(WORK_DIR / "corr_beta_pairs.parquet")

# If parquet already exists from cell 13, keep it.
# Otherwise rebuild it from x_to_best_etf_by_lvl3.csv (no API calls).
if not os.path.exists(out_path):
    src_csv = str(WORK_DIR / "x_to_best_etf_by_lvl3.csv")
    if not os.path.exists(src_csv):
        raise FileNotFoundError(f"Neither {out_path} nor {src_csv} exists")

    d = pd.read_csv(src_csv)
    need = ["x_ticker", "best_y_ticker", "best_corr", "beta_with_best"]
    miss = [c for c in need if c not in d.columns]
    if miss:
        raise KeyError(f"Missing columns in {src_csv}: {miss}")

    d = d.rename(columns={
        "best_y_ticker": "y_ticker",
        "best_corr": "corr",
        "beta_with_best": "beta",
    })[["x_ticker", "y_ticker", "corr", "beta"]]
    d.to_parquet(out_path, index=False)

print("Result:", out_path)



Result: C:\datum-api-examples-main\OriON\CRACEN\work\corr_beta_pairs.parquet


In [41]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def _load_corrbeta_mapping_parquet(path: str) -> pd.DataFrame:
    """
    Читає parquet з мапою:
      A) ['x_ticker','y_ticker','corr','beta'] (API-форма), або
      B) ['ticker','bench','corr','beta'] (вже нормалізована).
    Повертає DF: ['ticker','bench','corr','beta'] (UPPER, без дублікатів; при дублях
    лишаємо рядок із найбільшою corr).
    """
    if pq is None:
        raise RuntimeError("Потрібен pyarrow для читання parquet.")

    df = pq.read_table(path).to_pandas()
    inv = {c.strip().lower(): c for c in df.columns}

    has_api = ("x_ticker" in inv) and ("y_ticker" in inv)
    has_direct = ("ticker" in inv) and ("bench" in inv)
    if not (has_api or has_direct):
        available = ", ".join(df.columns.astype(str).tolist())
        raise KeyError(
            "Mapping parquet повинен містити або ['x_ticker','y_ticker','corr','beta'], "
            "або ['ticker','bench','corr','beta'].\n"
            f"У файлі наявні колонки: {available}"
        )

    if has_api:
        df = df.rename(columns={inv["x_ticker"]: "ticker", inv["y_ticker"]: "bench"})
    else:
        df = df.rename(columns={inv["ticker"]: "ticker", inv["bench"]: "bench"})

    for c in ("corr", "beta"):
        if c in inv:
            df = df.rename(columns={inv[c]: c})
        else:
            df[c] = pd.NA

    df = df[["ticker", "bench", "corr", "beta"]].copy()
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["bench"] = df["bench"].astype(str).str.strip().str.upper()
    df["corr"] = pd.to_numeric(df["corr"], errors="coerce")
    df["beta"] = pd.to_numeric(df["beta"], errors="coerce")

    return (df.sort_values(["ticker", "corr"], ascending=[True, False])
              .drop_duplicates(subset=["ticker"], keep="first")
              .reset_index(drop=True))


def attach_bench_corr_beta_parquet(
    input_parquet_path: str,
    corrbeta_parquet_path: str,
    out_parquet_path: str,
    *,
    batch_rows: int = BATCH_ROWS,
    log_every_batches: int = 5,
    round_decimals: Optional[int] = 4,
    insert_before: str = "pct_vs_prev_close",
    compression: str = "snappy",
) -> str:
    """
    Потоково читає великий Parquet (інтрадеї), додає 'bench','corr','beta' по 'ticker'.

    Замість merge на кожен батч (+ strip/upper над мільйонами рядків) нормалізація і
    пошук у мапі йдуть по унікальних тікерах, а результат розгортається за кодами.
    """
    if pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow (pa, pq) для роботи з Parquet.")

    t0 = time.time()
    mapping = _load_corrbeta_mapping_parquet(corrbeta_parquet_path)
    if round_decimals is not None:
        mapping["corr"] = mapping["corr"].round(round_decimals)
        mapping["beta"] = mapping["beta"].round(round_decimals)

    if mapping.empty:
        raise ValueError(
            f"Мапа bench/corr/beta порожня: {corrbeta_parquet_path}. "
            "Крок побудови x_to_best_etf_by_lvl3 не дав жодного рядка — "
            "інакше bench/corr/beta були б null у всіх рядках."
        )

    m_bench = dict(zip(mapping["ticker"], mapping["bench"]))
    m_corr = dict(zip(mapping["ticker"], mapping["corr"]))
    m_beta = dict(zip(mapping["ticker"], mapping["beta"]))

    if os.path.exists(out_parquet_path):
        os.remove(out_parquet_path)

    pf = pq.ParquetFile(input_parquet_path)
    writer = None
    total_rows = 0

    print(f"▶️ START merge → {out_parquet_path}", flush=True)

    for i, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        if "ticker" not in df.columns:
            raise KeyError("У вхідному parquet відсутній стовпець 'ticker'.")

        codes, cats = _ticker_codes(df["ticker"])
        df["ticker"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)
        # dtype="string" тримає bench рядковим і в батчі без жодного збігу: інакше
        # суцільний None дає pyarrow-тип null і схема розійдеться з першим батчем
        df["bench"] = pd.array(_map_by_ticker(codes, cats, m_bench, numeric=False), dtype="string")
        df["corr"] = _map_by_ticker(codes, cats, m_corr, numeric=True)
        df["beta"] = _map_by_ticker(codes, cats, m_beta, numeric=True)

        cols = [c for c in df.columns if c not in ("bench", "corr", "beta")]
        if insert_before in cols:
            idx = cols.index(insert_before)
            df = df[cols[:idx] + ["bench", "corr", "beta"] + cols[idx:]]
        else:
            df = df[cols + ["bench", "corr", "beta"]]

        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet_path, table.schema, compression=compression)
        elif not table.schema.equals(writer.schema, check_metadata=False):
            table = table.cast(writer.schema)
        writer.write_table(table)

        total_rows += len(df)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total_rows:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if writer is not None:
        writer.close()

    print(f"✅ DONE rows={total_rows:,} → {out_parquet_path}  in {time.time()-t0:,.1f}s", flush=True)
    return out_parquet_path


# опціональний аліас під старе ім'я (щоб існуючий код не міняти)
def attach_bench_corr_beta_file(**kwargs) -> str:
    return attach_bench_corr_beta_parquet(**kwargs)


In [42]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# out_path = attach_bench_corr_beta_parquet(
#     input_parquet_path=str(WORK_DIR / "premarket_rows_enriched_with_pct.parquet"),
#     corrbeta_parquet_path=str(WORK_DIR / "corr_beta_pairs.parquet"),
#     out_parquet_path=str(WORK_DIR / "premarket_rows_final.parquet"),
#     batch_rows=2_000_000,
#     round_decimals=4,
#     insert_before="pct_vs_prev_close",
#     compression="snappy",
# )
# print("Готово:", out_path)


In [43]:
import os
import pandas as pd

def peek_parquet(path: str, n: int = 20, columns=None):
    """
    Легкий перегляд великого Parquet:
      - друкує шлях, кількість рядків (із метаданих), кількість колонок і їх назви
      - потоково читає і показує перші n рядків (optionally лише вказані columns)
      - повертає DataFrame із прев’ю (n рядків)
    Вимагає pyarrow.
    """
    try:
        import pyarrow.parquet as pq
    except Exception as e:
        raise RuntimeError(
            "Потрібен пакет 'pyarrow' для читання parquet. Встанови: pip install pyarrow"
        ) from e

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    pf = pq.ParquetFile(path)
    total_rows = pf.metadata.num_rows if pf.metadata is not None else None
    schema_cols = [f.name for f in pf.schema_arrow]
    print(f"Файл: {path}")
    if total_rows is not None:
        print(f"Рядків (meta): {total_rows:,}")
    print(f"Колонок: {len(schema_cols)}")
    print("Схема:", ", ".join(schema_cols))

    # Збираємо перші n рядків потоково
    need = n
    parts = []
    for batch in pf.iter_batches(batch_size=min(100_000, max(1, n)), columns=columns):
        df_part = batch.to_pandas()
        parts.append(df_part)
        need -= len(df_part)
        if need <= 0:
            break

    head_df = pd.concat(parts, ignore_index=True).head(n) if parts else pd.DataFrame()
    # Коротка діагностика по ключових нових стовпцях (якщо є)
    for c in ["bench", "corr", "beta", "pct_vs_prev_close"]:
        if c in head_df.columns:
            na = int(head_df[c].isna().sum())
            print(f"NaN у '{c}' (у прев'ю): {na}/{len(head_df)}")

    print("\nПерші рядки:")
    with pd.option_context("display.max_rows", n, "display.max_columns", 200, "display.width", 200):
        print(head_df)

    return head_df


In [44]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Проміжного premarket_rows_final.parquet більше не існує: bench/corr/beta
# додаються за той самий прохід, що й решта. Готовий файл — у комірці 47.


In [45]:
# список тикерів
tickers = [
    'ARKK', 'COPX', 'DRAM', 'FCX', 'FXI', 'GDX', 'IBIT', 'IGV', 'ITA', 'IWM',
    'KRE', 'KWEB', 'NASA', 'QQQ', 'SLV', 'SOXX', 'SPY', 'UNG', 'URA', 'XBI',
    'XLB', 'XLE', 'XLF', 'XLP', 'XLU', 'XLV', 'XOP', 'XRT',
]

# створення DataFrame
df_bench = pd.DataFrame(tickers, columns=['ticker'])


In [46]:
keep = ["ticker","dt","o","h","l","c","v"]

errs = fetch_intraday_v3_for_all_tickers(
    tickers_df=df_bench,
    start=start,
    end=end,
    interval=1,
    chunk_days=30,
    max_workers=ETF_FETCH_WORKERS,
    save_every=0,
    flush_when_rows=FETCH_FLUSH_ROWS,
    out_dir=str(ETF_DIR),
    out_prefix="intraday",
    gzip=True,
    keep_columns=keep,
    verbose=True,
    include_blue_ocean=True,
)


   🧹 індексую 9 старих батчів (одноразово)...
   🧹 вікно від 2024-02-18: видалено файлів 0, переписано 0, лишилось 9
▶ Починаємо: 28 тікерів (пропущено 0 по старому маніфесту) | 2024-02-18 00:00:00 → 2026-11-12 23:59:59 | interval=1 | next batch #0010 | run=45d47b92
   workers: tickers=6, chunks=12 | blue_ocean=True | day-manifest=True
   day-manifest: повністю готових тікерів пропущено = 0
   📉 пропущені дні (в межах window): 2026-07-10 → 2026-10-09 (86 днів)
   📈 обробляються дні: 2024-02-18 → 2026-11-12 (913 днів)
   psutil · старт · 🧠 RAM: 2,281.7 MB
[10/28] ✓ IWM: 551,816 рядків |  35.71% | ETA 00:35
[20/28] ✓ XBI: 291,459 рядків |  71.43% | ETA 00:14
[28/28] ✓ XRT: 246,528 рядків | 100.00% | ETA 00:00
   ⏳ перед flush · 🧠 RAM: 3,662.2 MB
💾 batch 10 → 10,187,079 рядків у 739 файлах по днях (всього 10,187,079 у цьому запуску)
   ✅ після flush · 🧠 RAM: 4,822.1 MB
   📅 вікно 2024-02-18..2026-11-12: файлів 748, дані 2024-02-18..2026-11-12, будніх днів без даних (свята?) 0
   🔚 фінал ·

In [47]:
# Один прохід по ETF-батчах: premarket-рядки + денні open/close.
_, df_oc = collect_premarket_and_oc(
    folder=str(ETF_DIR),
    out_path=str(WORK_DIR / "premarket_etf_rows.parquet"),
    oc_out_path=str(WORK_DIR / "oc_etf.parquet"),
    start_time="00:00:00",
    end_time="23:59:00",
    build_oc=True,
    max_workers=IO_WORKERS,
    block_size=CSV_BLOCK_SIZE,
)


▶️ START: C:\datum-api-examples-main\OriON\CRACEN\work\ETF/intraday_*.csv.gz → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows.parquet  files=748 workers=4
  [1/748] intraday_0002.csv.gz  in=17,638 out=17,638 elapsed=0.0s
  [2/748] intraday_0005.csv.gz  in=115,374 out=115,374 elapsed=0.1s
  [3/748] intraday_0001.csv.gz  in=546,902 out=546,902 elapsed=0.5s
  [4/748] intraday_0004.csv.gz  in=1,023,448 out=1,023,448 elapsed=0.6s
  [5/748] intraday_0006.csv.gz  in=1,313,298 out=1,313,298 elapsed=0.6s
  [6/748] intraday_0003.csv.gz  in=2,066,416 out=2,066,416 elapsed=0.6s
  [7/748] intraday_2024-02-20_0010.csv.gz  in=2,079,640 out=2,079,640 elapsed=0.6s
  [8/748] intraday_2024-02-21_0010.csv.gz  in=2,092,946 out=2,092,946 elapsed=0.7s
  [9/748] intraday_2024-02-22_0010.csv.gz  in=2,106,356 out=2,106,356 elapsed=0.7s
  [10/748] intraday_2024-02-23_0010.csv.gz  in=2,118,725 out=2,118,725 elapsed=0.7s
  [11/748] intraday_2024-02-26_0010.csv.gz  in=2,131,513 out=2,131,513 elapse

In [48]:
# oc_etf.parquet уже побудовано в клітинці 34 (той самий прохід по файлах).
_oc_etf_path = str(WORK_DIR / "oc_etf.parquet")
if os.path.exists(_oc_etf_path):
    df_oc = pd.read_parquet(_oc_etf_path)
else:
    df_oc = build_oc_from_intraday_folder(folder=str(ETF_DIR), out_path=_oc_etf_path)
print(df_oc.head())


  ticker        date   open  close
0   ARKK  2024-02-20  49.52  48.71
1   ARKK  2024-02-21  47.64  47.37
2   ARKK  2024-02-22  48.12  48.47
3   ARKK  2024-02-23  48.89  48.41
4   ARKK  2024-02-26  48.44  50.07


In [49]:
# Ті самі оптимізовані функції, що й у клітинці 22 — тут лише інші дефолти обгортки
# (ETF-файли замість акційних).
def merge_intraday_and_oc_in_folder(
    folder: str = "CRACEN",
    intraday_filename: str = "premarket_etf_rows.parquet",
    oc_filename: str = "oc_etf.parquet",
    out_filename: str = "premarket_etf_rows_enriched.parquet",
    batch_rows: int = BATCH_ROWS,
    log_every_batches: int = 5,
) -> str:
    intraday_path = os.path.join(folder, intraday_filename)
    oc_path = os.path.join(folder, oc_filename)
    out_path = os.path.join(folder, out_filename)

    if not os.path.exists(intraday_path):
        raise FileNotFoundError(f"Не знайдено інтрадей: {intraday_path}")
    if not os.path.exists(oc_path):
        raise FileNotFoundError(f"Не знайдено OC-файл: {oc_path}")
    os.makedirs(folder, exist_ok=True)

    merge_intraday_and_oc_to_file(
        intraday_parquet_path=intraday_path,
        oc_parquet_path=oc_path,
        out_path=out_path,
        batch_rows=batch_rows,
        log_every_batches=log_every_batches,
    )
    return out_path


In [50]:
out_path = merge_intraday_and_oc_in_folder(folder=str(WORK_DIR))
print("Результат у файлі:", out_path)


▶️ START: merge → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet | batch_rows≈2,000,000
[batch    5] rows=10,000,000  elapsed=7.5s
✅ DONE  rows=13,631,970  -> C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet  in 10.2s
Результат у файлі: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet


In [51]:
# Та сама оптимізована add_pct_vs_prev_close_parquet із клітинки 24 —
# тут лише інші дефолти обгортки (ETF-файли, колонка Bench%).
def add_pct_vs_prev_close_in_folder(
    folder: str = "CRACEN",
    input_filename: str = "premarket_etf_rows_enriched.parquet",
    output_filename: str = "premarket_etf_rows_enriched_with_pct.parquet",
    batch_rows: int = BATCH_ROWS,
    round_to: Optional[int] = 2,
    log_every: int = 5,
    value_col: str = "o",                 # ← open vs prev_close
    out_col: str = "Bench%",              # назва нової колонки
) -> str:
    """Обгортка під твою папку CRACEN."""
    os.makedirs(folder, exist_ok=True)
    src = os.path.join(folder, input_filename)
    dst = os.path.join(folder, output_filename)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Не знайдено вхідний файл: {src}")
    return add_pct_vs_prev_close_parquet(
        input_path=src,
        output_path=dst,
        batch_rows=batch_rows,
        round_to=round_to,
        log_every=log_every,
        value_col=value_col,
        out_col=out_col,
    )


In [52]:
out_path = add_pct_vs_prev_close_in_folder(folder=str(WORK_DIR))
print("Готово:", out_path)


▶️ START  file=C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched.parquet  →  C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched_with_pct.parquet  batch_rows≈2,000,000  using=o
[batch    5] rows=10,000,000  elapsed=5.0s
✅ DONE  rows=13,631,970  →  C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched_with_pct.parquet  in 6.8s
Готово: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_etf_rows_enriched_with_pct.parquet


In [53]:
import os, gc, time
import pandas as pd
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def merge_bench_pct_from_etf(
    base_path: str = "CRACEN/work/premarket_rows_enriched_with_pct.parquet",
    etf_path:  str = "CRACEN/work/premarket_etf_rows_enriched_with_pct.parquet",
    out_path:  str = "CRACEN/work/premarket_rows_with_bench_pct.parquet",
    *,
    batch_rows: int = BATCH_ROWS,
    align_to_minute: bool = False,        # якщо в секундах відрізняється — підрівняти до хвилини
    round_decimals: Optional[int] = 2,    # округлити Bench% до 2 знаків
    insert_before: Optional[str] = None,  # наприклад, "pct_vs_prev_close"; інакше в кінець
    compression: str = "snappy",
    log_every_batches: int = 5,
) -> str:
    """
    Мерджить Bench% із ETF-файла у базовий по ключах ('bench'↔'ticker', 'dt').

    Прискорення: ETF-мапа читається одним pq.read_table (лише 3 потрібні колонки)
    замість iter_batches + concat + gc; ключ 'dt' лишається рядком —
    лексикографічний порядок 'YYYY-MM-DD HH:MM:SS' збігається з хронологічним,
    тож per-row парсинг дат не потрібен (крім align_to_minute=True).
    """
    if pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow: pip install pyarrow")

    if not os.path.exists(base_path):
        raise FileNotFoundError(base_path)
    if not os.path.exists(etf_path):
        raise FileNotFoundError(etf_path)

    t0 = time.time()

    # ---------- 1) Мапа з ETF-файла: (bench, dt) -> Bench% ----------
    etf_names = [f.name for f in pq.ParquetFile(etf_path).schema_arrow]
    cl = {c.lower().strip(): c for c in etf_names}
    for col in ("ticker", "dt"):
        if col not in cl:
            raise KeyError(f"У ETF-файлі відсутня колонка '{col}'")
    val_col = cl.get("bench%") or cl.get("pct_vs_prev_close")
    if val_col is None:
        raise KeyError("У ETF-файлі не знайдено 'bench%' або 'pct_vs_prev_close'.")

    etf_map = pq.read_table(etf_path, columns=[cl["ticker"], cl["dt"], val_col]).to_pandas()
    etf_map.columns = ["bench", "dt", "Bench%"]
    etf_map["bench"] = etf_map["bench"].astype(str).str.strip().str.upper()
    etf_map["Bench%"] = pd.to_numeric(etf_map["Bench%"], errors="coerce")
    if round_decimals is not None:
        etf_map["Bench%"] = etf_map["Bench%"].round(round_decimals)

    _dt_is_str = (etf_map["dt"].dtype == object) or pd.api.types.is_string_dtype(etf_map["dt"])
    if align_to_minute or not _dt_is_str:
        etf_map["dt"] = pd.to_datetime(etf_map["dt"], errors="coerce")
        if align_to_minute:
            etf_map["dt"] = etf_map["dt"].dt.floor("T")
    else:
        etf_map["dt"] = etf_map["dt"].astype(str)

    etf_map = (etf_map.dropna(subset=["dt"])
                      .sort_values(["bench", "dt"], kind="stable")
                      .drop_duplicates(subset=["bench", "dt"], keep="last")
                      .reset_index(drop=True))

    # ---------- 2) Потоково проходимо базовий файл ----------
    if os.path.exists(out_path):
        os.remove(out_path)

    pf_base = pq.ParquetFile(base_path)
    writer = None
    total_rows = 0

    print(f"▶️ START merge → {out_path}", flush=True)
    for i, batch in enumerate(pf_base.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        for col in ("bench", "dt"):
            if col not in df.columns:
                raise KeyError(f"У базовому файлі відсутня колонка '{col}'")

        codes, cats = _ticker_codes(df["bench"])
        df["bench"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)

        _base_dt_is_str = (df["dt"].dtype == object) or pd.api.types.is_string_dtype(df["dt"])
        if align_to_minute or not _base_dt_is_str:
            df["dt"] = pd.to_datetime(df["dt"], errors="coerce")
            if align_to_minute:
                df["dt"] = df["dt"].dt.floor("T")
        else:
            df["dt"] = df["dt"].astype(str)

        merged = df.merge(etf_map, on=["bench", "dt"], how="left")

        if "Bench%" in merged.columns and insert_before and insert_before in merged.columns:
            cols = [c for c in merged.columns if c != "Bench%"]
            pos = cols.index(insert_before)
            merged = merged[cols[:pos] + ["Bench%"] + cols[pos:]]

        table = pa.Table.from_pandas(merged, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema, compression=compression)
        writer.write_table(table)

        total_rows += len(merged)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total_rows:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if writer is not None:
        writer.close()

    print(f"✅ DONE rows={total_rows:,} → {out_path}  in {time.time()-t0:,.1f}s", flush=True)
    return out_path


In [54]:
# ── ЗЛИТО В ОДИН ПРОХІД ─────────────────────────────────────────────
# Цей крок тепер робить build_final_in_one_pass у комірці 46: весь файл
# читається й пишеться ОДИН раз замість п'яти. Старий виклик лишено
# закоментованим — щоб відкотити, розкоментуй його і прибери виклик у 46.
# merge_bench_pct_from_etf(
#     base_path=str(WORK_DIR / "premarket_rows_final.parquet"),
#     etf_path =str(WORK_DIR / "premarket_etf_rows_enriched_with_pct.parquet"),
#     out_path =str(WORK_DIR / "premarket_rows_with_bench_pct.parquet"),
#     batch_rows=2_000_000,
#     align_to_minute=False,       # вмикай, якщо секунди/мілісекунди «пливуть»
#     round_decimals=2,
#     insert_before=None,          # або, наприклад, "pct_vs_prev_close"
#     compression="snappy",
# )


In [55]:
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


def _pick_first_existing(df: pd.DataFrame, candidates):
    cols_l = {c.lower().strip(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols_l:
            return cols_l[c.lower()]
    return None


_REPORT_FAILS = {"n": 0}
_GAP_FAILS = {"n": 0}


def _datum_request_retry(endpoint, params, *, max_retries=6, backoff=1.0):
    """Спільний шар з клітинки 1 (keep-alive, пауза після 429, повтори лише 429/5xx/мережі)."""
    fn = globals().get("datum_get_df")
    if fn is None:                                       # запасний шлях, якщо клітинку 1 не виконано
        return DatumApi.data_request(endpoint, params)
    return fn(endpoint, params, max_retries=max_retries, backoff=backoff)


def get_reports(ticker, last_n_reports, start_date, end_date):
    try:
        params = {
            "ticker": str(ticker),
            "start_move_date": str(start_date)[:10],
            "end_move_date": str(end_date)[:10],
        }
        reports_df = _datum_request_retry("/reports", params)
        if reports_df is None or reports_df.empty:
            return pd.DataFrame(columns=["ticker", "report_date"])

        date_col = _pick_first_existing(
            reports_df,
            ["report_date", "move_date", "date", "dt", "datetime", "published_at", "created_at"],
        )
        if date_col is None:
            return pd.DataFrame(columns=["ticker", "report_date"])

        out = reports_df.copy()
        out["report_date"] = pd.to_datetime(out[date_col], errors="coerce").dt.strftime("%Y-%m-%d")
        out["ticker"] = str(ticker).upper()
        out = out.dropna(subset=["report_date"])
        out = out.sort_values("report_date").tail(int(last_n_reports))
        return out[["ticker", "report_date"]].reset_index(drop=True)
    except Exception as e:
        _REPORT_FAILS["n"] += 1
        print(f"Report fetch failed for {ticker}: {e}")
        return pd.DataFrame(columns=["ticker", "report_date"])


def get_reports_for_tickers(tickers_df, start_date, end_date, last_n_reports=20,
                            max_workers=8, max_fail_ratio=0.15):
    # 16 потоків без пауз давали суцільні 429; з retry вистачає меншого числа
    tickers = tickers_df["ticker"].dropna().astype(str).str.upper().unique().tolist()
    parts = []
    _REPORT_FAILS["n"] = 0

    def fetch(t):
        return get_reports(ticker=t, last_n_reports=last_n_reports, start_date=start_date, end_date=end_date)

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch, t): t for t in tickers}
        for f in tqdm(as_completed(futures), total=len(futures), desc="Reports"):
            rep = f.result()
            if rep is not None and not rep.empty:
                parts.append(rep)

    if tickers and _REPORT_FAILS["n"] / len(tickers) > max_fail_ratio:
        raise RuntimeError(
            f"API відмовив на {_REPORT_FAILS['n']} з {len(tickers)} тикерів "
            f"({_REPORT_FAILS['n'] / len(tickers) * 100:.1f}% > {max_fail_ratio * 100:.0f}%). "
            "Репорти вийшли б неповними."
        )

    if not parts:
        return pd.DataFrame(columns=["ticker", "report_date"]), {}

    reports = pd.concat(parts, ignore_index=True).drop_duplicates(["ticker", "report_date"])
    report_days_map = reports.groupby("ticker")["report_date"].apply(set).to_dict()
    return reports, report_days_map


def get_gaps_for_tickers_excluding_reports(tickers_df, start_date, end_date, report_days_map,
                                           max_workers=8, max_fail_ratio=0.15):
    tickers = tickers_df["ticker"].dropna().astype(str).str.upper().unique().tolist()
    _GAP_FAILS["n"] = 0

    def fetch(t):
        params = {
            "ticker": t,
            "start_date": str(start_date)[:10],
            "end_date": str(end_date)[:10],
            "format": "json_records",
        }
        try:
            df_gap = _datum_request_retry("/daily/gaps", params)
            if df_gap is None or df_gap.empty:
                return None

            date_col = _pick_first_existing(df_gap, ["date", "move_date", "dt", "datetime", "day"])
            if date_col is None:
                return None

            out = df_gap.copy()
            out["ticker"] = out["ticker"].astype(str).str.upper() if "ticker" in out.columns else t
            out["date"] = pd.to_datetime(out[date_col], errors="coerce").dt.strftime("%Y-%m-%d")
            out["gap"] = pd.to_numeric(out.get("gap"), errors="coerce")
            out = out.dropna(subset=["date", "gap"])

            report_days = report_days_map.get(t, set())
            if report_days:
                out = out[~out["date"].isin(report_days)]

            return out if not out.empty else None
        except Exception as e:
            _GAP_FAILS["n"] += 1
            print(f"Gap fetch failed for {t}: {e}")
            return None

    all_results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch, t): t for t in tickers}
        for f in tqdm(as_completed(futures), total=len(futures), desc="Gaps (ex-reports)"):
            result = f.result()
            if result is not None:
                all_results.append(result)

    if tickers and _GAP_FAILS["n"] / len(tickers) > max_fail_ratio:
        raise RuntimeError(
            f"API відмовив на {_GAP_FAILS['n']} з {len(tickers)} тикерів "
            f"({_GAP_FAILS['n'] / len(tickers) * 100:.1f}% > {max_fail_ratio * 100:.0f}%). "
            "Гепи вийшли б неповними."
        )

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame(columns=["ticker", "date", "gap"])


def add_rolling_sigma_cap(
    df,
    value_col,
    window=20,
    group_col="ticker",
    date_col="date",
    cap_q=0.8,
    min_periods=None,
):
    """Rolling std по попередньому вікну з обрізанням хвостів на квантилі cap_q.

    Раніше це був rolling(...).apply(python_func) — окремий виклик Python на кожне
    вікно кожного тікера. Тепер вікна будуються через sliding_window_view і квантиль/
    clip/std рахуються одразу по всій матриці вікон групи (чисто numpy).
    Результат збігається з попередньою реалізацією до 1e-12.
    """
    if min_periods is None:
        min_periods = window

    out = df.sort_values([group_col, date_col], kind="stable").copy()
    col = f"{value_col}_sigma_{window}d_cap"

    # Рідкісний випадок min_periods != window має іншу семантику (часткові вікна) —
    # там лишаємо стару, повільну, але точну реалізацію.
    if min_periods != window:
        def compute_sigma(s):
            x = s.shift(1)

            def f(arr):
                arr = arr[~np.isnan(arr)]
                if len(arr) < min_periods:
                    return np.nan
                cap = np.quantile(np.abs(arr), cap_q)
                arr_capped = np.clip(arr, -cap, cap)
                return np.std(arr_capped, ddof=1)

            return x.rolling(window, min_periods=min_periods).apply(f, raw=True)

        out[col] = out.groupby(group_col)[value_col].transform(compute_sigma)
        return out

    vals = pd.to_numeric(out[value_col], errors="coerce").to_numpy(dtype="float64")
    codes, _ = pd.factorize(out[group_col], sort=False)
    res = np.full(vals.shape, np.nan)

    order = np.argsort(codes, kind="stable")
    bounds = np.flatnonzero(np.diff(codes[order])) + 1
    for sl in np.split(order, bounds):
        n = sl.size
        if n <= window:
            continue
        x = vals[sl]
        # вікно з ПОПЕРЕДНІХ значень (еквівалент shift(1) у старій версії)
        w = np.lib.stride_tricks.sliding_window_view(x[:-1], window)   # (n-window, window)
        cap = np.quantile(np.abs(w), cap_q, axis=1, keepdims=True)
        res[sl[window:]] = np.clip(w, -cap, cap).std(axis=1, ddof=1)

    out[col] = res
    return out


# ===================== ковзні sigma / corr / beta / bench на ROLL_WINDOW попередніх торгових днях =====================
# Формули ті самі, що були для всього періоду, але для КОЖНОГО дня D вікно = останні ROLL_WINDOW рядків ряду гепів
# тікера ПЕРЕД D (торгові дні з даними, без днів звітів). Сам день D у вікно не входить (без look-ahead).
# Найкращий ETF (bench) теж вибирається щодня за найбільшою corr у цьому вікні.

def _row_quantile(a, q):
    """Лінійний квантиль (як np.quantile) по кожному рядку, ігноруючи NaN. a: (n, W). Повертає (n,) [NaN, якщо в рядку немає значень]."""
    cnt = (~np.isnan(a)).sum(axis=1)
    s = np.sort(np.where(np.isnan(a), np.inf, a), axis=1)          # NaN -> +inf і йдуть у кінець
    pos = (cnt - 1) * q
    lo = np.floor(np.clip(pos, 0, None)).astype(np.int64)
    hi = np.minimum(lo + 1, np.maximum(cnt - 1, 0))
    frac = np.clip(pos, 0, None) - lo
    rows = np.arange(a.shape[0])
    lo_v, hi_v = s[rows, lo], s[rows, hi]
    with np.errstate(invalid="ignore"):
        out = lo_v + (hi_v - lo_v) * frac
    out[cnt == 0] = np.nan
    return out


def _stats_for_ticker(x, y_cols, window, cap_q, min_hist_for_cap, min_overlap):
    """x: (n,) гепи тікера (лише торгові дні з даними, без днів звітів), y_cols: (n, E) гепи ETF на ті самі дати (NaN, якщо немає).
    Для k = 0..n повертає статистики по вікну з ОСТАННІХ `window` рядків ПЕРЕД k-м (k = кількість рядків до дня D):
      sigma (повне вікно; std ddof=1 після обрізання |gap| на квантилі cap_q),
      corr/beta/best_etf_index (рядки з |gap| > квантиль вікна відкидаються, якщо у вікні >= min_hist_for_cap значень;
      пари з ETF; потрібно >= min_overlap пар; найкращий ETF = максимум corr)."""
    n = len(x)
    W = int(window)
    pad = np.full(W, np.nan)
    wx = np.lib.stride_tricks.sliding_window_view(np.concatenate([pad, x]), W)          # (n+1, W): вікно k = x[k-W:k]
    cnt = (~np.isnan(wx)).sum(axis=1)

    cap = _row_quantile(np.abs(wx), cap_q)
    with np.errstate(invalid="ignore"):
        sigma = np.clip(wx, -cap[:, None], cap[:, None]).std(axis=1, ddof=1)
    sigma = np.where(cnt == W, sigma, np.nan)                                            # як min_periods=window

    E = y_cols.shape[1]
    corr_best = np.full(n + 1, np.nan)
    beta_best = np.full(n + 1, np.nan)
    arg_best = np.full(n + 1, -1, dtype=np.int64)
    if E:
        elig = cnt >= min_hist_for_cap
        with np.errstate(invalid="ignore"):
            keep = ~np.isnan(wx) & (~elig[:, None] | (np.abs(wx) <= cap[:, None]))
        x2 = np.where(keep, wx, np.nan)
        C = np.full((n + 1, E), np.nan)
        B = np.full((n + 1, E), np.nan)
        for j in range(E):
            wy = np.lib.stride_tricks.sliding_window_view(np.concatenate([pad, y_cols[:, j]]), W)
            m = ~np.isnan(x2) & ~np.isnan(wy)
            nn = m.sum(axis=1)
            with np.errstate(invalid="ignore", divide="ignore"):
                mx = np.where(m, x2, 0.0).sum(axis=1) / nn
                my = np.where(m, wy, 0.0).sum(axis=1) / nn
                xc = np.where(m, x2 - mx[:, None], 0.0)
                yc = np.where(m, wy - my[:, None], 0.0)
                d = nn - 1
                var_y = (yc * yc).sum(axis=1) / d
                cov = (xc * yc).sum(axis=1) / d
                std_x = np.sqrt((xc * xc).sum(axis=1) / d)
                ok = (nn >= min_overlap) & (var_y > 0) & (std_x > 0)
                C[:, j] = np.where(ok, cov / (std_x * np.sqrt(var_y)), np.nan)
                B[:, j] = np.where(ok, cov / var_y, np.nan)
        has = (~np.isnan(C)).any(axis=1)
        arg = np.where(np.isnan(C), -np.inf, C).argmax(axis=1)
        r = np.arange(n + 1)
        corr_best = np.where(has, C[r, arg], np.nan)
        beta_best = np.where(has, B[r, arg], np.nan)
        arg_best = np.where(has, arg, -1)
    return sigma, corr_best, beta_best, arg_best


def compute_rolling_stats(stock_gaps, etf_gaps, tickers_df, sector_to_etf, *, out_start_date,
                          window=60, cap_q=0.8, min_hist_for_cap=20, min_overlap=20,
                          sector_col="lvl3", ticker_col="ticker",
                          fallback_etfs=("SPY", "QQQ", "IWM"), max_workers=4, return_current=False):
    """Для КОЖНОГО (ticker, date) рахує sigma / corr / beta / bench на `window` попередніх торгових днях тікера
    (дні з даними: рядки stock_gaps, де вже немає днів звітів; сам день D у вікно не входить).
    Формули ті самі, що були для всього періоду: corr = Pearson, beta = cov/var(ETF), bench = ETF з найбільшою corr
    серед ETF сектора, відкидання викидів |gap| > квантиль cap_q, sigma = std обрізаних гепів.
    Повертає DataFrame [ticker, date, bench, corr, beta, sigma] для дат >= out_start_date.
    return_current=True: повертає (daily, current), де current - ОДИН рядок на тікер зі значеннями СТАНОМ НА ЗАРАЗ
    (вікно = останні `window` торгових днів тікера до кінця даних): [ticker, bench, corr, beta, sigma, asof_date, n_window]."""
    from concurrent.futures import ThreadPoolExecutor

    sg = stock_gaps[["ticker", "date", "gap"]].copy()
    sg["ticker"] = sg["ticker"].astype(str).str.upper().str.strip()
    sg["date"] = pd.to_datetime(sg["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    sg["gap"] = pd.to_numeric(sg["gap"], errors="coerce")
    sg = (sg.dropna().drop_duplicates(["ticker", "date"], keep="last").sort_values(["ticker", "date"]))

    eg = etf_gaps[["ticker", "date", "gap"]].copy()
    eg["ticker"] = eg["ticker"].astype(str).str.upper().str.strip()
    eg["date"] = pd.to_datetime(eg["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    eg["gap"] = pd.to_numeric(eg["gap"], errors="coerce")
    eg = eg.dropna().drop_duplicates(["ticker", "date"], keep="last")
    piv = eg.pivot(index="date", columns="ticker", values="gap").sort_index()
    etf_dates = np.array(piv.index.astype(str), dtype="U10")
    etf_names = [str(c) for c in piv.columns]
    etf_col = {c: i for i, c in enumerate(etf_names)}
    etf_arr = piv.to_numpy(dtype="float64")

    out_start = str(out_start_date)[:10]
    all_dates = np.array(sorted({d for d in sg["date"].unique() if d >= out_start} |
                                {d for d in etf_dates if d >= out_start}), dtype="U10")
    _cur_cols = ["ticker", "bench", "corr", "beta", "sigma", "asof_date", "n_window"]
    if len(all_dates) == 0:
        empty = pd.DataFrame(columns=["ticker", "date", "bench", "corr", "beta", "sigma"])
        return (empty, pd.DataFrame(columns=_cur_cols)) if return_current else empty

    sec = (tickers_df[[ticker_col, sector_col]].dropna(subset=[ticker_col]).copy())
    sec[ticker_col] = sec[ticker_col].astype(str).str.upper().str.strip()
    sec[sector_col] = sec[sector_col].fillna("UNKNOWN").astype(str)
    sector_of = dict(zip(sec[ticker_col], sec[sector_col]))
    fb = [str(e).upper() for e in fallback_etfs]

    def etfs_for(tk):
        raw = sector_to_etf.get(sector_of.get(tk, "UNKNOWN")) or fb
        return [str(e).upper() for e in raw if str(e).upper() in etf_col]

    groups = {tk: (g["date"].to_numpy(dtype="U10"), g["gap"].to_numpy(dtype="float64"))
              for tk, g in sg.groupby("ticker")}
    bench_names = np.array(etf_names + [None], dtype=object)

    def _one(tk):
        dates, x = groups[tk]
        names = etfs_for(tk)
        cols = [etf_col[e] for e in names]
        if cols:
            pos = np.searchsorted(etf_dates, dates)
            okp = (pos < len(etf_dates))
            okp[okp] = etf_dates[pos[okp]] == dates[okp]
            y = np.full((len(dates), len(cols)), np.nan)
            y[okp] = etf_arr[pos[okp]][:, cols]
        else:
            y = np.empty((len(dates), 0))
        sigma, corr, beta, arg = _stats_for_ticker(x, y, window, cap_q, min_hist_for_cap, min_overlap)
        k = np.searchsorted(dates, all_dates, side="left")                    # к-сть рядків тікера ДО дня D
        if cols:
            a_k = arg[k]
            bench_idx = np.array(cols + [-1])[np.where(a_k >= 0, a_k, len(cols))]     # локальний індекс -> глобальний індекс ETF
        else:
            bench_idx = np.full(len(k), -1)
        s_, c_, b_ = sigma[k], corr[k], beta[k]
        keep = ~(np.isnan(s_) & np.isnan(c_))
        n = len(x)                                                            # СТАНОМ НА ЗАРАЗ: вікно, що закінчується останнім рядком
        loc = int(arg[n])
        cur = (tk, (cols[loc] if (cols and loc >= 0) else -1), float(corr[n]), float(beta[n]),
               float(sigma[n]), str(dates[-1]), int(min(n, window)))
        return tk, np.flatnonzero(keep), bench_idx[keep], c_[keep], b_[keep], s_[keep], cur

    tickers = sorted(groups)
    parts = []
    with ThreadPoolExecutor(max_workers=max(1, int(max_workers))) as ex:
        for res in ex.map(_one, tickers):
            parts.append(res)

    n_rows = sum(len(p[1]) for p in parts)
    tk_col = np.empty(n_rows, dtype=object); di = np.empty(n_rows, dtype=np.int64)
    bi = np.empty(n_rows, dtype=np.int64)
    corr_a = np.empty(n_rows); beta_a = np.empty(n_rows); sig_a = np.empty(n_rows)
    o = 0
    for tk, d_idx, b_idx, c_, b_, s_, _cur in parts:
        m = len(d_idx)
        tk_col[o:o + m] = tk; di[o:o + m] = d_idx; bi[o:o + m] = b_idx
        corr_a[o:o + m] = c_; beta_a[o:o + m] = b_; sig_a[o:o + m] = s_
        o += m
    out = pd.DataFrame({
        "ticker": tk_col,
        "date": all_dates[di].astype(object),
        "bench": bench_names[np.where(bi >= 0, bi, len(bench_names) - 1)],
        "corr": corr_a, "beta": beta_a, "sigma": sig_a,
    })
    if not return_current:
        return out
    current = pd.DataFrame([p[6] for p in parts], columns=["ticker", "_b", "corr", "beta", "sigma", "asof_date", "n_window"])
    current["bench"] = bench_names[np.where(current["_b"].to_numpy() >= 0, current["_b"].to_numpy(), len(bench_names) - 1)]
    current = current.drop(columns="_b")[_cur_cols]
    missing = sorted(set(sector_of) - set(current["ticker"]))                  # тікери без жодного гепа теж мають рядок (значення NaN)
    if missing:
        current = pd.concat([current, pd.DataFrame({"ticker": missing})], ignore_index=True)
    return out, current.sort_values("ticker").reset_index(drop=True)


def get_etf_gaps(etfs, start_date, end_date, max_workers=8, max_fail_ratio=0.15):
    """Гепи ETF-кандидатів (/daily/gaps) для вирівнювання зі гепами акцій."""
    etfs = sorted({str(e).upper().strip() for e in etfs if e})
    fails = {"n": 0}

    def fetch(t):
        try:
            df = _datum_request_retry("/daily/gaps", {
                "ticker": t, "start_date": str(start_date)[:10], "end_date": str(end_date)[:10],
                "format": "json_records"})
            if df is None or df.empty:
                return None
            dcol = _pick_first_existing(df, ["date", "move_date", "dt", "datetime", "day"])
            if dcol is None or "gap" not in df.columns:
                return None
            out = pd.DataFrame({
                "ticker": t,
                "date": pd.to_datetime(df[dcol], errors="coerce").dt.strftime("%Y-%m-%d"),
                "gap": pd.to_numeric(df["gap"], errors="coerce"),
            }).dropna()
            return out if not out.empty else None
        except Exception as e:
            fails["n"] += 1
            print(f"ETF gap fetch failed for {t}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        parts = [r for r in ex.map(fetch, etfs) if r is not None]
    if etfs and fails["n"] / len(etfs) > max_fail_ratio:
        raise RuntimeError(f"API відмовив на {fails['n']} з {len(etfs)} ETF - bench/corr/beta вийшли б неповними.")
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["ticker", "date", "gap"])


# 1) Report days: вікно + запас історії для перших днів вікна
reports_df, report_days_map = get_reports_for_tickers(
    TICKERS,
    start_date=gaps_start_date_str,
    end_date=end_date_str,
    last_n_reports=20,
)
print("Report rows:", len(reports_df))

# 2) Gaps акцій без днів звітів (з запасом історії)
df_gaps_all = get_gaps_for_tickers_excluding_reports(
    TICKERS,
    start_date=gaps_start_date_str,
    end_date=end_date_str,
    report_days_map=report_days_map,
)
print("Rows in gaps (without report days):", len(df_gaps_all))

# 3) Gaps ETF-кандидатів
_etf_candidates = ({str(e).upper() for v in sector_to_etf.values() for e in (v or [])} | {"SPY", "QQQ", "IWM"})
etf_gaps_all = get_etf_gaps(_etf_candidates, gaps_start_date_str, end_date_str)
print("Rows in ETF gaps:", len(etf_gaps_all), "| ETF:", etf_gaps_all["ticker"].nunique())

# 4) Ковзна статистика на кожен (ticker, date) вікна
daily_stats, current_stats = compute_rolling_stats(
    df_gaps_all, etf_gaps_all, TICKERS, sector_to_etf,
    out_start_date=start_date_str,
    window=ROLL_WINDOW,
    cap_q=0.8,
    min_hist_for_cap=20,
    min_overlap=20,
    return_current=True,
)
daily_stats.to_parquet(str(WORK_DIR / "rolling_stats.parquet"), index=False)
print("rolling_stats:", len(daily_stats), "рядків,", daily_stats["ticker"].nunique(), "тікерів,",
      daily_stats["date"].min(), "..", daily_stats["date"].max())

# СТАНОМ НА ЗАРАЗ: один рядок на тікер - вікно = останні ROLL_WINDOW торгових днів до кінця даних.
# Ці значення йдуть окремим файлом і окремими колонками *_now у final.parquet (історичні corr/beta/sigma лишаються по днях).
current_stats.to_parquet(str(WORK_DIR / "current_stats.parquet"), index=False)
current_stats.round({"corr": 4, "beta": 4, "sigma": 6}).to_csv(str(WORK_DIR / "current_stats.csv"), index=False)
_cov = current_stats["sigma"].notna().mean() if len(current_stats) else 0.0
print(f"current_stats: {len(current_stats):,} тікерів | зі sigma/corr/beta: {_cov:.1%} | "
      f"найновіший день у вікні: {current_stats['asof_date'].max()}")

# сумісність: sigma станом на зараз (для старого шляху / швидкого огляду)
sigma_col = "sigma"
std_by_ticker = current_stats.dropna(subset=[sigma_col])[["ticker", sigma_col]].reset_index(drop=True)
print("Tickers with sigma:", len(std_by_ticker))
std_by_ticker.head()


Reports: 100%|██████████| 7140/7140 [01:15<00:00, 94.37it/s] 


Report rows: 49449


Gaps (ex-reports): 100%|██████████| 7140/7140 [01:24<00:00, 84.63it/s] 


Rows in gaps (without report days): 4773560
Rows in ETF gaps: 20425 | ETF: 28
rolling_stats: 4321768 рядків, 7108 тікерів, 2024-02-20 .. 2026-11-12
current_stats: 7,140 тікерів | зі sigma/corr/beta: 98.9% | найновіший день у вікні: 2026-11-12
Tickers with sigma: 7062


,ticker,sigma
0,A,0.710556
1,AA,1.449089
2,AAAU,1.053692
3,AACBR,7.796230
4,AACG,3.382086


In [56]:
std_by_ticker



,ticker,sigma
0,A,0.710556
1,AA,1.449089
2,AAAU,1.053692
3,AACBR,7.796230
4,AACG,3.382086
...,...,...
7057,ZVRA,0.628579
7058,ZWS,0.723994
7059,ZYBT,1.607616
7060,ZYME,0.757129


In [57]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception:
    pa = pq = None


def _prepare_sigma_df(
    sigma_df: Optional[pd.DataFrame] = None,
    sigma_path: Optional[str] = None
) -> pd.DataFrame:
    """
    Готує мапу ['ticker','sigma'] з DataFrame або з файлу (.parquet/.csv).
    Нормалізує регістр, типи і прибирає дублікати по 'ticker'.
    """
    if sigma_df is None and sigma_path is None:
        raise ValueError("Передай або sigma_df, або sigma_path.")

    if sigma_df is None:
        ext = os.path.splitext(sigma_path)[1].lower()
        if ext == ".parquet":
            sigma_df = pd.read_parquet(sigma_path)
        else:
            sigma_df = pd.read_csv(sigma_path, low_memory=False)

    df = sigma_df.copy()
    cols_lower = {c.lower().strip(): c for c in df.columns}
    tcol = cols_lower.get("ticker")
    scol = cols_lower.get("sigma")
    if tcol is None or scol is None:
        raise KeyError("Очікую колонки 'ticker' і 'sigma' у даних зі скріна.")

    df = df[[tcol, scol]].rename(columns={tcol: "ticker", scol: "sigma"})
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["sigma"] = pd.to_numeric(df["sigma"], errors="coerce")
    df = (df.sort_values(["ticker"])
            .drop_duplicates(subset=["ticker"], keep="last")
            .reset_index(drop=True))
    return df[["ticker", "sigma"]]


def add_sigma_by_ticker_to_parquet(
    base_path: str = "CRACEN/work/premarket_rows_with_bench_pct.parquet",
    out_path:  Optional[str] = None,
    *,
    sigma_df: Optional[pd.DataFrame] = None,   # напр. твій std_by_ticker
    sigma_path: Optional[str] = None,          # або шлях до файла з цими даними
    batch_rows: int = BATCH_ROWS,
    compression: str = "snappy",
    insert_before: Optional[str] = None,       # напр. "Bench%"; якщо None — в кінець
    log_every_batches: int = 5,
    round_decimals: Optional[int] = 6,
) -> str:
    """
    Потоково додає колонку 'sigma' (за 'ticker') до великого Parquet.

    Як і в клітинці 28: нормалізація тікера і пошук у мапі йдуть по унікальних
    значеннях, а не по кожному з мільйонів рядків; per-batch merge не потрібен.
    """
    if pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow, pyarrow.parquet).")

    if not os.path.exists(base_path):
        raise FileNotFoundError(base_path)

    if out_path is None:
        folder, fname = os.path.split(base_path)
        name, ext = os.path.splitext(fname)
        out_path = os.path.join(folder, f"{name}_with_sigma{ext or '.parquet'}")

    sigma_map = _prepare_sigma_df(sigma_df=sigma_df, sigma_path=sigma_path)
    if round_decimals is not None:
        sigma_map["sigma"] = sigma_map["sigma"].round(round_decimals)
    sd = dict(zip(sigma_map["ticker"], sigma_map["sigma"]))

    tmp_path = out_path + f".tmp_{os.getpid()}_{int(time.time())}"
    pf = pq.ParquetFile(base_path)
    writer = None
    total = 0
    t0 = time.time()

    print(f"▶️ START merge sigma → {out_path}", flush=True)
    for i, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()
        if "ticker" not in df.columns:
            raise KeyError("У базовому файлі відсутня колонка 'ticker'.")

        codes, cats = _ticker_codes(df["ticker"])
        df["ticker"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)
        df["sigma"] = _map_by_ticker(codes, cats, sd, numeric=True)

        cols = [c for c in df.columns if c != "sigma"]
        if insert_before and insert_before in cols:
            idx = cols.index(insert_before)
            df = df[cols[:idx] + ["sigma"] + cols[idx:]]
        else:
            df = df[cols + ["sigma"]]

        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(tmp_path, table.schema, compression=compression)
        writer.write_table(table)

        total += len(df)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total:,}  elapsed={time.time()-t0:,.1f}s", flush=True)

    if writer is not None:
        writer.close()

    if os.path.exists(out_path):
        os.remove(out_path)
    os.replace(tmp_path, out_path)

    print(f"✅ DONE rows={total:,} → {out_path}  in {time.time()-t0:,.1f}s", flush=True)
    return out_path


In [58]:
def _build_rolling_lookup(path, round_decimals=4):
    """rolling_stats.parquet -> щільні масиви для миттєвого пошуку (ticker, date) -> bench/corr/beta/sigma."""
    df = pq.read_table(path).to_pandas()
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["date"] = df["date"].astype(str).str.slice(0, 10)
    df = df.drop_duplicates(["ticker", "date"], keep="last")
    tickers = np.array(sorted(df["ticker"].unique()), dtype=object)
    dates = np.array(sorted(df["date"].unique()), dtype=object)
    names = sorted({b for b in df["bench"].dropna().unique()})
    tid = pd.Index(tickers).get_indexer(df["ticker"])
    did = pd.Index(dates).get_indexer(df["date"])
    D = len(dates)
    size = len(tickers) * D
    pos = tid.astype(np.int64) * D + did
    bid = np.full(size, -1, dtype=np.int32)
    bid[pos] = df["bench"].map({b: i for i, b in enumerate(names)}).fillna(-1).astype("int32").to_numpy()
    arrs = {}
    for c in ("corr", "beta", "sigma"):
        v = pd.to_numeric(df[c], errors="coerce")
        if round_decimals is not None and c in ("corr", "beta"):
            v = v.round(round_decimals)
        a = np.full(size, np.nan)
        a[pos] = v.to_numpy(dtype="float64")
        arrs[c] = a
    return {"tickers": tickers, "dates": dates, "D": D, "bid": bid,
            "bench_names": np.array(names + [None], dtype=object), **arrs}


def _apply_rolling_lookup(df, codes, cats, lookup, suffix=""):
    """Додає bench/corr/beta/sigma ПО ДНЯХ: значення на (ticker, date) рядка (вікно = попередні торгові дні)."""
    D = lookup["D"]
    t_map = pd.Index(lookup["tickers"]).get_indexer(cats)                 # cats уже strip+upper
    tid = np.where(codes >= 0, t_map[np.maximum(codes, 0)], -1)
    dcodes, dcats = pd.factorize(df["date"].to_numpy(dtype=object))
    d_map = pd.Index(lookup["dates"]).get_indexer(dcats)
    did = np.where(dcodes >= 0, d_map[np.maximum(dcodes, 0)], -1)
    ok = (tid >= 0) & (did >= 0)
    pos = np.where(ok, tid.astype(np.int64) * D + did, 0)
    df["bench" + suffix] = lookup["bench_names"][np.where(ok, lookup["bid"][pos], -1)]     # -1 -> None
    for c in ("corr", "beta", "sigma"):
        df[c + suffix] = np.where(ok, lookup[c][pos], np.nan)


def build_final_in_one_pass(
    intraday_parquet_path: str,
    oc_parquet_path: str,
    corrbeta_parquet_path: str,
    etf_parquet_path: str,
    out_parquet_path: str,
    *,
    sigma_df=None,
    sigma_path=None,
    current_stats_path=None,   # якщо задано: bench/corr/beta/sigma = НАЙНОВІШІ значення тікера (станом на зараз) - їх читають стратегії
    rolling_stats_path=None,   # якщо задано: ІСТОРИЧНІ bench/corr/beta/sigma ПО ДНЯХ (ковзне вікно) у колонках *_day; з них рахується dev_sig
    batch_rows: int = BATCH_ROWS,
    round_decimals=4,
    stack_round_to=None,
    bench_round_decimals=2,
    compression: str = "snappy",
    log_every_batches: int = 5,
) -> str:
    """Один потоковий прохід замість п'яти: oc-мердж → Stack% → bench/corr/beta
    → Bench% → sigma. Замінює merge_intraday_and_oc_to_file +
    add_pct_vs_prev_close_parquet + attach_bench_corr_beta_parquet +
    merge_bench_pct_from_etf + add_sigma_by_ticker_to_parquet.

    Результат побайтово збігається з ланцюжком; економія — у тому, що ~3 ГБ
    читаються й пишуться один раз, а не шість.
    """
    t0 = time.time()

    # ---------- side-входи: готуються ОДИН раз ----------
    oc_prepared = _prep_oc_prev_close(pq.read_table(oc_parquet_path).to_pandas())

    cur_maps = None
    if current_stats_path:
        cs = pq.read_table(current_stats_path).to_pandas()
        cs["ticker"] = cs["ticker"].astype(str).str.strip().str.upper()
        cs = cs.drop_duplicates("ticker", keep="last")
        for c in ("corr", "beta", "sigma"):
            cs[c] = pd.to_numeric(cs[c], errors="coerce")
            if round_decimals is not None and c in ("corr", "beta"):
                cs[c] = cs[c].round(round_decimals)
        cur_maps = {c: dict(zip(cs["ticker"], cs[c])) for c in ("bench", "corr", "beta", "sigma")}
        print(f"ℹ️ значення станом на зараз: {len(cs):,} тікерів (current_stats)", flush=True)

    lookup = None
    if rolling_stats_path:
        lookup = _build_rolling_lookup(rolling_stats_path, round_decimals)
        print(f"ℹ️ ковзна статистика: {len(lookup['tickers']):,} тікерів x {lookup['D']:,} днів "
              f"({lookup['dates'][0]} .. {lookup['dates'][-1]})", flush=True)
    else:
        mapping = _load_corrbeta_mapping_parquet(corrbeta_parquet_path)
        if mapping.empty:
            raise ValueError(
                f"Мапа bench/corr/beta порожня: {corrbeta_parquet_path}. "
                "Крок побудови x_to_best_etf_by_lvl3 не дав жодного рядка."
            )
        if round_decimals is not None:
            mapping["corr"] = mapping["corr"].round(round_decimals)
            mapping["beta"] = mapping["beta"].round(round_decimals)
        m_bench = dict(zip(mapping["ticker"], mapping["bench"]))
        m_corr = dict(zip(mapping["ticker"], mapping["corr"]))
        m_beta = dict(zip(mapping["ticker"], mapping["beta"]))

    etf_names = [f.name for f in pq.ParquetFile(etf_parquet_path).schema_arrow]
    cl = {c.lower().strip(): c for c in etf_names}
    val_col = cl.get("bench%") or cl.get("pct_vs_prev_close")
    etf_map = pq.read_table(etf_parquet_path,
                            columns=[cl["ticker"], cl["dt"], val_col]).to_pandas()
    etf_map.columns = ["bench", "dt", "Bench%"]
    etf_map["bench"] = etf_map["bench"].astype(str).str.strip().str.upper()
    etf_map["Bench%"] = pd.to_numeric(etf_map["Bench%"], errors="coerce")
    if bench_round_decimals is not None:
        etf_map["Bench%"] = etf_map["Bench%"].round(bench_round_decimals)
    etf_map["dt"] = etf_map["dt"].astype(str)
    etf_map = (etf_map.dropna(subset=["dt"])
                      .sort_values(["bench", "dt"], kind="stable")
                      .drop_duplicates(subset=["bench", "dt"], keep="last")
                      .reset_index(drop=True))
    # Bench% беремо за бенчмарком ЦЬОГО дня (bench_day), якщо є і історичні, і найновіші значення
    bkey = "bench_day" if (lookup is not None and cur_maps is not None) else "bench"
    etf_map_k = etf_map if bkey == "bench" else etf_map.rename(columns={"bench": bkey})

    if lookup is None:
        sig = _prepare_sigma_df(sigma_df=sigma_df, sigma_path=sigma_path)
        m_sigma = dict(zip(sig["ticker"], sig["sigma"]))

    if os.path.exists(out_parquet_path):
        os.remove(out_parquet_path)

    ORDER = ["ticker", "dt", "o", "h", "l", "c", "v", "date", "open", "prev_close",
             "Stack%", "bench", "corr", "beta", "Bench%", "sigma",
             "bench_day", "corr_day", "beta_day", "sigma_day"]

    pf = pq.ParquetFile(intraday_parquet_path)
    writer = None
    total_rows = 0
    print(f"▶️ START one-pass → {out_parquet_path}", flush=True)

    for i, batch in enumerate(pf.iter_batches(batch_size=batch_rows), 1):
        df = batch.to_pandas()

        # 1) день + open/prev_close (ключ — сирий ticker, як у ланцюжку)
        df = df[df["dt"].notna()]
        df = df.assign(date=_date_from_dt(df["dt"]))
        df = df.merge(oc_prepared, on=["ticker", "date"], how="left")

        # 2) Stack%
        num = pd.to_numeric(df["o"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        den = pd.to_numeric(df["prev_close"], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            pct = np.where(den == 0, np.nan, num / den - 1.0) * 100.0
        if stack_round_to is not None:
            pct = np.round(pct, int(stack_round_to))
        df["Stack%"] = pct

        # 3) bench/corr/beta (тут ticker нормалізується — як у ланцюжку)
        codes, cats = _ticker_codes(df["ticker"])
        df["ticker"] = _expand(np.asarray(cats.to_numpy(), dtype=object), codes, None)
        if lookup is not None:
            # ІСТОРИЧНІ bench/corr/beta/sigma ПО ДНЯХ: вікно = попередні торгові дні (клітинка 42).
            # Якщо є і найновіші значення - історичні йдуть у *_day, інакше прямо в стандартні колонки.
            _apply_rolling_lookup(df, codes, cats, lookup, suffix=("_day" if cur_maps is not None else ""))
        else:
            df["bench"] = _map_by_ticker(codes, cats, m_bench, numeric=False)
            df["corr"] = _map_by_ticker(codes, cats, m_corr, numeric=True)
            df["beta"] = _map_by_ticker(codes, cats, m_beta, numeric=True)

            # 4) sigma — теж по codes/cats, тому рахуємо ДО мерджа
            df["sigma"] = _map_by_ticker(codes, cats, m_sigma, numeric=True)

        # 4b) НАЙНОВІШІ значення тікера (станом на зараз, однакові в усіх рядках тікера) -> стандартні
        #     bench/corr/beta/sigma: саме їх читають стратегії як статичні параметри тікера
        if cur_maps is not None:
            df["bench"] = _map_by_ticker(codes, cats, cur_maps["bench"], numeric=False)
            df["corr"] = _map_by_ticker(codes, cats, cur_maps["corr"], numeric=True)
            df["beta"] = _map_by_ticker(codes, cats, cur_maps["beta"], numeric=True)
            df["sigma"] = _map_by_ticker(codes, cats, cur_maps["sigma"], numeric=True)

        # 5) Bench% по (bench, dt); bench лишається object для мерджа
        df["dt"] = df["dt"].astype(str)
        df = df.merge(etf_map_k, on=[bkey, "dt"], how="left")

        # string-dtype один раз наприкінці: тримає тип bench стабільним між батчами
        for _bc in ("bench", "bench_day"):
            if _bc in df.columns:
                df[_bc] = pd.array(df[_bc], dtype="string")

        df = df[[c for c in ORDER if c in df.columns]]
        table = pa.Table.from_pandas(df, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet_path, table.schema, compression=compression)
        elif not table.schema.equals(writer.schema, check_metadata=False):
            table = table.cast(writer.schema)
        writer.write_table(table)

        total_rows += len(df)
        if i % log_every_batches == 0:
            print(f"[batch {i:>4}] rows={total_rows:,} elapsed={time.time()-t0:,.1f}s", flush=True)
        del df, table, batch
        gc.collect()

    if writer is not None:
        writer.close()
    print(f"✅ DONE rows={total_rows:,} → {out_parquet_path} in {time.time()-t0:,.1f}s", flush=True)
    return out_parquet_path


In [59]:
# Один прохід: oc-мердж → Stack% → bench/corr/beta/sigma → Bench% (за бенчмарком ЦЬОГО дня).
#   bench / corr / beta / sigma           = НАЙНОВІШІ значення тікера (станом на зараз, current_stats.parquet).
#                                           Саме їх читають стратегії як статичні параметри тікера.
#   bench_day / corr_day / beta_day / sigma_day = ІСТОРИЧНІ значення на (ticker, date) з rolling_stats.parquet:
#                                           вікно = ROLL_WINDOW попередніх торгових днів, без look-ahead.
#                                           Саме з них (а не з найновіших) рахуються t_p / deviation / dev_sig.
# Аварійне відновлення: якщо фінальний етап упав, а sigma-файл уже готовий, створіть порожній файл
#   CRACEN\work\RESUME_SIGMA.flag  -> цей прогін пропустить перебудову sigma-файла (~80 хв). Прапор одноразовий.
_resume_flag = WORK_DIR / "RESUME_SIGMA.flag"
_sigma_target = WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"
_resumed = False
if _resume_flag.exists():
    try:
        if _sigma_target.exists() and pq.ParquetFile(str(_sigma_target)).metadata.num_rows > 0:   # недописаний файл без футера не відкриється
            sigma_out_path = str(_sigma_target)
            _resumed = True
            print(f"⏩ RESUME_SIGMA: пропускаю перебудову, беру готовий {_sigma_target.name}")
        else:
            print("⚠️ RESUME_SIGMA.flag є, але sigma-файла немає - будую заново")
    except Exception as _e:
        print("⚠️ sigma-файл не відкривається (", _e, ") - будую заново")
    try:
        _resume_flag.unlink()
    except Exception:
        pass
if not _resumed:
    sigma_out_path = build_final_in_one_pass(
        intraday_parquet_path=str(WORK_DIR / "premarket_rows.parquet"),
        oc_parquet_path=str(WORK_DIR / "oc.parquet"),
        corrbeta_parquet_path=None,
        etf_parquet_path=str(WORK_DIR / "premarket_etf_rows_enriched_with_pct.parquet"),
        out_parquet_path=str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"),
        rolling_stats_path=str(WORK_DIR / "rolling_stats.parquet"),
        current_stats_path=str(WORK_DIR / "current_stats.parquet"),
        batch_rows=BATCH_ROWS,
        round_decimals=4,
        bench_round_decimals=2,
        compression="snappy",
    )

# Validation 1: *_day збігаються з rolling_stats на (ticker, date)
_rs = pd.read_parquet(str(WORK_DIR / "rolling_stats.parquet"))
_rs["ticker"] = _rs["ticker"].astype(str).str.upper().str.strip()
_rs["corr"] = _rs["corr"].round(4)
_rs["beta"] = _rs["beta"].round(4)
_pf = pq.ParquetFile(sigma_out_path)
_rg = list(range(min(3, _pf.num_row_groups)))
_chk = _pf.read_row_groups(_rg, columns=["ticker", "date", "bench", "corr", "beta", "sigma",
                                          "bench_day", "corr_day", "beta_day", "sigma_day"]).to_pandas()
_m = _chk.drop_duplicates(["ticker", "date"]).merge(_rs, on=["ticker", "date"], how="left", suffixes=("", "_exp"))
_has = _m["sigma_exp"].notna() | _m["corr_exp"].notna()
if not _has.any():
    raise ValueError("Жоден (ticker, date) з файлу не знайшовся в rolling_stats - перевірте вікна дат")
_d = max((_m.loc[_has, c + "_day"] - _m.loc[_has, c + "_exp"]).abs().max() for c in ("sigma", "corr", "beta"))
_bad = int((_m.loc[_has, "bench_day"].astype(str) != _m.loc[_has, "bench_exp"].astype(str)).sum())
print("sigma_out_path:", sigma_out_path)
print("історичні *_day: перевірено (ticker, date):", int(_has.sum()), "| max diff:", _d, "| bench_day ≠ rolling:", _bad)
if _bad or _d > 1e-9:
    raise ValueError("Колонки *_day у файлі не збігаються з rolling_stats.parquet")

# Validation 2: bench/corr/beta/sigma = найновіші, однакові в усіх рядках тікера, збігаються з current_stats
_cs = pd.read_parquet(str(WORK_DIR / "current_stats.parquet"))
_cs["ticker"] = _cs["ticker"].astype(str).str.upper().str.strip()
_cs["corr"] = _cs["corr"].round(4)
_cs["beta"] = _cs["beta"].round(4)
_nun = int(_chk.groupby("ticker")[["corr", "beta", "sigma"]].nunique(dropna=False).max().max())
_mm = _chk.drop_duplicates("ticker").merge(_cs, on="ticker", how="left", suffixes=("", "_exp"))
_hn = _mm["sigma_exp"].notna() | _mm["corr_exp"].notna()
_dn = max((_mm.loc[_hn, c] - _mm.loc[_hn, c + "_exp"]).abs().max() for c in ("sigma", "corr", "beta"))
_bn = int((_mm.loc[_hn, "bench"].astype(str) != _mm.loc[_hn, "bench_exp"].astype(str)).sum())
print("найновіші bench/corr/beta/sigma: тікерів у вибірці", len(_mm), "| різних значень на тікер:", _nun,
      "| max diff:", _dn, "| bench ≠ current:", _bn)
if _nun > 1 or _bn or _dn > 1e-9:
    raise ValueError("Стандартні bench/corr/beta/sigma у файлі не збігаються з current_stats.parquet")

# premarket_rows.parquet (десятки ГБ) більше не потрібен: з нього вже зібрано sigma-файл -> звільняємо диск
if CLEANUP_INTERMEDIATE:
    _pm = WORK_DIR / "premarket_rows.parquet"
    if _pm.exists():
        _sz = _pm.stat().st_size / 1024**3
        _pm.unlink()
        print(f"🧹 premarket_rows.parquet видалено, звільнено {_sz:.1f} ГБ")
ram_guard("після build_final_in_one_pass")


ℹ️ значення станом на зараз: 7,140 тікерів (current_stats)
ℹ️ ковзна статистика: 7,108 тікерів x 687 днів (2024-02-20 .. 2026-11-12)
▶️ START one-pass → C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet
[batch    5] rows=10,000,000 elapsed=47.8s
[batch   10] rows=20,000,000 elapsed=79.4s
[batch   15] rows=30,000,000 elapsed=110.0s
[batch   20] rows=40,000,000 elapsed=140.5s
[batch   25] rows=50,000,000 elapsed=171.4s
[batch   30] rows=60,000,000 elapsed=201.8s
[batch   35] rows=70,000,000 elapsed=231.7s
[batch   40] rows=80,000,000 elapsed=262.1s
[batch   45] rows=90,000,000 elapsed=292.5s
[batch   50] rows=100,000,000 elapsed=324.4s
[batch   55] rows=110,000,000 elapsed=356.2s
[batch   60] rows=120,000,000 elapsed=388.1s
[batch   65] rows=130,000,000 elapsed=419.7s
[batch   70] rows=140,000,000 elapsed=452.0s
[batch   75] rows=150,000,000 elapsed=482.9s
[batch   80] rows=160,000,000 elapsed=514.5s
[batch   85] rows=170,000,000 elapsed=547.1s

In [60]:
peek_parquet(str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"), n=21150)



Файл: C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet
Рядків (meta): 1,223,499,149
Колонок: 20
Схема: ticker, dt, o, h, l, c, v, date, open, prev_close, Stack%, bench, corr, beta, Bench%, sigma, bench_day, corr_day, beta_day, sigma_day
NaN у 'bench' (у прев'ю): 0/21150
NaN у 'corr' (у прев'ю): 0/21150
NaN у 'beta' (у прев'ю): 0/21150

Перші рядки:
      ticker                   dt           o           h           l           c          v        date    open  prev_close        Stack% bench    corr    beta  Bench%     sigma bench_day  \
0        AEP  2026-05-04 00:57:00  137.490005  137.490005  137.490005  137.490005        1.0  2026-05-04  136.40      136.93  4.089721e-01   XLU  0.6261  0.4537     NaN  0.468532       XLU   
1        AEP  2026-05-04 00:58:00  137.490005  137.490005  137.490005  137.490005        3.0  2026-05-04  136.40      136.93  4.089721e-01   XLU  0.6261  0.4537     NaN  0.468532       XLU   
2        AEP  2026-05-04 01:

,ticker,dt,o,h,l,c,v,date,open,prev_close,Stack%,bench,corr,beta,Bench%,sigma,bench_day,corr_day,beta_day,sigma_day
0,AEP,2026-05-04 00:57:00,137.490005,137.490005,137.490005,137.490005,1.0,2026-05-04,136.40,136.93,0.408972,XLU,0.6261,0.4537,NaN,0.468532,XLU,0.6480,0.5553,0.455516
1,AEP,2026-05-04 00:58:00,137.490005,137.490005,137.490005,137.490005,3.0,2026-05-04,136.40,136.93,0.408972,XLU,0.6261,0.4537,NaN,0.468532,XLU,0.6480,0.5553,0.455516
2,AEP,2026-05-04 01:03:00,137.119995,137.119995,136.690002,136.750000,32.0,2026-05-04,136.40,136.93,0.138753,XLU,0.6261,0.4537,0.37,0.468532,XLU,0.6480,0.5553,0.455516
3,AEP,2026-05-04 01:37:00,136.770004,136.770004,136.770004,136.770004,1.0,2026-05-04,136.40,136.93,-0.116845,XLU,0.6261,0.4537,NaN,0.468532,XLU,0.6480,0.5553,0.455516
4,AEP,2026-05-04 02:00:00,136.929993,136.929993,136.779999,136.779999,2.0,2026-05-04,136.40,136.93,-0.000005,XLU,0.6261,0.4537,NaN,0.468532,XLU,0.6480,0.5553,0.455516
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21145,AG,2026-05-26 14:35:00,20.110001,20.110001,20.090000,20.105000,11430.0,2026-05-26,19.81,19.44,3.446505,GDX,0.9683,1.1483,3.38,2.363057,GDX,0.8826,1.0042,2.330115
21146,AG,2026-05-26 14:36:00,20.105000,20.110100,20.090000,20.090099,11898.0,2026-05-26,19.81,19.44,3.420780,GDX,0.9683,1.1483,3.33,2.363057,GDX,0.8826,1.0042,2.330115
21147,AG,2026-05-26 14:37:00,20.094999,20.115000,20.090000,20.100000,4497.0,2026-05-26,19.81,19.44,3.369338,GDX,0.9683,1.1483,3.27,2.363057,GDX,0.8826,1.0042,2.330115
21148,AG,2026-05-26 14:38:00,20.108999,20.108999,20.075001,20.080000,11465.0,2026-05-26,19.81,19.44,3.441354,GDX,0.9683,1.1483,3.33,2.363057,GDX,0.8826,1.0042,2.330115


In [61]:
import os, time, gc
import pandas as pd
import numpy as np
from typing import Optional, List

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.dataset as ds
    import pyarrow.compute as pc
except Exception:
    pa = pq = ds = pc = None


def _as_f64(col):
    """ChunkedArray -> float64 без походу в pandas (fallback лише для нечислових типів)."""
    if pa.types.is_floating(col.type) or pa.types.is_integer(col.type):
        return col.cast(pa.float64()) if col.type != pa.float64() else col
    if pa.types.is_decimal(col.type):
        return col.cast(pa.float64())
    return pa.chunked_array([pa.array(
        pd.to_numeric(col.to_pandas(), errors="coerce").to_numpy(dtype="float64"))])


def _round_like_numpy(col, ndigits):
    """np.round над Arrow-колонкою + NaN -> null (як робив pa.Table.from_pandas)."""
    if len(col) == 0:
        return col
    v = col.to_numpy(zero_copy_only=False)   # може бути read-only zero-copy view
    if ndigits is not None:
        v = np.round(v, int(ndigits))
    return pa.array(v, from_pandas=True)


def _add_tp_deviation_devsig(table, *, round_tp_dev=2, round_devsig=6, dt_as_timestamp=False):
    """t_p / deviation / dev_sig як арифметика Arrow над колонками таблиці.

    Раніше кожен батч ходив у pandas і назад (batch.to_pandas() +
    pa.Table.from_pandas()). Для рядкових колонок ticker/dt/date/bench це означало
    матеріалізацію мільйонів python-об'єктів str на кожному батчі й повне
    перезбирання Arrow-масивів — найдорожча частина того циклу. Тут таблиця
    лишається в Arrow від читання до запису.
    """
    names = {n.strip().lower(): n for n in table.column_names}
    stack_col = names.get("stack%") or names.get("pct_vs_prev_close")
    bench_col = names.get("bench%")
    # відхилення рахується ІСТОРИЧНИМИ (по днях) beta/sigma, якщо вони є (*_day); інакше - стандартними
    beta_col = names.get("beta_day") or names.get("beta")
    sigma_col = names.get("sigma_day") or names.get("sigma")

    need = {
        "stack%/pct_vs_prev_close": stack_col,
        "bench%": bench_col,
        "beta": beta_col,
        "sigma": sigma_col,
    }
    missing = [k for k, v in need.items() if v is None]
    if missing:
        raise KeyError(f"У вхідному файлі бракує колонок: {missing}")

    drop = [c for c in ("t_p", "deviation", "dev_sig") if c in table.column_names]
    if drop:
        table = table.drop_columns(drop) if hasattr(table, "drop_columns") else table.drop(drop)

    stack = _as_f64(table[stack_col])
    beta = _as_f64(table[beta_col])
    bench = _as_f64(table[bench_col])
    sigma = _as_f64(table[sigma_col])

    tp = pc.multiply(beta, bench)      # очікуваний рух акції = beta * Bench%
    dev = pc.subtract(stack, tp)       # відхилення = Stack% - beta * Bench%
    denom = pc.if_else(pc.equal(sigma, 0), pa.scalar(None, pa.float64()), sigma)  # sigma 0 -> NaN
    dsg = pc.divide(dev, denom)

    # Округлення саме через np.round, а не pc.round: на значеннях, які вже округлені
    # вище по пайплайну (Stack% до 2 знаків, beta до 4), добуток лягає впритул до
    # межі .005, і pc.round подекуди дає результат на 1 ULP інший за np.round.
    # np.round тут ще й трохи швидший, а pa.array(..., from_pandas=True) відтворює
    # ту саму семантику NaN -> null, що давав старий pa.Table.from_pandas().
    tp = _round_like_numpy(tp, round_tp_dev)
    dev = _round_like_numpy(dev, round_tp_dev)
    dsg = _round_like_numpy(dsg, round_devsig)

    if dt_as_timestamp and "dt" in table.column_names and pa.types.is_string(table["dt"].type):
        i = table.column_names.index("dt")
        table = table.set_column(i, "dt", pc.strptime(table["dt"], format="%Y-%m-%d %H:%M:%S",
                                                      unit="ns", error_is_null=True))

    return (table.append_column("t_p", tp)
                 .append_column("deviation", dev)
                 .append_column("dev_sig", dsg))


def _sort_keys(schema_names):
    keys = [("ticker", "ascending")]
    if "date" in schema_names:
        keys.append(("date", "ascending"))
    if "dt" in schema_names:
        keys.append(("dt", "ascending"))
    return keys


_FINAL_STATS = {"dups_removed": 0, "conflicts": 0}


def _drop_sorted_duplicates(table):
    """table ВЖЕ відсортована за (ticker, date, dt), тож дублікати (ticker, dt) лежать поруч - лишаємо перший.
    Причина дублікатів: той самий тікер-день потрапив у два файли-батчі з різних запусків; без цього кроку
    у фінальному файлі обсяги/середні подвоюються. Окремо рахуємо дублікати з РІЗНИМИ o/h/l/c/v (мають бути 0)."""
    n = table.num_rows
    if n < 2 or "ticker" not in table.column_names or "dt" not in table.column_names:
        return table
    tk = table["ticker"].combine_chunks()
    dt = table["dt"].combine_chunks()
    same = pc.fill_null(pc.and_kleene(pc.equal(tk.slice(1), tk.slice(0, n - 1)),
                                      pc.equal(dt.slice(1), dt.slice(0, n - 1))), False)
    ndup = pc.sum(pc.cast(same, pa.int64())).as_py() or 0
    if ndup == 0:
        return table
    diff = None
    for c in ("o", "h", "l", "c", "v"):
        if c in table.column_names:
            a = table[c].combine_chunks()
            d = pc.fill_null(pc.not_equal(a.slice(1), a.slice(0, n - 1)), False)
            diff = d if diff is None else pc.or_(diff, d)
    if diff is not None:
        _FINAL_STATS["conflicts"] += pc.sum(pc.cast(pc.and_(same, diff), pa.int64())).as_py() or 0
    _FINAL_STATS["dups_removed"] += ndup
    keep = pa.concat_arrays([pa.array([True]), pc.invert(same)])
    return table.filter(keep)


def _trim_ram():
    """gc + повернути ОС невикористану пам'ять Arrow-пулу (mimalloc на Windows тримає вільні сторінки,
    а Job-ліміт рахує саме їх -> без цього ArrowMemoryError навіть коли реально все звільнено)."""
    gc.collect()
    try:
        pa.default_memory_pool().release_unused()
    except Exception:
        pass


def _scatter_to_bucket_parts(dataset, chunksize, ticker_set, rank_to_bucket_arr, n_buckets,
                             tmp_out, compression, flush_rows, flush_bytes=None):
    """Розкладає рядки по бакетах-тикерах БЕЗ відкритих ParquetWriter-ів.

    Раніше тримали ~170 ParquetWriter одночасно: кожен має свої буфери колонок/словників
    (~десятки МБ) -> кілька ГБ + читання з readahead=16 батчів -> ArrowMemoryError.
    Тепер: накопичуємо шматки по бакетах у RAM до flush_rows рядків, потім скидаємо кожен
    бакет в окремий part-файл і закриваємо. Порядок рядків усередині бакета зберігається.
    Повертає список списків шляхів (по бакету)."""
    _trim_ram()
    if flush_bytes is None:
        flush_bytes = globals().get("SORT_FLUSH_BYTES", 1_500_000_000)
    _md = globals().get("mem_diag", lambda l="": "")
    print(_md("старт розкладання"), flush=True)
    parts = [[] for _ in range(n_buckets)]
    buf = [[] for _ in range(n_buckets)]
    st = {"rows": 0, "seq": 0}

    def _flush():
        for b in range(n_buckets):
            if buf[b]:
                p = tmp_out + f".b{b}.p{st['seq']}"
                pq.write_table(pa.concat_tables(buf[b]), p, compression=compression)
                parts[b].append(p)
                buf[b] = []
        st["rows"] = 0
        st["seq"] += 1
        _trim_ram()
        if st["seq"] % 5 == 1:
            print(_md(f"flush #{st['seq']}"), flush=True)

    try:
        for rb in dataset.to_batches(batch_size=chunksize, batch_readahead=2, fragment_readahead=1):
            if rb.num_rows == 0:
                continue
            tbl = pa.Table.from_batches([rb])
            ranks = pc.index_in(tbl.column("ticker").cast(pa.string()), value_set=ticker_set)
            bn = pc.fill_null(pc.take(rank_to_bucket_arr, ranks), n_buckets).to_numpy(zero_copy_only=False)
            order = np.argsort(bn, kind="stable")
            counts = np.bincount(bn, minlength=n_buckets + 1)
            tbl = tbl.take(pa.array(order))
            off = 0
            for b in np.nonzero(counts)[0]:
                n = int(counts[b])
                if b < n_buckets:          # b == n_buckets: рядки без тикера - відкидаються (як і раніше)
                    buf[b].append(tbl.slice(off, n))
                off += n
            st["rows"] += rb.num_rows
            del tbl, rb
            if st["rows"] >= flush_rows or pa.total_allocated_bytes() >= flush_bytes:
                _flush()
        _flush()
    except BaseException as _e:
        print(f"❌ розкладання перервано: {type(_e).__name__}; " + _md("у момент помилки")
              + f" | рядків у буфері={st['rows']:,} | flush #{st['seq']}", flush=True)
        for lst in parts:
            for p in lst:
                if os.path.exists(p):
                    os.remove(p)
        raise
    return parts


def _read_bucket_parts(paths):
    """Читає всі part-файли одного бакета (у порядку запису) і видаляє їх."""
    t = pa.concat_tables([pq.read_table(p) for p in paths])
    for p in paths:
        os.remove(p)
    return t


def recompute_tp_deviation_devsig(
    input_path: str = "CRACEN/work/premarket_rows_with_bench_pct_with_sigma.parquet",
    output_path: str = "CRACEN/work/premarket_rows_with_bench_pct_with_sigma_v2.parquet",
    chunksize: int = BATCH_ROWS,   # batch_rows для потокового режиму / бакетування
    round_tp_dev: Optional[int] = 2,
    round_devsig: Optional[int] = 6,
    compression: str = "snappy",
    *,
    sort_final_by_ticker: bool = True,
    max_rows_in_memory: int = SORT_ROWS_IN_MEMORY,
    dt_as_timestamp: bool = False,
    row_group_size: int = 2_000_000,
) -> str:
    """
    Перерахунок:
      - t_p       = beta * bench%        (очікуваний рух акції)
      - deviation = stack% - t_p         (>0: акція випередила бенчмарк)
      - dev_sig   = deviation / sigma  (0 -> NaN)
    Стирає старі t_p / deviation / dev_sig, якщо були.

    Якщо sort_final_by_ticker=True, output_path фізично відсортований:
    ticker A→Z, всередині тикера date,dt.

    Що змінилось проти попередньої версії:
      1. Обчислення і сортування — ОДИН прохід. Раніше спершу писався повний
         tmp_unsorted, потім він читався назад і переписувався відсортованим:
         3 читання + 3 записи всього файла. Тепер 1 читання + 1 запис
         (або 2+2, якщо файл не влазить у пам'ять і потрібні бакети).
      2. Уся арифметика в Arrow — без batch.to_pandas()/pa.Table.from_pandas()
         на кожному батчі (це матеріалізувало рядкові колонки як python-об'єкти).
      3. Сортування ПЕРЕД додаванням t_p/deviation/dev_sig — take() переставляє
         на 3 float64-колонки менше.
      4. Бакетування по тикерах — через pc.index_in замість pandas .map по рядках.
      5. row_group_size=2M замість дефолтного 1M: менше row-груп → швидший запис,
         трохи менший файл, і стратегії-споживачі (вони читають
         iter_batches(batch_size=1_000_000)) теж читають його швидше.
    """
    if pq is None or pa is None or ds is None:
        raise RuntimeError("Потрібен pyarrow (pyarrow, pyarrow.parquet, pyarrow.dataset).")

    if not os.path.exists(input_path):
        raise FileNotFoundError(input_path)

    t0 = time.time()
    _FINAL_STATS.update(dups_removed=0, conflicts=0)
    dataset = ds.dataset(input_path, format="parquet")
    schema_names = [f.name for f in dataset.schema]
    total_rows = dataset.count_rows()

    tmp_out = output_path + f".tmp_{os.getpid()}_{int(time.time())}"
    if os.path.exists(tmp_out):
        os.remove(tmp_out)

    def _finish(rows, mode):
        if os.path.exists(output_path):
            os.remove(output_path)
        os.replace(tmp_out, output_path)
        print(f"✅ DONE ({mode}): {rows:,} rows → {output_path} | "
              f"total_time={time.time()-t0:,.1f}s", flush=True)
        return output_path

    # ---------- 1) без сортування: чистий стрім, один прохід ----------
    if not sort_final_by_ticker:
        print(f"▶ START (stream): {input_path} → {output_path} | "
              f"rows={total_rows:,} batch={chunksize:,}", flush=True)
        writer = None
        written = 0
        try:
            for bi, rb in enumerate(dataset.to_batches(batch_size=chunksize), 1):
                if rb.num_rows == 0:
                    continue
                tbl = _add_tp_deviation_devsig(
                    pa.Table.from_batches([rb]), round_tp_dev=round_tp_dev,
                    round_devsig=round_devsig, dt_as_timestamp=dt_as_timestamp)
                if writer is None:
                    writer = pq.ParquetWriter(tmp_out, tbl.schema, compression=compression)
                writer.write_table(tbl, row_group_size=row_group_size)
                written += tbl.num_rows
                if bi % 5 == 0:
                    print(f"[batch {bi:>4}] rows={written:,}  "
                          f"elapsed={time.time()-t0:,.1f}s", flush=True)
        finally:
            if writer is not None:
                writer.close()
        return _finish(written, "stream")

    keys = _sort_keys(schema_names)
    if "ticker" not in schema_names:
        raise KeyError("У вхідному parquet немає колонки 'ticker' — неможливо відсортувати по тикерах.")

    # ---------- 2) влазить у пам'ять: читання → сорт → рахунок → запис ----------
    if total_rows <= max_rows_in_memory:
        print(f"▶ START (in-memory sort): {input_path} → {output_path} | "
              f"rows={total_rows:,}", flush=True)
        table = _drop_sorted_duplicates(dataset.to_table().sort_by(keys))
        table = _add_tp_deviation_devsig(
            table, round_tp_dev=round_tp_dev, round_devsig=round_devsig,
            dt_as_timestamp=dt_as_timestamp)
        pq.write_table(table, tmp_out, compression=compression,
                       row_group_size=row_group_size)
        written = table.num_rows
        del table
        _trim_ram()
        return _finish(written, "in-memory sort")

    # ---------- 3) не влазить: один прохід у бакети, далі сорт кожного ----------
    n_buckets = int(np.ceil(total_rows / float(max_rows_in_memory)))
    _tk = set()                      # унікальні тікери батчами, без читання всієї колонки в RAM
    for _rb in dataset.to_batches(columns=["ticker"], batch_size=chunksize):
        _tk.update(pc.unique(_rb.column(0)).to_pylist())
    tickers = list(_tk)
    tickers = sorted(str(t) for t in tickers if t is not None)
    # суміжні діапазони тикерів → порядок бакетів = порядок тикерів
    edges = np.array_split(np.arange(len(tickers)), n_buckets)
    rank_to_bucket = np.zeros(len(tickers), dtype="int32")
    for b, idxs in enumerate(edges):
        rank_to_bucket[idxs] = b
    ticker_set = pa.array(tickers, type=pa.string())
    rank_to_bucket_arr = pa.array(rank_to_bucket)

    print(f"▶ START (bucketed sort): {input_path} → {output_path} | rows={total_rows:,} "
          f"tickers={len(tickers):,} buckets={n_buckets}", flush=True)

    bucket_paths = _scatter_to_bucket_parts(
        dataset, chunksize, ticker_set, rank_to_bucket_arr, n_buckets,
        tmp_out, compression, flush_rows=2 * max_rows_in_memory,
        flush_bytes=globals().get("SORT_FLUSH_BYTES"))

    writer = None
    written = 0
    try:
        for b, bp in enumerate(bucket_paths):
            if not bp:
                continue
            tbl = _drop_sorted_duplicates(_read_bucket_parts(bp).sort_by(keys))
            tbl = _add_tp_deviation_devsig(
                tbl, round_tp_dev=round_tp_dev, round_devsig=round_devsig,
                dt_as_timestamp=dt_as_timestamp)
            if writer is None:
                writer = pq.ParquetWriter(tmp_out, tbl.schema, compression=compression)
            writer.write_table(tbl, row_group_size=row_group_size)
            written += tbl.num_rows
            del tbl
            _trim_ram()
            print(f"  bucket {b+1}/{n_buckets}  written={written:,}  "
                  f"elapsed={time.time()-t0:,.1f}s", flush=True)
    finally:
        if writer is not None:
            writer.close()
        for lst in bucket_paths:
            for bp in lst:
                if os.path.exists(bp):
                    os.remove(bp)

    return _finish(written, "bucketed sort")


def _write_final_sorted_alphabetical(
    input_path: str,
    output_path: str,
    *,
    compression: str = "snappy",
    max_rows_in_memory: int = SORT_ROWS_IN_MEMORY,
    log_every: int = 200,            # лишено для сумісності виклику
    roundtrip_pandas: bool = True,   # лишено для сумісності виклику
):
    """Лише сортування parquet: ticker A→Z, всередині тикера (date, dt).

    Обчислення t_p/deviation/dev_sig тепер робить сам recompute_tp_deviation_devsig
    у тому ж проході, тож окремо ця функція потрібна лише якщо треба відсортувати
    вже готовий файл.
    """
    if ds is None or pq is None or pa is None:
        raise RuntimeError("Потрібен pyarrow.dataset (pyarrow>=6).")

    dataset = ds.dataset(input_path, format="parquet")
    schema_names = [f.name for f in dataset.schema]
    if "ticker" not in schema_names:
        raise KeyError("У вхідному parquet немає колонки 'ticker' — неможливо відсортувати по тикерах.")
    keys = _sort_keys(schema_names)

    total_rows = dataset.count_rows()
    t0 = time.time()
    tmp_out = output_path + f".tmp_sorted_{os.getpid()}_{int(time.time())}"
    if os.path.exists(tmp_out):
        os.remove(tmp_out)

    if total_rows <= max_rows_in_memory:
        print(f"▶ SORT (in-memory): rows={total_rows:,}", flush=True)
        table = _drop_sorted_duplicates(dataset.to_table().sort_by(keys))
        pq.write_table(table, tmp_out, compression=compression)
        written = table.num_rows
        del table
        _trim_ram()
    else:
        n_buckets = int(np.ceil(total_rows / float(max_rows_in_memory)))
        _tk = set()
        for _rb in dataset.to_batches(columns=["ticker"], batch_size=BATCH_ROWS):
            _tk.update(pc.unique(_rb.column(0)).to_pylist())
        tickers = list(_tk)
        tickers = sorted(str(t) for t in tickers if t is not None)
        rank_to_bucket = np.zeros(len(tickers), dtype="int32")
        for b, idxs in enumerate(np.array_split(np.arange(len(tickers)), n_buckets)):
            rank_to_bucket[idxs] = b
        ticker_set = pa.array(tickers, type=pa.string())
        rank_to_bucket_arr = pa.array(rank_to_bucket)
        print(f"▶ SORT (bucketed): rows={total_rows:,} tickers={len(tickers):,} "
              f"buckets={n_buckets}", flush=True)

        bucket_paths = _scatter_to_bucket_parts(
            dataset, BATCH_ROWS, ticker_set, rank_to_bucket_arr, n_buckets,
            tmp_out, compression, flush_rows=2 * max_rows_in_memory,
        flush_bytes=globals().get("SORT_FLUSH_BYTES"))

        writer = None
        written = 0
        try:
            for b, bp in enumerate(bucket_paths):
                if not bp:
                    continue
                tbl = _drop_sorted_duplicates(_read_bucket_parts(bp).sort_by(keys))
                if writer is None:
                    writer = pq.ParquetWriter(tmp_out, tbl.schema, compression=compression)
                writer.write_table(tbl)
                written += tbl.num_rows
                del tbl
                _trim_ram()
        finally:
            if writer is not None:
                writer.close()
            for lst in bucket_paths:
                for bp in lst:
                    if os.path.exists(bp):
                        os.remove(bp)

    if os.path.exists(output_path):
        os.remove(output_path)
    os.replace(tmp_out, output_path)
    print(f"✅ SORT DONE: {written:,} rows → {output_path} | time={time.time()-t0:,.1f}s", flush=True)
    return output_path


def validate_final_parquet(path, *, expected_rows=None, start_date=None, end_date=None,
                           batch_rows=1_000_000, raise_on_error=True, verbose=True):
    """Потокова перевірка (пам'ять = один батч) що фінальний parquet придатний для аналізу:
      1. рядки впорядковані: ticker A→Z, далі date, далі dt (нестрого зростаюче, в т.ч. між батчами);
      2. немає дублікатів (ticker, dt);
      3. немає null у ticker/dt, а колонка date збігається з датою з dt;
      4. усі дати всередині вікна [start_date, end_date] і найновіший день присутній;
      5. кількість рядків збігається з вхідним файлом (expected_rows), тобто сортування нічого не з'їло.
    Повертає dict зі статистикою; при помилках кидає ValueError (raise_on_error=True)."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pyarrow.compute as pc

    pf = pq.ParquetFile(path)
    names = pf.schema_arrow.names
    for c in ("ticker", "dt"):
        if c not in names:
            raise KeyError(f"У {path} немає колонки '{c}'")
    has_date = "date" in names
    cols = ["ticker", "dt"] + (["date"] if has_date else [])

    rows = viol = dups = null_tk = null_dt = date_mismatch = 0
    first_bad = None
    days = set()
    tickers = 0
    prev = None
    last_ticker = None

    for rb in pf.iter_batches(batch_size=batch_rows, columns=cols):
        t = pa.Table.from_batches([rb])
        n = t.num_rows
        if n == 0:
            continue
        tk = t["ticker"].combine_chunks().cast(pa.string())
        dtc = t["dt"].combine_chunks()
        # timestamp -> date32 -> 'YYYY-MM-DD' (без strftime: на Windows він вимагає tz-базу)
        day = (dtc.cast(pa.date32()).cast(pa.string()) if pa.types.is_timestamp(dtc.type)
               else pc.utf8_slice_codeunits(dtc.cast(pa.string()), 0, 10))
        null_tk += tk.null_count
        null_dt += dtc.null_count
        days.update(pc.unique(day).drop_null().to_pylist())
        if has_date:
            dcol = t["date"].combine_chunks().cast(pa.string())
            date_mismatch += pc.sum(pc.cast(pc.not_equal(dcol, day), pa.int64())).as_py() or 0
            k2 = dcol
        else:
            k2 = day
        cur = pa.table({"k1": tk, "k2": k2, "k3": dtc})
        offset = rows
        if prev is not None:
            cur = pa.concat_tables([prev, cur])
            offset -= 1
        m = cur.num_rows
        if m >= 2:
            a1, a2, a3 = (cur[c].combine_chunks() for c in ("k1", "k2", "k3"))
            p1, n1 = a1.slice(0, m - 1), a1.slice(1)
            p2, n2 = a2.slice(0, m - 1), a2.slice(1)
            p3, n3 = a3.slice(0, m - 1), a3.slice(1)
            lt1, eq1 = pc.less(n1, p1), pc.equal(n1, p1)
            lt2, eq2 = pc.less(n2, p2), pc.equal(n2, p2)
            lt3, eq3 = pc.less(n3, p3), pc.equal(n3, p3)
            bad = pc.or_kleene(lt1, pc.and_kleene(eq1, pc.or_kleene(lt2, pc.and_kleene(eq2, lt3))))
            dup = pc.and_kleene(eq1, pc.and_kleene(eq2, eq3))
            nb = pc.sum(pc.cast(bad, pa.int64())).as_py() or 0
            nd = pc.sum(pc.cast(dup, pa.int64())).as_py() or 0
            viol += nb
            dups += nd
            if nb and first_bad is None:
                i = pc.index(bad, True).as_py()
                first_bad = {"row": offset + i + 1,
                             "prev": (p1[i].as_py(), p2[i].as_py(), str(p3[i].as_py())),
                             "next": (n1[i].as_py(), n2[i].as_py(), str(n3[i].as_py()))}
        prev = cur.slice(m - 1, 1)
        rows += n
        # кількість тикерів: зміни значення поміж сусідніми рядками (файл відсортований -> блоки)
        ch = pc.sum(pc.cast(pc.not_equal(tk.slice(1), tk.slice(0, n - 1)), pa.int64())).as_py() or 0 if n > 1 else 0
        tickers += ch + (1 if (last_ticker is None or tk[0].as_py() != last_ticker) else 0)
        last_ticker = tk[n - 1].as_py()

    first_day = min(days) if days else None
    last_day = max(days) if days else None
    problems = []
    if viol:
        problems.append(f"порушень порядку (ticker, date, dt): {viol:,}; перше: {first_bad}")
    if dups:
        problems.append(f"дублікатів (ticker, dt): {dups:,}")
    if null_tk or null_dt:
        problems.append(f"null: ticker={null_tk:,}, dt={null_dt:,}")
    if date_mismatch:
        problems.append(f"колонка date ≠ дата з dt у {date_mismatch:,} рядках")
    if expected_rows is not None and rows != expected_rows:
        problems.append(f"рядків {rows:,}, а у вхідному файлі {expected_rows:,} (сортування щось згубило)")
    if start_date and first_day and first_day < str(start_date)[:10]:
        problems.append(f"є дні раніше вікна: {first_day} < {str(start_date)[:10]}")
    if end_date and last_day and last_day > str(end_date)[:10]:
        problems.append(f"є дні пізніше вікна: {last_day} > {str(end_date)[:10]}")
    if end_date and last_day:
        import datetime as _dtm
        lag = (_dtm.date.fromisoformat(str(end_date)[:10]) - _dtm.date.fromisoformat(last_day)).days
        if lag > 4:
            problems.append(f"найновіший день {last_day} відстає від кінця вікна {str(end_date)[:10]} на {lag} дн.")

    info = {"rows": rows, "tickers": tickers, "first_day": first_day, "last_day": last_day,
            "days": len(days), "order_violations": viol, "duplicates": dups,
            "null_ticker": null_tk, "null_dt": null_dt, "date_mismatch": date_mismatch,
            "problems": problems, "ok": not problems}
    if verbose:
        print(f"📋 {path}")
        print(f"   рядків {rows:,} | тикерів {tickers:,} | днів {len(days)} ({first_day} .. {last_day})")
        if problems:
            print("   ❌ ПРОБЛЕМИ:")
            for p in problems:
                print("     -", p)
        else:
            print("   ✅ порядок ticker→date→dt, без дублікатів і null, дати в межах вікна")
    if problems and raise_on_error:
        raise ValueError("final parquet не пройшов перевірку: " + "; ".join(problems))
    return info


In [63]:
ram_guard("перед фінальним сортуванням")
print(mem_diag("перед фінальним сортуванням"))

_final_in_rows = pq.ParquetFile(str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet")).metadata.num_rows

recompute_tp_deviation_devsig(
    input_path=str(WORK_DIR / "premarket_rows_with_bench_pct_with_sigma.parquet"),
    output_path=str(FINAL_PATH),
    chunksize=BATCH_ROWS,
    round_tp_dev=2,
    round_devsig=6,
)

print(f"🧹 дублікатів (ticker, dt) прибрано: {_FINAL_STATS['dups_removed']:,} "
      f"(з різними значеннями: {_FINAL_STATS['conflicts']:,})")

# перевірка: порядок ticker→date→dt, без дублікатів і null, дати у вікні, кількість рядків сходиться
final_check = validate_final_parquet(
    str(FINAL_PATH),
    expected_rows=_final_in_rows - _FINAL_STATS["dups_removed"],
    start_date=start_date_str,
    end_date=end_date_str,
)

# ── Прибирання проміжних файлів (final.parquet уже готовий і перевірений) ──────────────
# Лишаються: final.parquet, oc*.parquet, rolling/current_stats (малі кеші) і gz-батчі в
# CRACEN_batch/ETF (ковзне вікно, з нього щодня докачуються лише нові дні).
# Щоб перезапустити окремо клітинку з обробкою, спершу знову виконайте клітинки collect (premarket_*).
if CLEANUP_INTERMEDIATE:
    _freed = 0
    _victims = [WORK_DIR / n for n in (
        "premarket_rows.parquet",
        "premarket_rows_with_bench_pct_with_sigma.parquet",
        "premarket_etf_rows.parquet",
        "premarket_etf_rows_enriched_with_pct.parquet",
        "premarket_rows_enriched_with_pct.parquet",
        "premarket_rows_final.parquet",
        "premarket_rows_with_bench_pct.parquet",
    )]
    _victims += list(WORK_DIR.glob("*.parquet.tmp_*")) + list(BATCH_DIR.glob("*.tmp")) + list(ETF_DIR.glob("*.tmp"))
    for _f in _victims:
        if _f.exists() and _f.resolve() != FINAL_PATH.resolve():
            _freed += _f.stat().st_size
            _f.unlink()
    print(f"🧹 проміжні файли видалено, звільнено {_freed / 1024**3:.2f} ГБ")
ram_guard("кінець")


▶ START (bucketed sort): C:\datum-api-examples-main\OriON\CRACEN\work\premarket_rows_with_bench_pct_with_sigma.parquet → C:\datum-api-examples-main\OriON\CRACEN\final.parquet | rows=1,223,499,149 tickers=8,103 buckets=62
  bucket 1/62  written=19,285,282  elapsed=1,080.4s
  bucket 2/62  written=34,041,096  elapsed=1,096.6s
  bucket 3/62  written=53,800,597  elapsed=1,119.4s
  bucket 4/62  written=71,118,866  elapsed=1,138.7s
  bucket 5/62  written=87,177,901  elapsed=1,156.3s
  bucket 6/62  written=103,345,463  elapsed=1,173.8s
  bucket 7/62  written=116,961,128  elapsed=1,188.1s
  bucket 8/62  written=135,951,288  elapsed=1,209.4s
  bucket 9/62  written=151,432,249  elapsed=1,226.5s
  bucket 10/62  written=167,023,159  elapsed=1,243.1s
  bucket 11/62  written=183,528,497  elapsed=1,261.1s
  bucket 12/62  written=199,429,657  elapsed=1,278.8s
  bucket 13/62  written=215,774,802  elapsed=1,296.2s
  bucket 14/62  written=231,731,877  elapsed=1,313.0s
  bucket 15/62  written=249,416,787  